In [ ]:
!pip install pymupdf pytesseract pillow pandas openpyxl opencv-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 MB 35.2 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 42.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [pymupdf]


In [ ]:
import fitz
import pytesseract
import pandas as pd
import cv2
import openpyxl

print("All libraries imported successfully!")

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


All libraries imported successfully!


In [ ]:
import pymupdf
import pytesseract
import pandas as pd
import cv2
import openpyxl
from PIL import Image
import os
import json
import io

print("Document processing environment is ready!")

Document processing environment is ready!


In [ ]:
import os

folders = [
    "document_intelligence",
    "document_intelligence/uploads",
    "document_intelligence/extracted_text",
    "document_intelligence/results",
    "document_intelligence/logs"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project folders created successfully!")

for folder in folders:
    print(folder)

Project folders created successfully!
document_intelligence
document_intelligence/uploads
document_intelligence/extracted_text
document_intelligence/results
document_intelligence/logs


In [ ]:
import pymupdf
import json
import os

def extract_pdf_text(pdf_path):

    results = []

    try:
        doc = pymupdf.open(pdf_path)

        if doc.is_encrypted:
            return {
                "status": "failed",
                "error": "Password-protected PDF"
            }

        total_pages = len(doc)

        for page_number, page in enumerate(doc, start=1):

            try:
                text = page.get_text("text")

                results.append({
                    "page_number": page_number,
                    "text": text.strip(),
                    "extraction_status": (
                        "success" if text.strip() else "no_text_found"
                    )
                })

            except Exception as page_error:
                results.append({
                    "page_number": page_number,
                    "text": "",
                    "extraction_status": "failed",
                    "error": str(page_error)
                })

        doc.close()

        return {
            "status": "success",
            "filename": os.path.basename(pdf_path),
            "total_pages": total_pages,
            "extracted_pages": sum(
                1 for page in results
                if page["extraction_status"] == "success"
            ),
            "pages": results
        }

    except Exception as error:
        return {
            "status": "failed",
            "filename": os.path.basename(pdf_path),
            "error": str(error)
        }

In [ ]:
from google.colab import files

uploaded = files.upload()

for filename in uploaded.keys():
    print("Uploaded:", filename)

Saving FINAL paper.pdf to FINAL paper.pdf
Uploaded: FINAL paper.pdf


In [ ]:
for filename in uploaded.keys():

    if filename.lower().endswith(".pdf"):

        result = extract_pdf_text(filename)

        print("Status:", result["status"])

        if result["status"] == "success":

            print("Total pages:", result["total_pages"])
            print("Extracted pages:", result["extracted_pages"])

            if result["pages"]:
                print("\nFirst page preview:")
                print(result["pages"][0]["text"][:1000])

            output_path = (
                "document_intelligence/extracted_text/"
                + filename + ".json"
            )

            with open(output_path, "w", encoding="utf-8") as f:
                json.dump(result, f, indent=4, ensure_ascii=False)

            print("\nJSON saved:", output_path)

        else:
            print("Error:", result.get("error"))

Status: success
Total pages: 6
Extracted pages: 6

First page preview:
XXX-X-XXXX-XXXX-X/XX/$XX.00 ©20XX IEEE 
Enhancing Parkinson’s Diagnosis: Unleashing the 
Potential of Machine Learning via Vocal Features Fusion 
 
         Abstract—Parkinson's disease (PD) is a degenerative 
brain disease that worsens with age, causing areas of the brain 
to weaken. The main focus of this study has been on 
the characteristics of the voice modality with a wide range of 
features including vocal fold, MFCC, baseline features, wavelet 
features and temporal frequency features. The proposed models 
introduced dimensionality reduction techniques such as PCA 
and ICA to improve the quality of the data with different 
machine learning classifiers (K-Nearest Neighbors, Support 
Vector Machines, Naive Bayes etc) for classification of PD. Then 
the introduction of deep neural network (DNN) models, which 
focused on their built-in capacity to interpret complex patterns 
in the dataset, changed the analytica

In [ ]:
!apt-get update -qq
!apt-get install -y tesseract-ocr

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (5.3.4-1build5).
0 upgraded, 0 newly installed, 0 to remove and 21 not upgraded.


In [ ]:
import pytesseract

print("Tesseract version:")
print(pytesseract.get_tesseract_version())

Tesseract version:
5.3.4


In [ ]:
import pymupdf
import pytesseract
from PIL import Image
import io
import os

def extract_pdf_with_ocr(pdf_path):

    results = []

    try:
        doc = pymupdf.open(pdf_path)

        if doc.is_encrypted:
            return {"status": "failed", "error": "Encrypted PDF"}

        for page_number, page in enumerate(doc, start=1):

            try:
                text = page.get_text("text").strip()

                if text:
                    method = "digital_text"

                else:
                    pix = page.get_pixmap(
                        matrix=pymupdf.Matrix(2, 2),
                        alpha=False
                    )

                    image = Image.open(
                        io.BytesIO(pix.tobytes("png"))
                    )

                    text = pytesseract.image_to_string(
                        image,
                        config="--psm 3"
                    ).strip()

                    method = "ocr"

                results.append({
                    "page_number": page_number,
                    "text": text,
                    "extraction_method": method,
                    "status": "success" if text else "no_text_found"
                })

            except Exception as error:
                results.append({
                    "page_number": page_number,
                    "text": "",
                    "status": "failed",
                    "error": str(error)
                })

        doc.close()

        return {
            "status": "success",
            "filename": os.path.basename(pdf_path),
            "total_pages": len(results),
            "pages": results
        }

    except Exception as error:
        return {
            "status": "failed",
            "error": str(error)
        }

In [ ]:
import pymupdf
import pytesseract
from PIL import Image
import io
import os

def extract_pdf_with_ocr(pdf_path):

    results = []

    try:
        doc = pymupdf.open(pdf_path)

        if doc.is_encrypted:
            return {"status": "failed", "error": "Encrypted PDF"}

        for page_number, page in enumerate(doc, start=1):

            try:
                text = page.get_text("text").strip()

                if text:
                    method = "digital_text"

                else:
                    pix = page.get_pixmap(
                        matrix=pymupdf.Matrix(2, 2),
                        alpha=False
                    )

                    image = Image.open(
                        io.BytesIO(pix.tobytes("png"))
                    )

                    text = pytesseract.image_to_string(
                        image,
                        config="--psm 3"
                    ).strip()

                    method = "ocr"

                results.append({
                    "page_number": page_number,
                    "text": text,
                    "extraction_method": method,
                    "status": "success" if text else "no_text_found"
                })

            except Exception as error:
                results.append({
                    "page_number": page_number,
                    "text": "",
                    "status": "failed",
                    "error": str(error)
                })

        doc.close()

        return {
            "status": "success",
            "filename": os.path.basename(pdf_path),
            "total_pages": len(results),
            "pages": results
        }

    except Exception as error:
        return {
            "status": "failed",
            "error": str(error)
        }

In [ ]:
from google.colab import files

uploaded_images = files.upload()

Saving mm.png to mm.png


In [ ]:
from PIL import Image
import pytesseract

for filename in uploaded_images:
    image = Image.open(filename).convert("RGB")
    text = pytesseract.image_to_string(image)

    print("File:", filename)
    print("Extracted text:")
    print(text)

File: mm.png
Extracted text:
Training

set

Testing set

Ensemble
voting
classifier

decision

=== Parkinson Patient

|= Healthy control



In [ ]:
import json
import os

image_results = []

for filename in uploaded_images:
    try:
        image = Image.open(filename).convert("RGB")
        extracted_text = pytesseract.image_to_string(image)

        image_results.append({
            "filename": filename,
            "extraction_method": "ocr",
            "status": "success" if extracted_text.strip() else "no_text_found",
            "extracted_text": extracted_text.strip()
        })

    except Exception as error:
        image_results.append({
            "filename": filename,
            "status": "failed",
            "error": str(error)
        })

output_path = "document_intelligence/extracted_text/image_ocr_results.json"

with open(output_path, "w", encoding="utf-8") as file:
    json.dump(image_results, file, indent=4, ensure_ascii=False)

print("Image OCR results saved successfully!")
print("Saved at:", output_path)

Image OCR results saved successfully!
Saved at: document_intelligence/extracted_text/image_ocr_results.json


In [ ]:
import pandas as pd
import os
import json

def extract_spreadsheet(file_path):
    try:
        extension = os.path.splitext(file_path)[1].lower()

        if extension == ".csv":
            sheets = {"Sheet1": pd.read_csv(file_path)}
        elif extension in [".xlsx", ".xls"]:
            sheets = pd.read_excel(file_path, sheet_name=None)
        else:
            return {"status": "failed", "error": "Unsupported file format"}

        results = {}

        for sheet_name, dataframe in sheets.items():
            results[sheet_name] = {
                "columns": dataframe.columns.tolist(),
                "row_count": len(dataframe),
                "data": dataframe.fillna("").to_dict(orient="records")
            }

        return {
            "status": "success",
            "filename": os.path.basename(file_path),
            "sheets": results
        }

    except Exception as error:
        return {
            "status": "failed",
            "filename": os.path.basename(file_path),
            "error": str(error)
        }

In [ ]:
from google.colab import files

uploaded_sheets = files.upload()

for filename in uploaded_sheets:
    result = extract_spreadsheet(filename)

    print("\nFile:", filename)
    print("Status:", result["status"])

    if result["status"] == "success":
        print("Sheets:", list(result["sheets"].keys()))
        for sheet, content in result["sheets"].items():
            print("Rows in", sheet, ":", content["row_count"])

Saving dataset_variable_description.xlsx to dataset_variable_description.xlsx

File: dataset_variable_description.xlsx
Status: success
Sheets: ['Sheet1']
Rows in Sheet1 : 33


In [ ]:
for filename in uploaded_sheets:
    result = extract_spreadsheet(filename)

    if result["status"] == "success":
        output_name = os.path.splitext(filename)[0] + ".json"
        output_path = os.path.join(
            "document_intelligence",
            "extracted_text",
            output_name
        )

        with open(output_path, "w", encoding="utf-8") as file:
            json.dump(result, file, indent=4, ensure_ascii=False)

        print("Saved successfully:", output_path)
    else:
        print("Failed:", result["error"])

Saved successfully: document_intelligence/extracted_text/dataset_variable_description.json


In [ ]:
import re
import json

def extract_invoice_fields(text):
    patterns = {
        "invoice_number": r"(?i)invoice\s*(?:number|no|#)?\s*[:#-]?\s*([A-Z0-9-]+)",
        "invoice_date": r"(?i)(?:invoice\s*date|date)\s*[:#-]?\s*(\d{1,2}[/-]\d{1,2}[/-]\d{2,4})",
        "subtotal": r"(?i)subtotal\s*[:#-]?\s*([0-9,]+\.?\d*)",
        "tax": r"(?i)(?:tax|vat)\s*[:#-]?\s*([0-9,]+\.?\d*)",
        "total": r"(?i)(?:grand\s*total|total\s*amount|total)\s*[:#-]?\s*([0-9,]+\.?\d*)"
    }

    extracted = {}

    for field, pattern in patterns.items():
        match = re.search(pattern, text)
        extracted[field] = match.group(1) if match else None

    extracted["supplier_name"] = None
    extracted["customer_name"] = None
    extracted["currency"] = None
    extracted["line_items"] = []

    return extracted

In [ ]:
sample_invoice = """
INVOICE
Invoice Number: INV-2026-001
Invoice Date: 04/11/2026

Supplier: ABC Technologies
Customer: XYZ Company

Subtotal: 1000.00
Tax: 150.00
Grand Total: 1150.00
"""

invoice_result = extract_invoice_fields(sample_invoice)

print(json.dumps(invoice_result, indent=4))

{
    "invoice_number": "Invoice",
    "invoice_date": "04/11/2026",
    "subtotal": "1000.00",
    "tax": "150.00",
    "total": "1000.00",
    "supplier_name": null,
    "customer_name": null,
    "currency": null,
    "line_items": []
}


In [ ]:
import re
import json

def extract_invoice_fields(text):

    patterns = {
        "invoice_number": r"(?im)^Invoice\s*(?:Number|No\.?|#)\s*[:#-]?\s*([A-Z0-9-]+)",
        "invoice_date": r"(?im)^(?:Invoice\s*Date|Date)\s*[:#-]?\s*(\d{1,2}[/-]\d{1,2}[/-]\d{2,4})",
        "subtotal": r"(?im)^Subtotal\s*[:#-]?\s*([0-9,]+\.?\d*)",
        "tax": r"(?im)^(?:Tax|VAT)\s*[:#-]?\s*([0-9,]+\.?\d*)",
        "total": r"(?im)^(?:Grand\s*Total|Total\s*Amount|Total)\s*[:#-]?\s*([0-9,]+\.?\d*)"
    }

    extracted = {}

    for field, pattern in patterns.items():
        match = re.search(pattern, text)
        extracted[field] = match.group(1) if match else None

    supplier = re.search(r"(?im)^Supplier\s*:\s*(.+)$", text)
    customer = re.search(r"(?im)^Customer\s*:\s*(.+)$", text)

    extracted["supplier_name"] = supplier.group(1).strip() if supplier else None
    extracted["customer_name"] = customer.group(1).strip() if customer else None

    currency_match = re.search(r"\b(USD|BDT|EUR|GBP|INR)\b", text, re.I)
    extracted["currency"] = currency_match.group(1).upper() if currency_match else None

    extracted["line_items"] = []

    return extracted

In [ ]:
invoice_result = extract_invoice_fields(sample_invoice)
print(json.dumps(invoice_result, indent=4))

{
    "invoice_number": "INV-2026-001",
    "invoice_date": "04/11/2026",
    "subtotal": "1000.00",
    "tax": "150.00",
    "total": "1150.00",
    "supplier_name": "ABC Technologies",
    "customer_name": "XYZ Company",
    "currency": null,
    "line_items": []
}


In [ ]:
import re

def extract_invoice_fields(text):

    patterns = {
        "invoice_number": r"(?im)^Invoice\s*(?:Number|No\.?|#)\s*[:#-]?\s*([A-Z0-9-]+)",
        "invoice_date": r"(?im)^(?:Invoice\s*Date|Date)\s*[:#-]?\s*(\d{1,2}[/-]\d{1,2}[/-]\d{2,4})",
        "subtotal": r"(?im)^Subtotal\s*[:#-]?\s*([0-9,]+\.?\d*)",
        "tax": r"(?im)^(?:Tax|VAT)\s*[:#-]?\s*([0-9,]+\.?\d*)",
        "total": r"(?im)^(?:Grand\s*Total|Total\s*Amount|Total)\s*[:#-]?\s*([0-9,]+\.?\d*)"
    }

    extracted = {}

    for field, pattern in patterns.items():
        match = re.search(pattern, text)
        extracted[field] = match.group(1) if match else None

    supplier = re.search(r"(?im)^Supplier\s*:\s*(.+)$", text)
    customer = re.search(r"(?im)^Customer\s*:\s*(.+)$", text)

    extracted["supplier_name"] = supplier.group(1).strip() if supplier else None
    extracted["customer_name"] = customer.group(1).strip() if customer else None

    currency_patterns = {
        "USD": r"\bUSD\b|\$",
        "BDT": r"\bBDT\b|৳",
        "EUR": r"\bEUR\b|€",
        "GBP": r"\bGBP\b|£",
        "INR": r"\bINR\b|₹"
    }

    extracted["currency"] = next(
        (code for code, pattern in currency_patterns.items()
         if re.search(pattern, text, re.I)),
        None
    )

    extracted["line_items"] = []

    return extracted

In [ ]:
sample_invoice = """
INVOICE
Invoice Number: INV-2026-001
Invoice Date: 04/11/2026

Supplier: ABC Technologies
Customer: XYZ Company

Item             Quantity    Unit Price    Amount
Laptop           2           500.00         1000.00
Mouse            1           50.00          50.00

Subtotal: 1050.00
Tax: 157.50
Grand Total: 1207.50 BDT
"""

invoice_result = extract_invoice_fields(sample_invoice)

print(json.dumps(invoice_result, indent=4))

{
    "invoice_number": "INV-2026-001",
    "invoice_date": "04/11/2026",
    "subtotal": "1050.00",
    "tax": "157.50",
    "total": "1207.50",
    "supplier_name": "ABC Technologies",
    "customer_name": "XYZ Company",
    "currency": "BDT",
    "line_items": []
}


In [ ]:
import re

def extract_invoice_fields(text):

    patterns = {
        "invoice_number": r"(?im)^Invoice\s*(?:Number|No\.?|#)\s*[:#-]?\s*([A-Z0-9-]+)",
        "invoice_date": r"(?im)^(?:Invoice\s*Date|Date)\s*[:#-]?\s*(\d{1,2}[/-]\d{1,2}[/-]\d{2,4})",
        "subtotal": r"(?im)^Subtotal\s*[:#-]?\s*([0-9,]+\.?\d*)",
        "tax": r"(?im)^(?:Tax|VAT)\s*[:#-]?\s*([0-9,]+\.?\d*)",
        "total": r"(?im)^(?:Grand\s*Total|Total\s*Amount|Total)\s*[:#-]?\s*([0-9,]+\.?\d*)"
    }

    extracted = {}

    for field, pattern in patterns.items():
        match = re.search(pattern, text)
        extracted[field] = match.group(1) if match else None

    supplier = re.search(r"(?im)^Supplier\s*:\s*(.+)$", text)
    customer = re.search(r"(?im)^Customer\s*:\s*(.+)$", text)

    extracted["supplier_name"] = supplier.group(1).strip() if supplier else None
    extracted["customer_name"] = customer.group(1).strip() if customer else None

    currency_patterns = {
        "USD": r"\bUSD\b|\$",
        "BDT": r"\bBDT\b|৳",
        "EUR": r"\bEUR\b|€",
        "GBP": r"\bGBP\b|£",
        "INR": r"\bINR\b|₹"
    }

    extracted["currency"] = next(
        (code for code, pattern in currency_patterns.items()
         if re.search(pattern, text, re.I)),
        None
    )

    # Extract line items
    line_items = []

    item_pattern = (
        r"(?m)^([A-Za-z][A-Za-z0-9 &-]*?)\s+"
        r"(\d+)\s+"
        r"([\d,]+\.\d{2})\s+"
        r"([\d,]+\.\d{2})\s*$"
    )

    for match in re.finditer(item_pattern, text):
        line_items.append({
            "description": match.group(1).strip(),
            "quantity": int(match.group(2)),
            "unit_price": float(match.group(3).replace(",", "")),
            "amount": float(match.group(4).replace(",", ""))
        })

    extracted["line_items"] = line_items

    return extracted

In [ ]:
invoice_result = extract_invoice_fields(sample_invoice)
print(json.dumps(invoice_result, indent=4))

{
    "invoice_number": "INV-2026-001",
    "invoice_date": "04/11/2026",
    "subtotal": "1050.00",
    "tax": "157.50",
    "total": "1207.50",
    "supplier_name": "ABC Technologies",
    "customer_name": "XYZ Company",
    "currency": "BDT",
    "line_items": [
        {
            "description": "Laptop",
            "quantity": 2,
            "unit_price": 500.0,
            "amount": 1000.0
        },
        {
            "description": "Mouse",
            "quantity": 1,
            "unit_price": 50.0,
            "amount": 50.0
        }
    ]
}


In [ ]:
def validate_invoice(invoice):

    errors = []

    # Check required fields
    required_fields = [
        "invoice_number",
        "invoice_date",
        "supplier_name",
        "total"
    ]

    for field in required_fields:
        if not invoice.get(field):
            errors.append(f"Missing required field: {field}")

    # Convert numeric fields safely
    def to_number(value):
        try:
            return float(str(value).replace(",", "")) if value is not None else None
        except (ValueError, TypeError):
            return None

    subtotal = to_number(invoice.get("subtotal"))
    tax = to_number(invoice.get("tax"))
    total = to_number(invoice.get("total"))

    # Check subtotal + tax against total
    if subtotal is not None and tax is not None and total is not None:
        if abs((subtotal + tax) - total) > 0.01:
            errors.append("Subtotal + tax does not match total")

    # Check line-item amounts
    for item in invoice.get("line_items", []):
        quantity = to_number(item.get("quantity"))
        unit_price = to_number(item.get("unit_price"))
        amount = to_number(item.get("amount"))

        if quantity is None or unit_price is None or amount is None:
            errors.append(
                f"Incomplete line item: {item.get('description', 'Unknown')}"
            )
        elif abs((quantity * unit_price) - amount) > 0.01:
            errors.append(
                f"Incorrect amount for line item: {item.get('description')}"
            )

    return {
        "validation_status": "passed" if not errors else "needs_review",
        "errors": errors
    }

In [ ]:
validation_result = validate_invoice(invoice_result)

print(json.dumps(validation_result, indent=4))

{
    "validation_status": "passed",
    "errors": []
}


In [ ]:
incorrect_invoice = {
    "invoice_number": "INV-2026-002",
    "invoice_date": "05/11/2026",
    "supplier_name": "ABC Technologies",
    "customer_name": "XYZ Company",
    "subtotal": "1000.00",
    "tax": "150.00",
    "total": "1300.00",
    "currency": "BDT",
    "line_items": [
        {
            "description": "Laptop",
            "quantity": 2,
            "unit_price": 500.00,
            "amount": 900.00
        }
    ]
}

test_result = validate_invoice(incorrect_invoice)

print(json.dumps(test_result, indent=4))

{
    "validation_status": "needs_review",
    "errors": [
        "Subtotal + tax does not match total",
        "Incorrect amount for line item: Laptop"
    ]
}


In [ ]:
import os
import json
import pymupdf
from PIL import Image
import pytesseract
import pandas as pd

def process_document(file_path):

    extension = os.path.splitext(file_path)[1].lower()

    # PDF processing
    if extension == ".pdf":
        result = extract_pdf_with_ocr(file_path)

        if result["status"] == "success":
            full_text = "\n".join(
                page["text"] for page in result["pages"]
            )
            invoice_data = extract_invoice_fields(full_text)
            validation = validate_invoice(invoice_data)

            result["invoice_data"] = invoice_data
            result["validation"] = validation

    # Image processing
    elif extension in [".png", ".jpg", ".jpeg", ".tiff", ".tif"]:
        image = Image.open(file_path).convert("RGB")
        text = pytesseract.image_to_string(image)

        invoice_data = extract_invoice_fields(text)
        validation = validate_invoice(invoice_data)

        result = {
            "status": "success",
            "filename": os.path.basename(file_path),
            "extraction_method": "ocr",
            "extracted_text": text,
            "invoice_data": invoice_data,
            "validation": validation
        }

    # Spreadsheet processing
    elif extension in [".xlsx", ".xls", ".csv"]:
        result = extract_spreadsheet(file_path)

    else:
        return {
            "status": "failed",
            "error": "Unsupported file format"
        }

    # Save final result
    output_name = os.path.basename(file_path) + ".json"
    output_path = os.path.join(
        "document_intelligence",
        "results",
        output_name
    )

    with open(output_path, "w", encoding="utf-8") as file:
        json.dump(result, file, indent=4, ensure_ascii=False)

    return result

In [ ]:
!pip install reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 11.5 MB/s  0:00:00


In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas

invoice_path = "sample_invoice.pdf"

c = canvas.Canvas(invoice_path, pagesize=A4)

c.setFont("Helvetica-Bold", 20)
c.drawString(50, 800, "INVOICE")

c.setFont("Helvetica", 11)
c.drawString(50, 750, "Invoice Number: INV-2026-003")
c.drawString(50, 730, "Invoice Date: 04/11/2026")

c.drawString(50, 690, "Supplier: ABC Technologies")
c.drawString(50, 670, "Customer: XYZ Company")

c.setFont("Helvetica-Bold", 11)
c.drawString(50, 620, "Item")
c.drawString(250, 620, "Quantity")
c.drawString(350, 620, "Unit Price")
c.drawString(460, 620, "Amount")

c.setFont("Helvetica", 11)
c.drawString(50, 590, "Laptop")
c.drawString(250, 590, "2")
c.drawString(350, 590, "500.00")
c.drawString(460, 590, "1000.00")

c.drawString(50, 560, "Mouse")
c.drawString(250, 560, "1")
c.drawString(350, 560, "50.00")
c.drawString(460, 560, "50.00")

c.drawString(350, 500, "Subtotal: 1050.00")
c.drawString(350, 480, "Tax: 157.50")
c.drawString(350, 460, "Grand Total: 1207.50 BDT")

c.save()

print("Sample invoice PDF created successfully!")

Sample invoice PDF created successfully!


In [ ]:
result = process_document("sample_invoice.pdf")

print(json.dumps(result, indent=4))

{
    "status": "success",
    "filename": "sample_invoice.pdf",
    "total_pages": 1,
    "pages": [
        {
            "page_number": 1,
            "text": "INVOICE\nInvoice Number: INV-2026-003\nInvoice Date: 04/11/2026\nSupplier: ABC Technologies\nCustomer: XYZ Company\nItem\nQuantity\nUnit Price\nAmount\nLaptop\n2\n500.00\n1000.00\nMouse\n1\n50.00\n50.00\nSubtotal: 1050.00\nTax: 157.50\nGrand Total: 1207.50 BDT",
            "extraction_method": "digital_text",
            "status": "success"
        }
    ],
    "invoice_data": {
        "invoice_number": "INV-2026-003",
        "invoice_date": "04/11/2026",
        "subtotal": "1050.00",
        "tax": "157.50",
        "total": "1207.50",
        "supplier_name": "ABC Technologies",
        "customer_name": "XYZ Company",
        "currency": "BDT",
        "line_items": [
            {
                "description": "Laptop",
                "quantity": 2,
                "unit_price": 500.0,
                "amount": 1000.

In [ ]:
import os

result_folder = "document_intelligence/results"

print("Files saved:")

for filename in os.listdir(result_folder):
    print(filename)

Files saved:
sample_invoice.pdf.json


In [ ]:
import sqlite3

db_path = "document_intelligence/documents.db"

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS documents (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    filename TEXT NOT NULL,
    file_type TEXT,
    processing_status TEXT,
    upload_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS invoices (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    document_id INTEGER,
    invoice_number TEXT,
    invoice_date TEXT,
    supplier_name TEXT,
    customer_name TEXT,
    subtotal REAL,
    tax REAL,
    total REAL,
    currency TEXT,
    FOREIGN KEY (document_id) REFERENCES documents(id)
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS validation_errors (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    document_id INTEGER,
    error_message TEXT,
    FOREIGN KEY (document_id) REFERENCES documents(id)
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS correction_history (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    document_id INTEGER,
    field_name TEXT,
    old_value TEXT,
    new_value TEXT,
    corrected_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (document_id) REFERENCES documents(id)
)
""")

conn.commit()

print("Database and all four tables created successfully!")

Database and all four tables created successfully!


In [ ]:
cursor.execute("""
SELECT name FROM sqlite_master
WHERE type='table'
""")

tables = cursor.fetchall()

print("Database tables:")
for table in tables:
    print("-", table[0])

Database tables:
- documents
- sqlite_sequence
- invoices
- validation_errors
- correction_history


In [ ]:
def save_invoice_to_database(file_path, result):

    cursor = conn.cursor()

    # Save document information
    cursor.execute("""
    INSERT INTO documents (filename, file_type, processing_status)
    VALUES (?, ?, ?)
    """, (
        os.path.basename(file_path),
        os.path.splitext(file_path)[1].lower(),
        result.get("status", "failed")
    ))

    document_id = cursor.lastrowid

    # Save invoice information
    if result.get("invoice_data"):
        invoice = result["invoice_data"]

        def safe_float(value):
            try:
                return float(str(value).replace(",", "")) if value is not None else None
            except (ValueError, TypeError):
                return None

        cursor.execute("""
        INSERT INTO invoices (
            document_id, invoice_number, invoice_date,
            supplier_name, customer_name, subtotal,
            tax, total, currency
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            document_id,
            invoice.get("invoice_number"),
            invoice.get("invoice_date"),
            invoice.get("supplier_name"),
            invoice.get("customer_name"),
            safe_float(invoice.get("subtotal")),
            safe_float(invoice.get("tax")),
            safe_float(invoice.get("total")),
            invoice.get("currency")
        ))

    # Save validation errors
    validation = result.get("validation", {})

    for error in validation.get("errors", []):
        cursor.execute("""
        INSERT INTO validation_errors (document_id, error_message)
        VALUES (?, ?)
        """, (document_id, error))

    conn.commit()

    return document_id


print("Database saving function created successfully!")

Database saving function created successfully!


In [ ]:
document_id = save_invoice_to_database(
    "sample_invoice.pdf",
    result
)

print("Invoice saved successfully!")
print("Document ID:", document_id)

Invoice saved successfully!
Document ID: 1


In [ ]:
cursor.execute("""
SELECT
    invoice_number,
    supplier_name,
    customer_name,
    subtotal,
    tax,
    total,
    currency
FROM invoices
""")

for row in cursor.fetchall():
    print(row)

('INV-2026-003', 'ABC Technologies', 'XYZ Company', 1050.0, 157.5, 1207.5, 'BDT')


In [ ]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS line_items (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    invoice_id INTEGER,
    description TEXT,
    quantity REAL,
    unit_price REAL,
    amount REAL,
    FOREIGN KEY (invoice_id) REFERENCES invoices(id)
)
""")

conn.commit()

print("Line items table created successfully!")

Line items table created successfully!


In [ ]:
cursor.execute("""
SELECT id FROM invoices
WHERE invoice_number = ?
""", ("INV-2026-003",))

invoice_id = cursor.fetchone()[0]

for item in result["invoice_data"]["line_items"]:

    cursor.execute("""
    INSERT INTO line_items (
        invoice_id, description, quantity, unit_price, amount
    )
    VALUES (?, ?, ?, ?, ?)
    """, (
        invoice_id,
        item["description"],
        item["quantity"],
        item["unit_price"],
        item["amount"]
    ))

conn.commit()

print("Invoice line items saved successfully!")

Invoice line items saved successfully!


In [ ]:
cursor.execute("""
SELECT description, quantity, unit_price, amount
FROM line_items
WHERE invoice_id = ?
""", (invoice_id,))

for row in cursor.fetchall():
    print(row)

('Laptop', 2.0, 500.0, 1000.0)
('Mouse', 1.0, 50.0, 50.0)


In [ ]:
!pip install fastapi uvicorn python-multipart nest-asyncio

In [ ]:
from fastapi import FastAPI, UploadFile, File, HTTPException
import shutil
import os
import sqlite3
import json

app = FastAPI(
    title="AI Document Intelligence API",
    description="Invoice extraction and document processing system",
    version="1.0.0"
)

DB_PATH = "document_intelligence/documents.db"
UPLOAD_DIR = "document_intelligence/uploads"

os.makedirs(UPLOAD_DIR, exist_ok=True)

def get_connection():
    return sqlite3.connect(DB_PATH)

@app.get("/")
def home():
    return {
        "message": "AI Document Intelligence API is running",
        "status": "success"
    }

@app.get("/documents")
def get_documents():
    connection = get_connection()
    cursor = connection.cursor()

    cursor.execute("""
        SELECT id, filename, file_type, processing_status
        FROM documents
    """)

    rows = cursor.fetchall()
    connection.close()

    return {"documents": rows}

@app.get("/invoices")
def get_invoices():
    connection = get_connection()
    cursor = connection.cursor()

    cursor.execute("""
        SELECT invoice_number, supplier_name,
               customer_name, total, currency
        FROM invoices
    """)

    rows = cursor.fetchall()
    connection.close()

    return {"invoices": rows}

In [ ]:
from fastapi import Form

@app.post("/upload")
async def upload_document(file: UploadFile = File(...)):

    allowed_extensions = [
        ".pdf", ".png", ".jpg", ".jpeg",
        ".tiff", ".tif", ".xlsx", ".xls", ".csv"
    ]

    extension = os.path.splitext(file.filename)[1].lower()

    if extension not in allowed_extensions:
        raise HTTPException(
            status_code=400,
            detail="Unsupported file format"
        )

    safe_filename = os.path.basename(file.filename)
    file_path = os.path.join(UPLOAD_DIR, safe_filename)

    with open(file_path, "wb") as buffer:
        shutil.copyfileobj(file.file, buffer)

    try:
        result = process_document(file_path)
        document_id = save_invoice_to_database(file_path, result)

        return {
            "document_id": document_id,
            "filename": safe_filename,
            "processing_status": result["status"],
            "validation": result.get("validation")
        }

    except Exception as error:
        raise HTTPException(status_code=500, detail=str(error))


@app.get("/search")
def search_invoices(keyword: str):

    connection = get_connection()
    cursor = connection.cursor()

    cursor.execute("""
        SELECT invoice_number, supplier_name,
               customer_name, total, currency
        FROM invoices
        WHERE invoice_number LIKE ?
        OR supplier_name LIKE ?
        OR customer_name LIKE ?
    """, (
        f"%{keyword}%",
        f"%{keyword}%",
        f"%{keyword}%"
    ))

    rows = cursor.fetchall()
    connection.close()

    return {"results": rows}


@app.put("/invoices/{invoice_id}/correct")
def correct_invoice(
    invoice_id: int,
    field_name: str = Form(...),
    new_value: str = Form(...)
):

    allowed_fields = [
        "invoice_number", "invoice_date",
        "supplier_name", "customer_name",
        "subtotal", "tax", "total", "currency"
    ]

    if field_name not in allowed_fields:
        raise HTTPException(status_code=400, detail="Invalid field")

    connection = get_connection()
    cursor = connection.cursor()

    cursor.execute(
        f"SELECT {field_name} FROM invoices WHERE id = ?",
        (invoice_id,)
    )

    row = cursor.fetchone()

    if row is None:
        connection.close()
        raise HTTPException(status_code=404, detail="Invoice not found")

    old_value = row[0]

    cursor.execute(
        f"UPDATE invoices SET {field_name} = ? WHERE id = ?",
        (new_value, invoice_id)
    )

    cursor.execute("""
        INSERT INTO correction_history
        (document_id, field_name, old_value, new_value)
        SELECT document_id, ?, ?, ?
        FROM invoices WHERE id = ?
    """, (field_name, str(old_value), new_value, invoice_id))

    connection.commit()
    connection.close()

    return {"message": "Invoice corrected successfully"}


@app.post("/retry/{document_id}")
def retry_document(document_id: int):

    connection = get_connection()
    cursor = connection.cursor()

    cursor.execute(
        "SELECT filename FROM documents WHERE id = ?",
        (document_id,)
    )

    row = cursor.fetchone()
    connection.close()

    if row is None:
        raise HTTPException(status_code=404, detail="Document not found")

    file_path = os.path.join(UPLOAD_DIR, row[0])

    if not os.path.exists(file_path):
        raise HTTPException(status_code=404, detail="Uploaded file not found")

    result = process_document(file_path)

    return {
        "document_id": document_id,
        "processing_status": result["status"]
    }

In [ ]:
@app.get("/documents/{document_id}/status")
def document_status(document_id: int):

    connection = get_connection()
    cursor = connection.cursor()

    cursor.execute("""
        SELECT id, filename, processing_status
        FROM documents
        WHERE id = ?
    """, (document_id,))

    row = cursor.fetchone()
    connection.close()

    if row is None:
        raise HTTPException(status_code=404, detail="Document not found")

    return {
        "document_id": row[0],
        "filename": row[1],
        "processing_status": row[2]
    }

In [ ]:
import nest_asyncio
import threading
import uvicorn

nest_asyncio.apply()

def run_api():
    uvicorn.run(app, host="0.0.0.0", port=8000)

api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()

print("FastAPI server is starting...")

FastAPI server is starting...


In [ ]:
from google.colab.output import eval_js

public_url = eval_js(
    "google.colab.kernel.proxyPort(8000)"
)

print("API Documentation:", public_url + "/docs")

API Documentation: https://8000-m-s-usc1b3-v2cut3cufcnt-b.us-central1.prod.r.colab.dev/docs


In [ ]:
from google.colab import files

files.download("sample_invoice.pdf")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Close the old database connection
conn.close()

# Reopen it with cross-thread access enabled
conn = sqlite3.connect(
    "document_intelligence/documents.db",
    check_same_thread=False
)

print("Database connection fixed successfully!")

Database connection fixed successfully!


In [ ]:
cursor = conn.cursor()

cursor.execute("""
SELECT id, invoice_number, supplier_name
FROM invoices
WHERE invoice_number = 'INV-2026-003'
""")

print(cursor.fetchall())

[(1, 'INV-2026-003', 'ABC Technologies'), (2, 'INV-2026-003', 'ABC Technologies')]


In [ ]:
cursor = conn.cursor()

cursor.execute("""
SELECT field_name, old_value, new_value
FROM correction_history
WHERE document_id = 2
""")

print(cursor.fetchall())

[('supplier_name', 'ABC Technologies', 'ABC Technologies Ltd')]


In [ ]:
def save_invoice_to_database(file_path, result):

    cursor = conn.cursor()

    # Save document information
    cursor.execute("""
    INSERT INTO documents (
        filename, file_type, processing_status
    )
    VALUES (?, ?, ?)
    """, (
        os.path.basename(file_path),
        os.path.splitext(file_path)[1].lower(),
        result.get("status", "failed")
    ))

    document_id = cursor.lastrowid

    invoice = result.get("invoice_data")

    if invoice:

        def safe_float(value):
            try:
                return float(str(value).replace(",", "")) if value is not None else None
            except (ValueError, TypeError):
                return None

        # Save invoice details
        cursor.execute("""
        INSERT INTO invoices (
            document_id, invoice_number, invoice_date,
            supplier_name, customer_name, subtotal,
            tax, total, currency
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            document_id,
            invoice.get("invoice_number"),
            invoice.get("invoice_date"),
            invoice.get("supplier_name"),
            invoice.get("customer_name"),
            safe_float(invoice.get("subtotal")),
            safe_float(invoice.get("tax")),
            safe_float(invoice.get("total")),
            invoice.get("currency")
        ))

        invoice_id = cursor.lastrowid

        # Automatically save line items
        for item in invoice.get("line_items", []):
            cursor.execute("""
            INSERT INTO line_items (
                invoice_id, description, quantity,
                unit_price, amount
            )
            VALUES (?, ?, ?, ?, ?)
            """, (
                invoice_id,
                item.get("description"),
                safe_float(item.get("quantity")),
                safe_float(item.get("unit_price")),
                safe_float(item.get("amount"))
            ))

    # Save validation errors
    validation = result.get("validation", {})

    for error in validation.get("errors", []):
        cursor.execute("""
        INSERT INTO validation_errors (
            document_id, error_message
        )
        VALUES (?, ?)
        """, (document_id, error))

    conn.commit()

    return document_id


print("Improved database-saving function is ready!")

Improved database-saving function is ready!


In [ ]:
# Process the sample invoice again
test_result = process_document("sample_invoice.pdf")

# Save it using the improved function
test_document_id = save_invoice_to_database(
    "sample_invoice.pdf",
    test_result
)

print("New document ID:", test_document_id)

# Check the saved line items
cursor = conn.cursor()

cursor.execute("""
SELECT description, quantity, unit_price, amount
FROM line_items
WHERE invoice_id = (
    SELECT id FROM invoices
    WHERE document_id = ?
)
""", (test_document_id,))

print("Saved line items:")
for item in cursor.fetchall():
    print(item)

New document ID: 3
Saved line items:
('Laptop', 2.0, 500.0, 1000.0)
('Mouse', 1.0, 50.0, 50.0)


In [ ]:
def check_duplicate_invoice(invoice_number, supplier_name):

    cursor = conn.cursor()

    cursor.execute("""
    SELECT id, document_id, invoice_number, supplier_name
    FROM invoices
    WHERE LOWER(invoice_number) = LOWER(?)
    AND LOWER(supplier_name) = LOWER(?)
    """, (invoice_number, supplier_name))

    matches = cursor.fetchall()

    return {
        "is_duplicate": len(matches) > 1,
        "matching_records": matches
    }


# Test duplicate detection
duplicate_result = check_duplicate_invoice(
    "INV-2026-003",
    "ABC Technologies"
)

print(duplicate_result)

{'is_duplicate': True, 'matching_records': [(1, 1, 'INV-2026-003', 'ABC Technologies'), (3, 3, 'INV-2026-003', 'ABC Technologies')]}


In [ ]:
def is_duplicate_before_save(invoice):

    invoice_number = invoice.get("invoice_number")
    supplier_name = invoice.get("supplier_name")

    if not invoice_number or not supplier_name:
        return False

    cursor = conn.cursor()

    cursor.execute("""
    SELECT COUNT(*)
    FROM invoices
    WHERE LOWER(invoice_number) = LOWER(?)
    AND LOWER(supplier_name) = LOWER(?)
    """, (invoice_number, supplier_name))

    count = cursor.fetchone()[0]

    return count > 0


# Test using our sample invoice
sample_data = test_result["invoice_data"]

if is_duplicate_before_save(sample_data):
    print("Duplicate invoice detected! Manual review required.")
else:
    print("New invoice. Safe to proceed.")

Duplicate invoice detected! Manual review required.


In [ ]:
from fastapi import UploadFile, File, HTTPException
import uuid

@app.post("/upload")
async def upload_document(file: UploadFile = File(...)):

    allowed_extensions = [
        ".pdf", ".png", ".jpg", ".jpeg",
        ".tiff", ".tif", ".xlsx", ".xls", ".csv"
    ]

    extension = os.path.splitext(file.filename)[1].lower()

    if extension not in allowed_extensions:
        raise HTTPException(
            status_code=400,
            detail="Unsupported file format"
        )

    # Generate unique filename to avoid overwriting files
    unique_filename = str(uuid.uuid4()) + extension
    file_path = os.path.join(UPLOAD_DIR, unique_filename)

    try:
        content = await file.read()

        if not content:
            raise HTTPException(
                status_code=400,
                detail="Uploaded file is empty"
            )

        with open(file_path, "wb") as f:
            f.write(content)

        # Process document
        result = process_document(file_path)

        if result.get("status") != "success":
            raise HTTPException(
                status_code=422,
                detail=result.get("error", "Processing failed")
            )

        invoice = result.get("invoice_data")

        duplicate = False

        if invoice:
            duplicate = is_duplicate_before_save(invoice)

        # Save document and invoice
        document_id = save_invoice_to_database(
            file_path,
            result
        )

        # Flag duplicate for manual review
        if duplicate:
            cursor = conn.cursor()
            cursor.execute("""
            UPDATE documents
            SET processing_status = 'needs_review'
            WHERE id = ?
            """, (document_id,))

            cursor.execute("""
            INSERT INTO validation_errors (
                document_id, error_message
            )
            VALUES (?, ?)
            """, (
                document_id,
                "Possible duplicate invoice detected"
            ))

            conn.commit()

        return {
            "document_id": document_id,
            "filename": file.filename,
            "status": "needs_review" if duplicate else "success",
            "duplicate_detected": duplicate,
            "invoice_data": invoice,
            "validation": result.get("validation", {})
        }

    except HTTPException:
        raise

    except Exception as error:
        raise HTTPException(
            status_code=500,
            detail=str(error)
        )

In [ ]:
import nest_asyncio
import threading
import uvicorn

nest_asyncio.apply()

config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=8000,
    log_level="info"
)

server = uvicorn.Server(config)

api_thread = threading.Thread(
    target=server.run,
    daemon=True
)

api_thread.start()

print("Updated API server starting!")

Updated API server starting!


INFO:     Started server process [3013]
INFO:     Waiting for application startup.
INFO:     Application startup complete.


In [ ]:
from google.colab.output import eval_js

public_url = eval_js(
    "google.colab.kernel.proxyPort(8000)"
)

print("API Documentation:", public_url + "/docs")

API Documentation: https://8000-m-s-usc1b3-v2cut3cufcnt-b.us-central1.prod.r.colab.dev/docs


In [ ]:
for route in app.routes:
    if route.path == "/upload":
        print("Upload endpoint:", route.path)
        print("Endpoint function:", route.endpoint.__name__)

Upload endpoint: /upload
Endpoint function: upload_document
Upload endpoint: /upload
Endpoint function: upload_document


In [ ]:
import inspect

for route in app.routes:
    if route.path == "/upload" and "POST" in route.methods:
        print(inspect.getsource(route.endpoint))

@app.post("/upload")
async def upload_document(file: UploadFile = File(...)):

    allowed_extensions = [
        ".pdf", ".png", ".jpg", ".jpeg",
        ".tiff", ".tif", ".xlsx", ".xls", ".csv"
    ]

    extension = os.path.splitext(file.filename)[1].lower()

    if extension not in allowed_extensions:
        raise HTTPException(
            status_code=400,
            detail="Unsupported file format"
        )

    safe_filename = os.path.basename(file.filename)
    file_path = os.path.join(UPLOAD_DIR, safe_filename)

    with open(file_path, "wb") as buffer:
        shutil.copyfileobj(file.file, buffer)

    try:
        result = process_document(file_path)
        document_id = save_invoice_to_database(file_path, result)

        return {
            "document_id": document_id,
            "filename": safe_filename,
            "processing_status": result["status"],
            "validation": result.get("validation")
        }

    except Exception as error:
        raise HTTPEx

In [ ]:
print("Server running:", server.started)
print("Port:", 8000)

Server running: False
Port: 8000


In [ ]:
import threading
import uvicorn

config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=8000,
    log_level="info"
)

server = uvicorn.Server(config)

api_thread = threading.Thread(
    target=server.run,
    daemon=True
)

api_thread.start()

INFO:     Started server process [3013]
INFO:     Waiting for application startup.
INFO:     Application startup complete.


In [ ]:
import time

time.sleep(3)
print("Server running:", server.started)

Server running: False


In [ ]:
import threading
import uvicorn
import time

config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=8001,
    log_level="info"
)

server2 = uvicorn.Server(config)

api_thread2 = threading.Thread(
    target=server2.run,
    daemon=True
)

api_thread2.start()

time.sleep(5)

print("Server running:", server2.started)

INFO:     Started server process [3013]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)


Server running: True


In [ ]:
from google.colab.output import eval_js

public_url_2 = eval_js(
    "google.colab.kernel.proxyPort(8001)"
)

print("Updated API Documentation:", public_url_2 + "/docs")

Updated API Documentation: https://8001-m-s-usc1b3-v2cut3cufcnt-b.us-central1.prod.r.colab.dev/docs


In [ ]:
for route in app.routes:
    if route.path == "/upload":
        print("Path:", route.path)
        print("Methods:", route.methods)
        print("Function:", route.endpoint.__name__)
        print("Response model:", route.response_model)
        print("-------------------")

Path: /upload
Methods: {'POST'}
Function: upload_document
Response model: None
-------------------
Path: /upload
Methods: {'POST'}
Function: upload_document
Response model: None
-------------------


In [ ]:

upload_routes = [
    route for route in app.router.routes
    if route.path == "/upload"
    and "POST" in route.methods
]

print("Routes found:", len(upload_routes))

# Keep the latest upload endpoint
latest_upload_route = upload_routes[-1]

app.router.routes[:] = [
    route for route in app.router.routes
    if not (
        route.path == "/upload"
        and "POST" in route.methods
    )
]

app.router.routes.append(latest_upload_route)

print("Remaining upload routes:", sum(
    1 for route in app.router.routes
    if route.path == "/upload"
    and "POST" in route.methods
))

Routes found: 2
Remaining upload routes: 1


In [ ]:
server2.should_exit = True

In [ ]:
import threading
import uvicorn
import time

config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=8001,
    log_level="info"
)

server2 = uvicorn.Server(config)

api_thread2 = threading.Thread(
    target=server2.run,
    daemon=True
)

api_thread2.start()

time.sleep(5)

print("Updated server running:", server2.started)

INFO:     Started server process [3013]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)


Updated server running: True


In [ ]:
import os

print("Project folder exists:",
      os.path.exists("document_intelligence"))

if os.path.exists("document_intelligence"):
    print(os.listdir("document_intelligence"))

Project folder exists: False


In [ ]:
import os

folders = [
    "document_intelligence",
    "document_intelligence/uploads",
    "document_intelligence/extracted_text",
    "document_intelligence/results",
    "document_intelligence/logs"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project folders recreated successfully!")

Project folders recreated successfully!


In [ ]:
import sqlite3

db_path = "document_intelligence/documents.db"

conn = sqlite3.connect(db_path, check_same_thread=False)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS documents (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    filename TEXT,
    file_type TEXT,
    processing_status TEXT,
    upload_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS invoices (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    document_id INTEGER,
    invoice_number TEXT,
    invoice_date TEXT,
    supplier_name TEXT,
    customer_name TEXT,
    subtotal REAL,
    tax REAL,
    total REAL,
    currency TEXT,
    FOREIGN KEY (document_id) REFERENCES documents(id)
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS validation_errors (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    document_id INTEGER,
    error_message TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS correction_history (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    document_id INTEGER,
    field_name TEXT,
    old_value TEXT,
    new_value TEXT,
    corrected_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS line_items (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    invoice_id INTEGER,
    description TEXT,
    quantity REAL,
    unit_price REAL,
    amount REAL
)
""")

conn.commit()

print("Database and all five tables restored successfully!")

Database and all five tables restored successfully!


In [ ]:
!pip install pymupdf pytesseract pillow pandas openpyxl opencv-python-headless fastapi uvicorn python-multipart nest-asyncio reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 MB 61.0 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 68.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 142.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [pymupdf]


In [ ]:
!apt-get install -y tesseract-ocr

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (5.3.4-1build5).
0 upgraded, 0 newly installed, 0 to remove and 0 not upgraded.


In [ ]:
import pytesseract

print("Tesseract version:")
print(pytesseract.get_tesseract_version())

Tesseract version:
5.3.4


In [ ]:
import os
import re
import json
import sqlite3
import fitz
import pytesseract
import pandas as pd
import cv2
import numpy as np

from PIL import Image
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import JSONResponse
from typing import Optional

print("All project imports loaded successfully!")

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


All project imports loaded successfully!


In [ ]:
import fitz

def extract_pdf_text(file_path):
    text = ""

    pdf = fitz.open(file_path)

    for page in pdf:
        text += page.get_text()

    pdf.close()

    return text

print("PDF extraction function restored!")

PDF extraction function restored!


In [ ]:
def extract_scanned_pdf_text(file_path):
    pdf = fitz.open(file_path)
    extracted_text = ""

    for page_number, page in enumerate(pdf):
        pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
        image = Image.frombytes(
            "RGB",
            [pix.width, pix.height],
            pix.samples
        )

        page_text = pytesseract.image_to_string(image)
        extracted_text += page_text + "\n"

    pdf.close()
    return extracted_text

print("Scanned PDF OCR function restored!")

Scanned PDF OCR function restored!


In [ ]:
def extract_image_text(file_path):
    image = Image.open(file_path)
    text = pytesseract.image_to_string(image)
    return text

print("Image OCR function restored!")

Image OCR function restored!


In [ ]:
def extract_spreadsheet_data(file_path):
    extension = os.path.splitext(file_path)[1].lower()

    if extension == ".csv":
        data = pd.read_csv(file_path)

    elif extension in [".xlsx", ".xls"]:
        data = pd.read_excel(
            file_path,
            sheet_name=None
        )

    else:
        raise ValueError("Unsupported spreadsheet format")

    return data

print("Spreadsheet parsing function restored!")

Spreadsheet parsing function restored!


In [ ]:
def extract_invoice_data(text):
    invoice = {
        "invoice_number": None,
        "invoice_date": None,
        "supplier_name": None,
        "customer_name": None,
        "subtotal": None,
        "tax": None,
        "total": None,
        "currency": None,
        "line_items": []
    }

    patterns = {
        "invoice_number": r"Invoice\s*(?:No|Number|#)\s*[:\-]?\s*([A-Z0-9\-]+)",
        "invoice_date": r"Date\s*[:\-]?\s*(\d{2}[/-]\d{2}[/-]\d{4})",
        "subtotal": r"Subtotal\s*[:\-]?\s*([\d,]+\.?\d*)",
        "tax": r"(?:Tax|VAT)\s*[:\-]?\s*([\d,]+\.?\d*)",
        "total": r"Total\s*[:\-]?\s*([\d,]+\.?\d*)"
    }

    for field, pattern in patterns.items():
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            invoice[field] = match.group(1)

    supplier = re.search(
        r"Supplier\s*[:\-]?\s*(.+)",
        text,
        re.IGNORECASE
    )

    customer = re.search(
        r"Customer\s*[:\-]?\s*(.+)",
        text,
        re.IGNORECASE
    )

    if supplier:
        invoice["supplier_name"] = supplier.group(1).strip()

    if customer:
        invoice["customer_name"] = customer.group(1).strip()

    currency = re.search(r"\b(BDT|USD|EUR|GBP)\b", text, re.IGNORECASE)

    if currency:
        invoice["currency"] = currency.group(1).upper()

    return invoice


print("Invoice extraction function restored!")

Invoice extraction function restored!


In [ ]:
def validate_invoice(invoice):
    errors = []

    required_fields = [
        "invoice_number",
        "invoice_date",
        "supplier_name",
        "customer_name",
        "subtotal",
        "tax",
        "total"
    ]

    for field in required_fields:
        if not invoice.get(field):
            errors.append(f"Missing required field: {field}")

    try:
        subtotal = float(str(invoice["subtotal"]).replace(",", ""))
        tax = float(str(invoice["tax"]).replace(",", ""))
        total = float(str(invoice["total"]).replace(",", ""))

        if abs((subtotal + tax) - total) > 0.01:
            errors.append("Subtotal + tax does not equal total")

    except (ValueError, TypeError, KeyError):
        errors.append("Invalid or missing numeric invoice values")

    return {
        "validation_status": "passed" if not errors else "failed",
        "errors": errors
    }

print("Invoice validation function restored!")

Invoice validation function restored!


In [ ]:
def process_document(file_path):
    extension = os.path.splitext(file_path)[1].lower()

    try:
        if extension == ".pdf":
            text = extract_pdf_text(file_path)

            if not text.strip():
                text = extract_scanned_pdf_text(file_path)

        elif extension in [".jpg", ".jpeg", ".png", ".tiff", ".tif"]:
            text = extract_image_text(file_path)

        elif extension in [".xlsx", ".xls", ".csv"]:
            data = extract_spreadsheet_data(file_path)

            return {
                "status": "success",
                "document_type": "spreadsheet",
                "spreadsheet_data": (
                    data.to_dict()
                    if isinstance(data, pd.DataFrame)
                    else {
                        sheet: df.to_dict()
                        for sheet, df in data.items()
                    }
                )
            }

        else:
            raise ValueError("Unsupported file format")

        invoice = extract_invoice_data(text)
        validation = validate_invoice(invoice)

        return {
            "status": "success",
            "extracted_text": text,
            "invoice_data": invoice,
            "validation": validation
        }

    except Exception as e:
        return {
            "status": "failed",
            "error": str(e)
        }

print("Document processing function restored!")

Document processing function restored!


In [ ]:
def save_invoice_to_database(file_path, result):
    cursor = conn.cursor()

    cursor.execute("""
    INSERT INTO documents (filename, file_type, processing_status)
    VALUES (?, ?, ?)
    """, (
        os.path.basename(file_path),
        os.path.splitext(file_path)[1].lower(),
        result.get("status", "failed")
    ))

    document_id = cursor.lastrowid
    invoice = result.get("invoice_data")

    if invoice:
        def safe_float(value):
            try:
                return float(str(value).replace(",", "")) if value is not None else None
            except (ValueError, TypeError):
                return None

        cursor.execute("""
        INSERT INTO invoices (
            document_id, invoice_number, invoice_date,
            supplier_name, customer_name, subtotal,
            tax, total, currency
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            document_id,
            invoice.get("invoice_number"),
            invoice.get("invoice_date"),
            invoice.get("supplier_name"),
            invoice.get("customer_name"),
            safe_float(invoice.get("subtotal")),
            safe_float(invoice.get("tax")),
            safe_float(invoice.get("total")),
            invoice.get("currency")
        ))

        invoice_id = cursor.lastrowid

        for item in invoice.get("line_items", []):
            cursor.execute("""
            INSERT INTO line_items (
                invoice_id, description, quantity, unit_price, amount
            )
            VALUES (?, ?, ?, ?, ?)
            """, (
                invoice_id,
                item.get("description"),
                safe_float(item.get("quantity")),
                safe_float(item.get("unit_price")),
                safe_float(item.get("amount"))
            ))

    for error in result.get("validation", {}).get("errors", []):
        cursor.execute("""
        INSERT INTO validation_errors (document_id, error_message)
        VALUES (?, ?)
        """, (document_id, error))

    conn.commit()
    return document_id


print("Database saving function restored!")

Database saving function restored!


In [ ]:
def check_duplicate_invoice(invoice_number, supplier_name):
    cursor = conn.cursor()

    cursor.execute("""
    SELECT id, document_id, invoice_number, supplier_name
    FROM invoices
    WHERE LOWER(invoice_number) = LOWER(?)
    AND LOWER(supplier_name) = LOWER(?)
    """, (invoice_number, supplier_name))

    matches = cursor.fetchall()

    return {
        "is_duplicate": len(matches) > 0,
        "matching_records": matches
    }


def is_duplicate_before_save(invoice):
    invoice_number = invoice.get("invoice_number")
    supplier_name = invoice.get("supplier_name")

    if not invoice_number or not supplier_name:
        return False

    cursor = conn.cursor()

    cursor.execute("""
    SELECT COUNT(*)
    FROM invoices
    WHERE LOWER(invoice_number) = LOWER(?)
    AND LOWER(supplier_name) = LOWER(?)
    """, (invoice_number, supplier_name))

    return cursor.fetchone()[0] > 0


print("Duplicate detection functions restored!")

Duplicate detection functions restored!


In [ ]:
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import JSONResponse
import uuid

app = FastAPI(title="AI Document Intelligence System")


@app.get("/")
def home():
    return {"message": "Document Intelligence API is running"}


@app.get("/documents")
def get_documents():
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM documents")
    return cursor.fetchall()


@app.get("/invoices")
def get_invoices():
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM invoices")
    return cursor.fetchall()


@app.post("/upload")
async def upload_document(file: UploadFile = File(...)):
    allowed = [".pdf", ".jpg", ".jpeg", ".png",
               ".tiff", ".tif", ".xlsx", ".xls", ".csv"]

    extension = os.path.splitext(file.filename)[1].lower()

    if extension not in allowed:
        raise HTTPException(
            status_code=400,
            detail="Unsupported file format"
        )

    unique_name = str(uuid.uuid4()) + extension
    file_path = os.path.join("document_intelligence/uploads", unique_name)

    with open(file_path, "wb") as f:
        f.write(await file.read())

    result = process_document(file_path)

    if result.get("status") == "failed":
        return {
            "status": "failed",
            "error": result.get("error")
        }

    invoice = result.get("invoice_data")

    duplicate = (
        is_duplicate_before_save(invoice)
        if invoice else False
    )

    document_id = save_invoice_to_database(file_path, result)

    if duplicate:
        cursor = conn.cursor()
        cursor.execute("""
        UPDATE documents
        SET processing_status = 'needs_review'
        WHERE id = ?
        """, (document_id,))

        cursor.execute("""
        INSERT INTO validation_errors (document_id, error_message)
        VALUES (?, ?)
        """, (document_id, "Duplicate invoice detected"))

        conn.commit()

    return {
        "document_id": document_id,
        "filename": file.filename,
        "status": "needs_review" if duplicate else result["status"],
        "duplicate_detected": duplicate,
        "invoice_data": invoice,
        "validation": result.get("validation")
    }


@app.get("/search")
def search_documents(keyword: str):
    cursor = conn.cursor()

    cursor.execute("""
    SELECT * FROM invoices
    WHERE invoice_number LIKE ?
    OR supplier_name LIKE ?
    OR customer_name LIKE ?
    """, (f"%{keyword}%", f"%{keyword}%", f"%{keyword}%"))

    return cursor.fetchall()


@app.put("/invoices/{invoice_id}/correct")
def correct_invoice(invoice_id: int, field_name: str, new_value: str):
    allowed_fields = [
        "invoice_number", "invoice_date", "supplier_name",
        "customer_name", "subtotal", "tax", "total", "currency"
    ]

    if field_name not in allowed_fields:
        raise HTTPException(status_code=400, detail="Invalid field")

    cursor = conn.cursor()

    cursor.execute(
        f"SELECT {field_name}, document_id FROM invoices WHERE id = ?",
        (invoice_id,)
    )
    record = cursor.fetchone()

    if not record:
        raise HTTPException(status_code=404, detail="Invoice not found")

    old_value, document_id = record

    cursor.execute(
        f"UPDATE invoices SET {field_name} = ? WHERE id = ?",
        (new_value, invoice_id)
    )

    cursor.execute("""
    INSERT INTO correction_history
    (document_id, field_name, old_value, new_value)
    VALUES (?, ?, ?, ?)
    """, (document_id, field_name, str(old_value), str(new_value)))

    conn.commit()

    return {"message": "Invoice corrected successfully"}


@app.get("/documents/{document_id}/status")
def get_document_status(document_id: int):
    cursor = conn.cursor()

    cursor.execute("""
    SELECT id, filename, processing_status
    FROM documents WHERE id = ?
    """, (document_id,))

    record = cursor.fetchone()

    if not record:
        raise HTTPException(status_code=404, detail="Document not found")

    return {
        "document_id": record[0],
        "filename": record[1],
        "status": record[2]
    }


@app.post("/retry/{document_id}")
def retry_document(document_id: int):
    cursor = conn.cursor()

    cursor.execute("""
    SELECT filename FROM documents WHERE id = ?
    """, (document_id,))

    record = cursor.fetchone()

    if not record:
        raise HTTPException(status_code=404, detail="Document not found")

    file_path = os.path.join(
        "document_intelligence/uploads",
        record[0]
    )

    if not os.path.exists(file_path):
        raise HTTPException(status_code=404, detail="Uploaded file not found")

    result = process_document(file_path)

    return {
        "document_id": document_id,
        "retry_result": result
    }


print("FastAPI application and endpoints restored!")

FastAPI application and endpoints restored!


In [ ]:
import nest_asyncio
import threading
import uvicorn
import time

nest_asyncio.apply()

config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=8000,
    log_level="info"
)

server = uvicorn.Server(config)

api_thread = threading.Thread(
    target=server.run,
    daemon=True
)

api_thread.start()

time.sleep(5)

print("Server running:", server.started)

INFO:     Started server process [1511]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Server running: True


In [ ]:
from google.colab.output import eval_js

public_url = eval_js(
    "google.colab.kernel.proxyPort(8000)"
)

print("API Documentation:", public_url + "/docs")

API Documentation: https://8000-m-s-usc1c9-2xw777i57big4-c.us-central1.prod.r.colab.dev/docs


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import time

# Stop the temporary API server before changing project location
server.should_exit = True
time.sleep(3)

# Create permanent project folder
project_root = "/content/drive/MyDrive/AI_Document_Intelligence_System"

os.makedirs(project_root, exist_ok=True)

os.chdir(project_root)

folders = [
    "document_intelligence",
    "document_intelligence/uploads",
    "document_intelligence/extracted_text",
    "document_intelligence/results",
    "document_intelligence/logs"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Permanent project folder created!")
print("Project location:", os.getcwd())

INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [1511]


Permanent project folder created!
Project location: /content/drive/MyDrive/AI_Document_Intelligence_System


In [ ]:
db_path = os.path.join(
    project_root,
    "document_intelligence",
    "documents.db"
)

conn = sqlite3.connect(
    db_path,
    check_same_thread=False
)

cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS documents (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    filename TEXT,
    file_type TEXT,
    processing_status TEXT,
    upload_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS invoices (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    document_id INTEGER,
    invoice_number TEXT,
    invoice_date TEXT,
    supplier_name TEXT,
    customer_name TEXT,
    subtotal REAL,
    tax REAL,
    total REAL,
    currency TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS validation_errors (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    document_id INTEGER,
    error_message TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS correction_history (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    document_id INTEGER,
    field_name TEXT,
    old_value TEXT,
    new_value TEXT,
    corrected_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS line_items (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    invoice_id INTEGER,
    description TEXT,
    quantity REAL,
    unit_price REAL,
    amount REAL
)
""")

conn.commit()

print("Permanent database connected successfully!")
print("Database location:", db_path)

Permanent database connected successfully!
Database location: /content/drive/MyDrive/AI_Document_Intelligence_System/document_intelligence/documents.db


In [ ]:
print("Database exists:", os.path.exists(db_path))
print("Database size:", round(os.path.getsize(db_path) / 1024, 2), "KB")

Database exists: True
Database size: 28.0 KB


In [ ]:
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS extracted_text (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    document_id INTEGER,
    extracted_content TEXT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (document_id) REFERENCES documents(id)
)
""")

conn.commit()

print("Extracted text table created successfully!")

Extracted text table created successfully!


In [ ]:
def save_extracted_text(document_id, text):
    cursor = conn.cursor()

    cursor.execute("""
    INSERT INTO extracted_text (
        document_id,
        extracted_content
    )
    VALUES (?, ?)
    """, (document_id, text))

    conn.commit()

    return cursor.lastrowid


print("Extracted text saving function created!")

Extracted text saving function created!


In [ ]:
def save_invoice_to_database(file_path, result):
    cursor = conn.cursor()

    cursor.execute("""
    INSERT INTO documents (filename, file_type, processing_status)
    VALUES (?, ?, ?)
    """, (
        os.path.basename(file_path),
        os.path.splitext(file_path)[1].lower(),
        result.get("status", "failed")
    ))

    document_id = cursor.lastrowid

    # Save extracted text
    extracted_text = result.get("extracted_text", "")

    if extracted_text:
        save_extracted_text(document_id, extracted_text)

    # Save invoice information
    invoice = result.get("invoice_data")

    if invoice:
        def safe_float(value):
            try:
                return float(str(value).replace(",", "")) if value is not None else None
            except (ValueError, TypeError):
                return None

        cursor.execute("""
        INSERT INTO invoices (
            document_id, invoice_number, invoice_date,
            supplier_name, customer_name, subtotal,
            tax, total, currency
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            document_id,
            invoice.get("invoice_number"),
            invoice.get("invoice_date"),
            invoice.get("supplier_name"),
            invoice.get("customer_name"),
            safe_float(invoice.get("subtotal")),
            safe_float(invoice.get("tax")),
            safe_float(invoice.get("total")),
            invoice.get("currency")
        ))

        invoice_id = cursor.lastrowid

        for item in invoice.get("line_items", []):
            cursor.execute("""
            INSERT INTO line_items (
                invoice_id, description, quantity, unit_price, amount
            )
            VALUES (?, ?, ?, ?, ?)
            """, (
                invoice_id,
                item.get("description"),
                safe_float(item.get("quantity")),
                safe_float(item.get("unit_price")),
                safe_float(item.get("amount"))
            ))

    # Save validation errors
    for error in result.get("validation", {}).get("errors", []):
        cursor.execute("""
        INSERT INTO validation_errors (document_id, error_message)
        VALUES (?, ?)
        """, (document_id, error))

    conn.commit()

    return document_id


print("Updated database-saving function ready!")

Updated database-saving function ready!


In [ ]:
test_text = """
Invoice Number: TEST-001
Supplier: Demo Technologies
Customer: Sample Company
Subtotal: 500
Tax: 75
Total: 575
"""

test_result = {
    "status": "success",
    "extracted_text": test_text,
    "invoice_data": {
        "invoice_number": "TEST-001",
        "invoice_date": "05/11/2026",
        "supplier_name": "Demo Technologies",
        "customer_name": "Sample Company",
        "subtotal": "500",
        "tax": "75",
        "total": "575",
        "currency": "BDT",
        "line_items": []
    },
    "validation": {
        "validation_status": "passed",
        "errors": []
    }
}

test_path = "test_invoice.pdf"

test_document_id = save_invoice_to_database(
    test_path,
    test_result
)

print("Test document ID:", test_document_id)

Test document ID: 1


In [ ]:
cursor = conn.cursor()

cursor.execute("""
SELECT document_id, extracted_content
FROM extracted_text
WHERE document_id = ?
""", (test_document_id,))

saved_text = cursor.fetchone()

print("Saved document ID:", saved_text[0])
print("Extracted text:")
print(saved_text[1])

Saved document ID: 1
Extracted text:

Invoice Number: TEST-001
Supplier: Demo Technologies
Customer: Sample Company
Subtotal: 500
Tax: 75
Total: 575



In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas

sample_path = os.path.join(
    project_root,
    "sample_invoice_v2.pdf"
)

pdf = canvas.Canvas(sample_path, pagesize=A4)

pdf.drawString(100, 750, "INVOICE")
pdf.drawString(100, 710, "Invoice Number: UPLOAD-TEST-002")
pdf.drawString(100, 680, "Date: 05/11/2026")
pdf.drawString(100, 650, "Supplier: Demo Systems")
pdf.drawString(100, 620, "Customer: Test Buyer")

pdf.drawString(100, 570, "Description    Quantity    Unit Price")
pdf.drawString(100, 540, "Keyboard       2           100")
pdf.drawString(100, 510, "Subtotal: 200")
pdf.drawString(100, 480, "Tax: 30")
pdf.drawString(100, 450, "Total: 230 BDT")

pdf.save()

print("Test invoice PDF created:", sample_path)

Test invoice PDF created: /content/drive/MyDrive/AI_Document_Intelligence_System/sample_invoice_v2.pdf


In [ ]:
config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=8000,
    log_level="info"
)

server = uvicorn.Server(config)

api_thread = threading.Thread(
    target=server.run,
    daemon=True
)

api_thread.start()

time.sleep(5)

print("Server running:", server.started)

INFO:     Started server process [1511]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Server running: True


In [ ]:
from google.colab import files

files.download(sample_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
public_url = eval_js(
    "google.colab.kernel.proxyPort(8000)"
)

print("API Documentation:", public_url + "/docs")

API Documentation: https://8000-m-s-usc1c9-2xw777i57big4-c.us-central1.prod.r.colab.dev/docs


In [ ]:
def extract_invoice_data(text):
    invoice = {
        "invoice_number": None,
        "invoice_date": None,
        "supplier_name": None,
        "customer_name": None,
        "subtotal": None,
        "tax": None,
        "total": None,
        "currency": None,
        "line_items": []
    }

    patterns = {
        "invoice_number": r"Invoice\s*(?:No|Number|#)\s*[:\-]?\s*([A-Z0-9\-]+)",
        "invoice_date": r"Date\s*[:\-]?\s*(\d{2}[/-]\d{2}[/-]\d{4})",
        "subtotal": r"\bSubtotal\b\s*[:\-]?\s*([\d,]+\.?\d*)",
        "tax": r"\b(?:Tax|VAT)\b\s*[:\-]?\s*([\d,]+\.?\d*)",
        "total": r"(?<!Sub)\bTotal\b\s*[:\-]?\s*([\d,]+\.?\d*)"
    }

    for field, pattern in patterns.items():
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            invoice[field] = match.group(1)

    supplier = re.search(
        r"Supplier\s*[:\-]?\s*(.+)",
        text,
        re.IGNORECASE
    )

    customer = re.search(
        r"Customer\s*[:\-]?\s*(.+)",
        text,
        re.IGNORECASE
    )

    if supplier:
        invoice["supplier_name"] = supplier.group(1).strip()

    if customer:
        invoice["customer_name"] = customer.group(1).strip()

    currency = re.search(
        r"\b(BDT|USD|EUR|GBP)\b",
        text,
        re.IGNORECASE
    )

    if currency:
        invoice["currency"] = currency.group(1).upper()

    return invoice


print("Updated invoice extraction function ready!")

Updated invoice extraction function ready!


In [ ]:
test_invoice_text = """
INVOICE
Invoice Number: UPLOAD-TEST-002
Date: 05/11/2026
Supplier: Demo Systems
Customer: Test Buyer
Subtotal: 200
Tax: 30
Total: 230 BDT
"""

test_extraction = extract_invoice_data(test_invoice_text)

print(json.dumps(test_extraction, indent=4))

{
    "invoice_number": "UPLOAD-TEST-002",
    "invoice_date": "05/11/2026",
    "supplier_name": "Demo Systems",
    "customer_name": "Test Buyer",
    "subtotal": "200",
    "tax": "30",
    "total": "230",
    "currency": "BDT",
    "line_items": []
}


In [ ]:
validation_result = validate_invoice(test_extraction)
print(json.dumps(validation_result, indent=4))

{
    "validation_status": "passed",
    "errors": []
}


In [ ]:
validation_result = validate_invoice(test_extraction)

print(json.dumps(validation_result, indent=4))

{
    "validation_status": "passed",
    "errors": []
}


In [ ]:
cursor = conn.cursor()

cursor.execute("""
SELECT document_id, extracted_content
FROM extracted_text
WHERE document_id = 2
""")

saved_record = cursor.fetchone()

print(saved_record)

(2, 'INVOICE\nInvoice Number: UPLOAD-TEST-002\nDate: 05/11/2026\nSupplier: Demo Systems\nCustomer: Test Buyer\nDescription    Quantity    Unit Price\nKeyboard       2           100\nSubtotal: 200\nTax: 30\nTotal: 230 BDT\n')


In [ ]:
@app.post("/upload")
async def upload_document(file: UploadFile = File(...)):
    allowed = [".pdf", ".jpg", ".jpeg", ".png", ".tiff",
               ".tif", ".xlsx", ".xls", ".csv"]

    extension = os.path.splitext(file.filename)[1].lower()

    if extension not in allowed:
        raise HTTPException(status_code=400,
                            detail="Unsupported file format")

    unique_name = str(uuid.uuid4()) + extension
    file_path = os.path.join(
        "document_intelligence/uploads", unique_name
    )

    with open(file_path, "wb") as f:
        f.write(await file.read())

    result = process_document(file_path)

    if result.get("status") == "failed":
        return {"status": "failed", "error": result.get("error")}

    invoice = result.get("invoice_data")
    duplicate = is_duplicate_before_save(invoice) if invoice else False

    validation_errors = result.get("validation", {}).get("errors", [])

    needs_review = duplicate or bool(validation_errors)

    document_id = save_invoice_to_database(file_path, result)

    if needs_review:
        cursor = conn.cursor()
        cursor.execute(
            "UPDATE documents SET processing_status=? WHERE id=?",
            ("needs_review", document_id)
        )

    if duplicate:
        cursor.execute(
            """INSERT INTO validation_errors
            (document_id, error_message) VALUES (?, ?)""",
            (document_id, "Duplicate invoice detected")
        )

    conn.commit()

    return {
        "document_id": document_id,
        "filename": file.filename,
        "status": "needs_review" if needs_review else "success",
        "duplicate_detected": duplicate,
        "invoice_data": invoice,
        "validation": result.get("validation")
    }

print("Updated upload endpoint is ready!")

Updated upload endpoint is ready!


In [ ]:
server.should_exit = True
time.sleep(3)

config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=8000,
    log_level="info"
)

server = uvicorn.Server(config)

api_thread = threading.Thread(target=server.run, daemon=True)
api_thread.start()

time.sleep(5)

print("Server running:", server.started)

INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [1511]
INFO:     Started server process [1511]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Server running: True


In [ ]:
from google.colab.output import eval_js

public_url = eval_js("google.colab.kernel.proxyPort(8000)")

print("API Documentation:", public_url + "/docs")

API Documentation: https://8000-m-s-usc1c9-2xw777i57big4-c.us-central1.prod.r.colab.dev/docs


In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas

invalid_invoice_path = os.path.join(
    project_root, "sample_invalid_invoice.pdf"
)

pdf = canvas.Canvas(invalid_invoice_path, pagesize=A4)

pdf.drawString(100, 750, "INVOICE")
pdf.drawString(100, 710, "Invoice Number: INVALID-TEST-001")
pdf.drawString(100, 680, "Date: 05/11/2026")
pdf.drawString(100, 650, "Supplier: Test Supplies")
pdf.drawString(100, 620, "Customer: Demo Buyer")
pdf.drawString(100, 570, "Subtotal: 200")
pdf.drawString(100, 540, "Tax: 30")
pdf.drawString(100, 510, "Total: 250 BDT")

pdf.save()

print("Test invoice created:", invalid_invoice_path)

Test invoice created: /content/drive/MyDrive/AI_Document_Intelligence_System/sample_invalid_invoice.pdf


In [ ]:
from google.colab import files

files.download(invalid_invoice_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
upload_routes = [
    route for route in app.routes
    if getattr(route, "path", None) == "/upload"
    and "POST" in getattr(route, "methods", [])
]

print("Number of upload routes:", len(upload_routes))

Number of upload routes: 2


In [ ]:
upload_routes = [
    route for route in app.routes
    if getattr(route, "path", None) == "/upload"
    and "POST" in getattr(route, "methods", [])
]

latest_upload_route = upload_routes[-1]

app.router.routes = [
    route for route in app.router.routes
    if route not in upload_routes
]

app.router.routes.append(latest_upload_route)

print("Remaining upload routes:",
      sum(1 for route in app.routes
          if getattr(route, "path", None) == "/upload"
          and "POST" in getattr(route, "methods", [])))

Remaining upload routes: 1


In [ ]:
server.should_exit = True
time.sleep(3)

config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=8000,
    log_level="info"
)

server = uvicorn.Server(config)

api_thread = threading.Thread(target=server.run, daemon=True)
api_thread.start()

time.sleep(5)

print("Server running:", server.started)

INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [1511]
INFO:     Started server process [1511]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Server running: True


In [ ]:
cursor = conn.cursor()

cursor.execute("""
SELECT id, filename, processing_status
FROM documents
WHERE id = 4
""")

print(cursor.fetchone())

(4, '45117bca-f5b5-4ded-a951-4ab2e99b9861.pdf', 'needs_review')


In [ ]:
cursor = conn.cursor()

for table in ["documents", "invoices", "line_items",
              "validation_errors", "extracted_text"]:
    cursor.execute(f"PRAGMA table_info({table})")
    print("\n", table)
    for column in cursor.fetchall():
        print(column[1], "-", column[2])


 documents
id - INTEGER
filename - TEXT
file_type - TEXT
processing_status - TEXT
upload_date - TIMESTAMP

 invoices
id - INTEGER
document_id - INTEGER
invoice_number - TEXT
invoice_date - TEXT
supplier_name - TEXT
customer_name - TEXT
subtotal - REAL
tax - REAL
total - REAL
currency - TEXT

 line_items
id - INTEGER
invoice_id - INTEGER
description - TEXT
quantity - REAL
unit_price - REAL
amount - REAL

 validation_errors
id - INTEGER
document_id - INTEGER
error_message - TEXT

 extracted_text
id - INTEGER
document_id - INTEGER
extracted_content - TEXT
created_at - TIMESTAMP


In [ ]:
def update_existing_document(document_id, file_path, result):
    cursor = conn.cursor()

    status = result.get("status", "failed")
    invoice = result.get("invoice_data")

    validation_errors = result.get("validation", {}).get("errors", [])

    duplicate = is_duplicate_before_save(invoice) if invoice else False

    if duplicate:
        validation_errors.append("Duplicate invoice detected")

    if validation_errors:
        status = "needs_review"

    cursor.execute("""
        UPDATE documents
        SET processing_status = ?
        WHERE id = ?
    """, (status, document_id))

    # Update extracted text
    if result.get("extracted_text"):
        cursor.execute("""
            UPDATE extracted_text
            SET extracted_content = ?
            WHERE document_id = ?
        """, (result["extracted_text"], document_id))

    conn.commit()

    return {
        "document_id": document_id,
        "status": status,
        "validation_errors": validation_errors
    }

print("Document update function ready!")

Document update function ready!


In [ ]:
@app.post("/retry/{document_id}")
def retry_document(document_id: int):

    cursor = conn.cursor()

    cursor.execute(
        "SELECT filename FROM documents WHERE id = ?",
        (document_id,)
    )

    record = cursor.fetchone()

    if not record:
        raise HTTPException(
            status_code=404,
            detail="Document not found"
        )

    file_path = os.path.join(
        "document_intelligence/uploads",
        record[0]
    )

    if not os.path.exists(file_path):
        raise HTTPException(
            status_code=404,
            detail="Uploaded file not found"
        )

    result = process_document(file_path)

    if result.get("status") == "failed":
        cursor.execute("""
            UPDATE documents
            SET processing_status = 'failed'
            WHERE id = ?
        """, (document_id,))
        conn.commit()

        return {
            "document_id": document_id,
            "status": "failed",
            "error": result.get("error")
        }

    update_result = update_existing_document(
        document_id,
        file_path,
        result
    )

    return update_result

print("Updated retry endpoint is ready!")

Updated retry endpoint is ready!


In [ ]:
retry_routes = [
    route for route in app.routes
    if getattr(route, "path", None) == "/retry/{document_id}"
    and "POST" in getattr(route, "methods", [])
]

print("Number of retry routes:", len(retry_routes))

Number of retry routes: 2


In [ ]:
retry_routes = [
    route for route in app.routes
    if getattr(route, "path", None) == "/retry/{document_id}"
    and "POST" in getattr(route, "methods", [])
]

latest_retry_route = retry_routes[-1]

app.router.routes = [
    route for route in app.router.routes
    if route not in retry_routes
]

app.router.routes.append(latest_retry_route)

print("Remaining retry routes:",
      sum(1 for route in app.routes
          if getattr(route, "path", None) == "/retry/{document_id}"
          and "POST" in getattr(route, "methods", [])))

Remaining retry routes: 1


In [ ]:
server.should_exit = True
time.sleep(3)

config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=8000,
    log_level="info"
)

server = uvicorn.Server(config)

api_thread = threading.Thread(target=server.run, daemon=True)
api_thread.start()

time.sleep(5)

print("Server running:", server.started)

INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [1511]
INFO:     Started server process [1511]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Server running: True


In [ ]:
def is_duplicate_before_save(invoice, exclude_document_id=None):
    invoice_number = invoice.get("invoice_number")
    supplier_name = invoice.get("supplier_name")

    if not invoice_number or not supplier_name:
        return False

    cursor = conn.cursor()

    query = """
        SELECT COUNT(*)
        FROM invoices
        WHERE LOWER(invoice_number) = LOWER(?)
        AND LOWER(supplier_name) = LOWER(?)
    """

    params = [invoice_number, supplier_name]

    if exclude_document_id is not None:
        query += " AND document_id != ?"
        params.append(exclude_document_id)

    cursor.execute(query, params)

    return cursor.fetchone()[0] > 0

print("Updated duplicate detection function ready!")

Updated duplicate detection function ready!


In [ ]:
def update_existing_document(document_id, file_path, result):
    cursor = conn.cursor()

    status = result.get("status", "failed")
    invoice = result.get("invoice_data")
    validation_errors = result.get("validation", {}).get("errors", [])

    duplicate = (
        is_duplicate_before_save(invoice, exclude_document_id=document_id)
        if invoice else False
    )

    if duplicate:
        validation_errors.append("Duplicate invoice detected")

    if validation_errors:
        status = "needs_review"

    cursor.execute("""
        UPDATE documents
        SET processing_status = ?
        WHERE id = ?
    """, (status, document_id))

    if result.get("extracted_text"):
        cursor.execute("""
            UPDATE extracted_text
            SET extracted_content = ?
            WHERE document_id = ?
        """, (result["extracted_text"], document_id))

    conn.commit()

    return {
        "document_id": document_id,
        "status": status,
        "duplicate_detected": duplicate,
        "validation_errors": validation_errors
    }

print("Retry update function improved!")

Retry update function improved!


In [ ]:
cursor = conn.cursor()

cursor.execute("""
SELECT COUNT(*)
FROM documents
WHERE id = 4
""")

print("Document 4 record count:", cursor.fetchone()[0])

Document 4 record count: 1


In [ ]:
import pandas as pd
import os

sample_invoice_data = {
    "Description": ["Keyboard", "Mouse"],
    "Quantity": [2, 3],
    "Unit Price": [100, 50],
    "Amount": [200, 150]
}

sample_df = pd.DataFrame(sample_invoice_data)

spreadsheet_path = os.path.join(
    project_root, "sample_spreadsheet_invoice.xlsx"
)

with pd.ExcelWriter(spreadsheet_path) as writer:
    sample_df.to_excel(
        writer,
        sheet_name="Invoice Items",
        index=False
    )

    summary_df = pd.DataFrame({
        "Field": [
            "Invoice Number",
            "Date",
            "Supplier",
            "Customer",
            "Subtotal",
            "Tax",
            "Total",
            "Currency"
        ],
        "Value": [
            "XLS-TEST-001",
            "05/11/2026",
            "Spreadsheet Supplies",
            "Demo Buyer",
            350,
            52.5,
            402.5,
            "BDT"
        ]
    })

    summary_df.to_excel(
        writer,
        sheet_name="Invoice Summary",
        index=False
    )

print("Sample spreadsheet invoice created!")

Sample spreadsheet invoice created!


In [ ]:
spreadsheet_data = extract_spreadsheet_data(spreadsheet_path)

print("Available sheets:", list(spreadsheet_data.keys()))

for sheet_name, df in spreadsheet_data.items():
    print("\nSheet:", sheet_name)
    display(df)

Available sheets: ['Invoice Items', 'Invoice Summary']

Sheet: Invoice Items


,Description,Quantity,Unit Price,Amount
0,Keyboard,2,100,200
1,Mouse,3,50,150



Sheet: Invoice Summary


,Field,Value
0,Invoice Number,XLS-TEST-001
1,Date,05/11/2026
2,Supplier,Spreadsheet Supplies
3,Customer,Demo Buyer
4,Subtotal,350
5,Tax,52.5
6,Total,402.5
7,Currency,BDT


In [ ]:
def extract_spreadsheet_invoice(file_path):

    sheets = pd.read_excel(file_path, sheet_name=None)

    invoice = {
        "invoice_number": None,
        "invoice_date": None,
        "supplier_name": None,
        "customer_name": None,
        "subtotal": None,
        "tax": None,
        "total": None,
        "currency": None,
        "line_items": []
    }

    # Read invoice summary
    for sheet_name, df in sheets.items():

        if "Field" in df.columns and "Value" in df.columns:

            summary = dict(zip(df["Field"], df["Value"]))

            field_mapping = {
                "Invoice Number": "invoice_number",
                "Date": "invoice_date",
                "Supplier": "supplier_name",
                "Customer": "customer_name",
                "Subtotal": "subtotal",
                "Tax": "tax",
                "Total": "total",
                "Currency": "currency"
            }

            for source, target in field_mapping.items():
                value = summary.get(source)

                if pd.notna(value):
                    invoice[target] = str(value)

        # Read line items
        if all(col in df.columns for col in
               ["Description", "Quantity", "Unit Price"]):

            for _, row in df.iterrows():

                quantity = float(row["Quantity"])
                unit_price = float(row["Unit Price"])

                invoice["line_items"].append({
                    "description": str(row["Description"]),
                    "quantity": quantity,
                    "unit_price": unit_price,
                    "amount": quantity * unit_price
                })

    return invoice


print("Spreadsheet invoice parser ready!")

Spreadsheet invoice parser ready!


In [ ]:
spreadsheet_invoice = extract_spreadsheet_invoice(
    spreadsheet_path
)

print(json.dumps(spreadsheet_invoice, indent=4))

{
    "invoice_number": "XLS-TEST-001",
    "invoice_date": "05/11/2026",
    "supplier_name": "Spreadsheet Supplies",
    "customer_name": "Demo Buyer",
    "subtotal": "350",
    "tax": "52.5",
    "total": "402.5",
    "currency": "BDT",
    "line_items": [
        {
            "description": "Keyboard",
            "quantity": 2.0,
            "unit_price": 100.0,
            "amount": 200.0
        },
        {
            "description": "Mouse",
            "quantity": 3.0,
            "unit_price": 50.0,
            "amount": 150.0
        }
    ]
}


In [ ]:
spreadsheet_validation = validate_invoice(spreadsheet_invoice)

print(json.dumps(spreadsheet_validation, indent=4))

{
    "validation_status": "passed",
    "errors": []
}


In [ ]:
def process_document(file_path):
    extension = os.path.splitext(file_path)[1].lower()

    try:
        if extension == ".pdf":
            text = extract_pdf_text(file_path)

            if not text.strip():
                text = extract_scanned_pdf_text(file_path)

        elif extension in [".jpg", ".jpeg", ".png", ".tiff", ".tif"]:
            text = extract_image_text(file_path)

        elif extension in [".xlsx", ".xls", ".csv"]:

            if extension == ".csv":
                data = pd.read_csv(file_path)
                invoice = {
                    "invoice_number": None,
                    "invoice_date": None,
                    "supplier_name": None,
                    "customer_name": None,
                    "subtotal": None,
                    "tax": None,
                    "total": None,
                    "currency": None,
                    "line_items": []
                }
            else:
                invoice = extract_spreadsheet_invoice(file_path)

            validation = validate_invoice(invoice)

            return {
                "status": "success",
                "document_type": "spreadsheet",
                "invoice_data": invoice,
                "validation": validation
            }

        else:
            raise ValueError("Unsupported file format")

        invoice = extract_invoice_data(text)
        validation = validate_invoice(invoice)

        return {
            "status": "success",
            "extracted_text": text,
            "invoice_data": invoice,
            "validation": validation
        }

    except Exception as e:
        return {
            "status": "failed",
            "error": str(e)
        }

print("Updated document processing pipeline ready!")

Updated document processing pipeline ready!


In [ ]:
spreadsheet_result = process_document(spreadsheet_path)

print(json.dumps(spreadsheet_result, indent=4))

{
    "status": "success",
    "document_type": "spreadsheet",
    "invoice_data": {
        "invoice_number": "XLS-TEST-001",
        "invoice_date": "05/11/2026",
        "supplier_name": "Spreadsheet Supplies",
        "customer_name": "Demo Buyer",
        "subtotal": "350",
        "tax": "52.5",
        "total": "402.5",
        "currency": "BDT",
        "line_items": [
            {
                "description": "Keyboard",
                "quantity": 2.0,
                "unit_price": 100.0,
                "amount": 200.0
            },
            {
                "description": "Mouse",
                "quantity": 3.0,
                "unit_price": 50.0,
                "amount": 150.0
            }
        ]
    },
    "validation": {
        "validation_status": "passed",
        "errors": []
    }
}


In [ ]:
spreadsheet_document_id = save_invoice_to_database(
    spreadsheet_path,
    spreadsheet_result
)

print("Saved document ID:", spreadsheet_document_id)

Saved document ID: 5


In [ ]:
cursor = conn.cursor()
cursor.execute("""
SELECT d.id, d.filename, d.file_type, d.processing_status,
       i.invoice_number, i.total
FROM documents d
LEFT JOIN invoices i ON d.id = i.document_id
WHERE d.id = 5
""")

print(cursor.fetchone())

(5, 'sample_spreadsheet_invoice.xlsx', '.xlsx', 'success', 'XLS-TEST-001', 402.5)


In [ ]:
cursor = conn.cursor()

cursor.execute("""
SELECT li.description, li.quantity, li.unit_price, li.amount
FROM line_items li
JOIN invoices i ON li.invoice_id = i.id
WHERE i.document_id = 5
""")

for item in cursor.fetchall():
    print(item)

('Keyboard', 2.0, 100.0, 200.0)
('Mouse', 3.0, 50.0, 150.0)


In [ ]:
import pandas as pd
import os

csv_data = {
    "Field": [
        "Invoice Number",
        "Date",
        "Supplier",
        "Customer",
        "Subtotal",
        "Tax",
        "Total",
        "Currency"
    ],
    "Value": [
        "CSV-TEST-001",
        "05/11/2026",
        "CSV Supplies",
        "Demo Buyer",
        500,
        75,
        575,
        "BDT"
    ]
}

csv_path = os.path.join(project_root, "sample_csv_invoice.csv")

pd.DataFrame(csv_data).to_csv(csv_path, index=False)

print("CSV file created:", csv_path)

CSV file created: /content/drive/MyDrive/AI_Document_Intelligence_System/sample_csv_invoice.csv


In [ ]:
csv_result = process_document(csv_path)

print(json.dumps(csv_result, indent=4))

{
    "status": "success",
    "document_type": "spreadsheet",
    "invoice_data": {
        "invoice_number": null,
        "invoice_date": null,
        "supplier_name": null,
        "customer_name": null,
        "subtotal": null,
        "tax": null,
        "total": null,
        "currency": null,
        "line_items": []
    },
    "validation": {
        "validation_status": "failed",
        "errors": [
            "Missing required field: invoice_number",
            "Missing required field: invoice_date",
            "Missing required field: supplier_name",
            "Missing required field: customer_name",
            "Missing required field: subtotal",
            "Missing required field: tax",
            "Missing required field: total",
            "Invalid or missing numeric invoice values"
        ]
    }
}


In [ ]:
def extract_csv_invoice(file_path):
    df = pd.read_csv(file_path)

    invoice = {
        "invoice_number": None,
        "invoice_date": None,
        "supplier_name": None,
        "customer_name": None,
        "subtotal": None,
        "tax": None,
        "total": None,
        "currency": None,
        "line_items": []
    }

    if "Field" not in df.columns or "Value" not in df.columns:
        raise ValueError("CSV must contain Field and Value columns")

    summary = dict(zip(df["Field"], df["Value"]))

    field_mapping = {
        "Invoice Number": "invoice_number",
        "Date": "invoice_date",
        "Supplier": "supplier_name",
        "Customer": "customer_name",
        "Subtotal": "subtotal",
        "Tax": "tax",
        "Total": "total",
        "Currency": "currency"
    }

    for source, target in field_mapping.items():
        value = summary.get(source)

        if pd.notna(value):
            invoice[target] = str(value)

    return invoice


print("CSV invoice extraction function ready!")

CSV invoice extraction function ready!


In [ ]:
def process_document(file_path):
    extension = os.path.splitext(file_path)[1].lower()

    try:
        if extension == ".pdf":
            text = extract_pdf_text(file_path)
            if not text.strip():
                text = extract_scanned_pdf_text(file_path)

        elif extension in [".jpg", ".jpeg", ".png", ".tiff", ".tif"]:
            text = extract_image_text(file_path)

        elif extension == ".csv":
            invoice = extract_csv_invoice(file_path)
            validation = validate_invoice(invoice)

            return {
                "status": "success",
                "document_type": "spreadsheet",
                "invoice_data": invoice,
                "validation": validation
            }

        elif extension in [".xlsx", ".xls"]:
            invoice = extract_spreadsheet_invoice(file_path)
            validation = validate_invoice(invoice)

            return {
                "status": "success",
                "document_type": "spreadsheet",
                "invoice_data": invoice,
                "validation": validation
            }

        else:
            raise ValueError("Unsupported file format")

        invoice = extract_invoice_data(text)
        validation = validate_invoice(invoice)

        return {
            "status": "success",
            "extracted_text": text,
            "invoice_data": invoice,
            "validation": validation
        }

    except Exception as e:
        return {
            "status": "failed",
            "error": str(e)
        }


print("Updated CSV and Excel processing pipeline ready!")

Updated CSV and Excel processing pipeline ready!


In [ ]:
csv_result = process_document(csv_path)

print(json.dumps(csv_result, indent=4))

{
    "status": "success",
    "document_type": "spreadsheet",
    "invoice_data": {
        "invoice_number": "CSV-TEST-001",
        "invoice_date": "05/11/2026",
        "supplier_name": "CSV Supplies",
        "customer_name": "Demo Buyer",
        "subtotal": "500",
        "tax": "75",
        "total": "575",
        "currency": "BDT",
        "line_items": []
    },
    "validation": {
        "validation_status": "passed",
        "errors": []
    }
}


In [ ]:
csv_document_id = save_invoice_to_database(csv_path, csv_result)

print("CSV document saved. Document ID:", csv_document_id)

CSV document saved. Document ID: 6


In [ ]:
cursor = conn.cursor()

cursor.execute("""
SELECT d.id, d.filename, d.file_type, d.processing_status,
       i.invoice_number, i.subtotal, i.tax, i.total
FROM documents d
LEFT JOIN invoices i ON d.id = i.document_id
WHERE d.id = 6
""")

print(cursor.fetchone())

(6, 'sample_csv_invoice.csv', '.csv', 'success', 'CSV-TEST-001', 500.0, 75.0, 575.0)


In [ ]:
def validate_invoice(invoice):
    errors = []

    required_fields = [
        "invoice_number",
        "invoice_date",
        "supplier_name",
        "customer_name",
        "subtotal",
        "tax",
        "total"
    ]

    for field in required_fields:
        if not invoice.get(field):
            errors.append(f"Missing required field: {field}")

    try:
        subtotal = float(str(invoice["subtotal"]).replace(",", ""))
        tax = float(str(invoice["tax"]).replace(",", ""))
        total = float(str(invoice["total"]).replace(",", ""))

        if abs((subtotal + tax) - total) > 0.01:
            errors.append("Subtotal + tax does not equal total")

        line_items = invoice.get("line_items", [])

        if line_items:
            line_item_total = sum(
                float(item["amount"]) for item in line_items
            )

            if abs(line_item_total - subtotal) > 0.01:
                errors.append(
                    "Sum of line item amounts does not equal subtotal"
                )

    except (ValueError, TypeError, KeyError):
        errors.append("Invalid or missing numeric invoice values")

    return {
        "validation_status": "passed" if not errors else "failed",
        "errors": errors
    }


print("Updated invoice validation ready!")

Updated invoice validation ready!


In [ ]:
print("Valid invoice test:")
print(json.dumps(validate_invoice(spreadsheet_result["invoice_data"]), indent=4))

print("\nInvalid line-item test:")

test_invoice = spreadsheet_result["invoice_data"].copy()
test_invoice["line_items"] = [
    {
        "description": "Keyboard",
        "quantity": 2,
        "unit_price": 100,
        "amount": 200
    },
    {
        "description": "Mouse",
        "quantity": 3,
        "unit_price": 50,
        "amount": 100
    }
]

print(json.dumps(validate_invoice(test_invoice), indent=4))

Valid invoice test:
{
    "validation_status": "passed",
    "errors": []
}

Invalid line-item test:
{
    "validation_status": "failed",
    "errors": [
        "Sum of line item amounts does not equal subtotal"
    ]
}


In [ ]:
duplicate_test = {
    "invoice_number": "XLS-TEST-001",
    "supplier_name": "Spreadsheet Supplies"
}

print(
    "Duplicate detected:",
    is_duplicate_before_save(duplicate_test)
)

Duplicate detected: True


In [ ]:
cursor = conn.cursor()

cursor.execute("""
SELECT id, filename, processing_status
FROM documents
WHERE id = 7
""")

print(cursor.fetchone())

(7, 'faabfae7-dc3e-40af-aece-497efabee163.pdf', 'needs_review')


In [ ]:
cursor = conn.cursor()

cursor.execute("""
SELECT error_message
FROM validation_errors
WHERE document_id = 7
""")

print(cursor.fetchall())

[('Subtotal + tax does not equal total',), ('Duplicate invoice detected',)]


In [ ]:
cursor = conn.cursor()

cursor.execute("""
SELECT id, invoice_number, total
FROM invoices
WHERE document_id = 7
""")

print(cursor.fetchone())

(7, 'INVALID-TEST-001', 250.0)


In [ ]:
cursor = conn.cursor()

cursor.execute("""
SELECT invoice_number, subtotal, tax, total
FROM invoices
WHERE id = 7
""")

print("Updated invoice:", cursor.fetchone())

cursor.execute("""
SELECT document_id, field_name, old_value, new_value
FROM correction_history
WHERE document_id = 7
""")

print("Correction history:", cursor.fetchall())

Updated invoice: ('INVALID-TEST-001', 200.0, 30.0, 230.0)
Correction history: [(7, 'total', '250.0', '230')]


In [ ]:
cursor = conn.cursor()

cursor.execute("""
SELECT processing_status
FROM documents
WHERE id = 7
""")

print("Current status:", cursor.fetchone())

cursor.execute("""
SELECT error_message
FROM validation_errors
WHERE document_id = 7
""")

print("Existing errors:", cursor.fetchall())

Current status: ('needs_review',)
Existing errors: [('Subtotal + tax does not equal total',), ('Duplicate invoice detected',)]


In [ ]:
def correct_invoice(invoice_id, field_name, new_value):
    cursor = conn.cursor()

    allowed_fields = [
        "invoice_number",
        "invoice_date",
        "supplier_name",
        "customer_name",
        "subtotal",
        "tax",
        "total",
        "currency"
    ]

    if field_name not in allowed_fields:
        raise HTTPException(status_code=400, detail="Invalid field name")

    cursor.execute(
        "SELECT document_id FROM invoices WHERE id = ?",
        (invoice_id,)
    )
    record = cursor.fetchone()

    if not record:
        raise HTTPException(status_code=404, detail="Invoice not found")

    document_id = record[0]

    cursor.execute(
        f"SELECT {field_name} FROM invoices WHERE id = ?",
        (invoice_id,)
    )
    old_value = cursor.fetchone()[0]

    cursor.execute(
        f"UPDATE invoices SET {field_name} = ? WHERE id = ?",
        (new_value, invoice_id)
    )

    cursor.execute("""
        INSERT INTO correction_history
        (document_id, field_name, old_value, new_value)
        VALUES (?, ?, ?, ?)
    """, (document_id, field_name, str(old_value), str(new_value)))

    conn.commit()

    return {
        "message": "Corrected successfully",
        "invoice_id": invoice_id,
        "field_name": field_name,
        "old_value": old_value,
        "new_value": new_value
    }

print("Updated correction function ready!")

Updated correction function ready!


In [ ]:
def revalidate_invoice_record(invoice_id):
    cursor = conn.cursor()

    cursor.execute("""
        SELECT document_id, invoice_number, invoice_date,
               supplier_name, customer_name, subtotal, tax,
               total, currency
        FROM invoices
        WHERE id = ?
    """, (invoice_id,))

    record = cursor.fetchone()

    if not record:
        return {"status": "failed", "error": "Invoice not found"}

    document_id = record[0]

    invoice = {
        "invoice_number": record[1],
        "invoice_date": record[2],
        "supplier_name": record[3],
        "customer_name": record[4],
        "subtotal": record[5],
        "tax": record[6],
        "total": record[7],
        "currency": record[8],
        "line_items": []
    }

    cursor.execute("""
        SELECT description, quantity, unit_price, amount
        FROM line_items
        WHERE invoice_id = ?
    """, (invoice_id,))

    for item in cursor.fetchall():
        invoice["line_items"].append({
            "description": item[0],
            "quantity": item[1],
            "unit_price": item[2],
            "amount": item[3]
        })

    validation = validate_invoice(invoice)
    errors = validation["errors"]

    if is_duplicate_before_save(
        invoice,
        exclude_document_id=document_id
    ):
        errors.append("Duplicate invoice detected")

    cursor.execute(
        "DELETE FROM validation_errors WHERE document_id = ?",
        (document_id,)
    )

    for error in errors:
        cursor.execute("""
            INSERT INTO validation_errors
            (document_id, error_message)
            VALUES (?, ?)
        """, (document_id, error))

    status = "needs_review" if errors else "success"

    cursor.execute("""
        UPDATE documents
        SET processing_status = ?
        WHERE id = ?
    """, (status, document_id))

    conn.commit()

    return {
        "document_id": document_id,
        "status": status,
        "validation_errors": errors
    }


print("Invoice revalidation function ready!")

Invoice revalidation function ready!


In [ ]:
@app.put("/invoices/{invoice_id}/correct")
def correct_invoice_api(invoice_id: int, correction: dict):

    field_name = correction.get("field_name")
    new_value = correction.get("new_value")

    if not field_name or new_value is None:
        raise HTTPException(
            status_code=400,
            detail="field_name and new_value are required"
        )

    correction_result = correct_invoice(
        invoice_id,
        field_name,
        new_value
    )

    validation_result = revalidate_invoice_record(invoice_id)

    return {
        "correction": correction_result,
        "revalidation": validation_result
    }


# Keep only the latest correction route
correction_routes = [
    route for route in app.router.routes
    if getattr(route, "path", None) == "/invoices/{invoice_id}/correct"
    and "PUT" in getattr(route, "methods", set())
]

if len(correction_routes) > 1:
    latest_route = correction_routes[-1]
    app.router.routes = [
        route for route in app.router.routes
        if route not in correction_routes
    ]
    app.router.routes.append(latest_route)

print("Updated correction API ready!")

Updated correction API ready!


In [ ]:
server.should_exit = True
time.sleep(3)

config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=8000,
    log_level="info"
)

server = uvicorn.Server(config)

api_thread = threading.Thread(
    target=server.run,
    daemon=True
)

api_thread.start()
time.sleep(5)

print("Server running:", server.started)

INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [1511]
INFO:     Started server process [1511]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Server running: True


In [ ]:
for route in app.routes:
    if getattr(route, "path", None) == "/invoices/{invoice_id}/correct":
        print("Endpoint:", route.path)
        print("Methods:", route.methods)
        print("Function:", route.endpoint.__name__)

Endpoint: /invoices/{invoice_id}/correct
Methods: {'PUT'}
Function: correct_invoice_api


In [ ]:
from pydantic import BaseModel

class CorrectionRequest(BaseModel):
    field_name: str
    new_value: str

print("Correction request model ready!")

Correction request model ready!


In [ ]:
@app.put("/invoices/{invoice_id}/correct")
def correct_invoice_api(
    invoice_id: int,
    correction: CorrectionRequest
):
    correction_result = correct_invoice(
        invoice_id,
        correction.field_name,
        correction.new_value
    )

    validation_result = revalidate_invoice_record(invoice_id)

    return {
        "correction": correction_result,
        "revalidation": validation_result
    }


# Keep only the latest correction endpoint
correction_routes = [
    route for route in app.router.routes
    if getattr(route, "path", None) == "/invoices/{invoice_id}/correct"
    and "PUT" in getattr(route, "methods", set())
]

latest_route = correction_routes[-1]

app.router.routes = [
    route for route in app.router.routes
    if route not in correction_routes
]

app.router.routes.append(latest_route)

print("Correction endpoint updated!")

Correction endpoint updated!


In [ ]:
server.should_exit = True
time.sleep(3)

config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=8000,
    log_level="info"
)

server = uvicorn.Server(config)
api_thread = threading.Thread(target=server.run, daemon=True)
api_thread.start()

time.sleep(5)

print("Server running:", server.started)

INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [1511]
INFO:     Started server process [1511]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Server running: True


In [ ]:
cursor = conn.cursor()

cursor.execute("""
SELECT processing_status
FROM documents
WHERE id = 7
""")
print("Document status:", cursor.fetchone())

cursor.execute("""
SELECT error_message
FROM validation_errors
WHERE document_id = 7
""")
print("Remaining errors:", cursor.fetchall())

Document status: ('needs_review',)
Remaining errors: [('Duplicate invoice detected',)]


In [ ]:
!pip install xlwt xlrd

In [ ]:
import xlwt
import os

workbook = xlwt.Workbook()
sheet = workbook.add_sheet("Invoice Summary")

data = [
    ["Field", "Value"],
    ["Invoice Number", "XLS-TEST-001"],
    ["Date", "05/11/2026"],
    ["Supplier", "Legacy Supplies"],
    ["Customer", "Demo Buyer"],
    ["Subtotal", 400],
    ["Tax", 60],
    ["Total", 460],
    ["Currency", "BDT"]
]

for row_index, row in enumerate(data):
    for col_index, value in enumerate(row):
        sheet.write(row_index, col_index, value)

xls_test_path = os.path.join(
    project_root,
    "sample_xls_invoice.xls"
)

workbook.save(xls_test_path)

print("XLS test file created:", xls_test_path)

XLS test file created: /content/drive/MyDrive/AI_Document_Intelligence_System/sample_xls_invoice.xls


In [ ]:
xls_result = process_document(xls_test_path)

print(json.dumps(xls_result, indent=4))

{
    "status": "success",
    "document_type": "spreadsheet",
    "invoice_data": {
        "invoice_number": "XLS-TEST-001",
        "invoice_date": "05/11/2026",
        "supplier_name": "Legacy Supplies",
        "customer_name": "Demo Buyer",
        "subtotal": "400",
        "tax": "60",
        "total": "460",
        "currency": "BDT",
        "line_items": []
    },
    "validation": {
        "validation_status": "passed",
        "errors": []
    }
}


In [ ]:
from google.colab import files

files.download(xls_test_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
def extract_spreadsheet_text(file_path):
    extension = os.path.splitext(file_path)[1].lower()

    if extension == ".csv":
        df = pd.read_csv(file_path)
        return df.to_string(index=False)

    elif extension in [".xlsx", ".xls"]:
        sheets = pd.read_excel(file_path, sheet_name=None)

        extracted_text = ""

        for sheet_name, df in sheets.items():
            extracted_text += f"\nSheet: {sheet_name}\n"
            extracted_text += df.to_string(index=False)
            extracted_text += "\n"

        return extracted_text

    else:
        raise ValueError("Unsupported spreadsheet format")


spreadsheet_text = extract_spreadsheet_text(xls_test_path)
print(spreadsheet_text)


Sheet: Invoice Summary
         Field           Value
Invoice Number    XLS-TEST-001
          Date      05/11/2026
      Supplier Legacy Supplies
      Customer      Demo Buyer
      Subtotal             400
           Tax              60
         Total             460
      Currency             BDT



In [ ]:
def process_document(file_path):
    extension = os.path.splitext(file_path)[1].lower()

    try:
        if extension == ".pdf":
            text = extract_pdf_text(file_path)

            if not text.strip():
                text = extract_scanned_pdf_text(file_path)

            invoice = extract_invoice_data(text)

        elif extension in [".jpg", ".jpeg", ".png", ".tiff", ".tif"]:
            text = extract_image_text(file_path)
            invoice = extract_invoice_data(text)

        elif extension == ".csv":
            text = extract_spreadsheet_text(file_path)
            invoice = extract_csv_invoice(file_path)

        elif extension in [".xlsx", ".xls"]:
            text = extract_spreadsheet_text(file_path)
            invoice = extract_spreadsheet_invoice(file_path)

        else:
            raise ValueError("Unsupported file format")

        validation = validate_invoice(invoice)

        return {
            "status": "success",
            "document_type": (
                "spreadsheet"
                if extension in [".csv", ".xlsx", ".xls"]
                else "document"
            ),
            "extracted_text": text,
            "invoice_data": invoice,
            "validation": validation
        }

    except Exception as e:
        return {
            "status": "failed",
            "error": str(e)
        }

In [ ]:
xls_result = process_document(xls_test_path)

print("Status:", xls_result["status"])
print("\nExtracted text:")
print(xls_result["extracted_text"])
print("\nInvoice data:")
print(json.dumps(xls_result["invoice_data"], indent=4))

Status: success

Extracted text:

Sheet: Invoice Summary
         Field           Value
Invoice Number    XLS-TEST-001
          Date      05/11/2026
      Supplier Legacy Supplies
      Customer      Demo Buyer
      Subtotal             400
           Tax              60
         Total             460
      Currency             BDT


Invoice data:
{
    "invoice_number": "XLS-TEST-001",
    "invoice_date": "05/11/2026",
    "supplier_name": "Legacy Supplies",
    "customer_name": "Demo Buyer",
    "subtotal": "400",
    "tax": "60",
    "total": "460",
    "currency": "BDT",
    "line_items": []
}


In [ ]:
cursor = conn.cursor()

cursor.execute("""
    SELECT document_id, extracted_content
    FROM extracted_text
    WHERE document_id = 9
""")

result = cursor.fetchall()

for row in result:
    print("Document ID:", row[0])
    print("Extracted content:")
    print(row[1])

Document ID: 9
Extracted content:

Sheet: Invoice Summary
         Field           Value
Invoice Number    XLS-TEST-001
          Date      05/11/2026
      Supplier Legacy Supplies
      Customer      Demo Buyer
      Subtotal             400
           Tax              60
         Total             460
      Currency             BDT



In [ ]:
csv_test_path = os.path.join(
    project_root,
    "sample_csv_invoice.csv"
)

print(pd.read_csv(csv_test_path).to_string(index=False))

         Field        Value
Invoice Number CSV-TEST-001
          Date   05/11/2026
      Supplier CSV Supplies
      Customer   Demo Buyer
      Subtotal          500
           Tax           75
         Total          575
      Currency          BDT


In [ ]:
csv_test_data = [
    ["Field", "Value"],
    ["Invoice Number", "CSV-TEST-002"],
    ["Date", "05/11/2026"],
    ["Supplier", "CSV Supplies"],
    ["Customer", "Demo Buyer"],
    ["Subtotal", 500],
    ["Tax", 75],
    ["Total", 575],
    ["Currency", "BDT"],
    ["Description", "Keyboard"],
    ["Quantity", 2],
    ["Unit Price", 150],
    ["Amount", 300],
    ["Description", "Mouse"],
    ["Quantity", 4],
    ["Unit Price", 50],
    ["Amount", 200]
]

csv_test_path_v2 = os.path.join(
    project_root,
    "sample_csv_invoice_v2.csv"
)

pd.DataFrame(csv_test_data).to_csv(
    csv_test_path_v2,
    index=False,
    header=False
)

print("New CSV test file created:", csv_test_path_v2)

New CSV test file created: /content/drive/MyDrive/AI_Document_Intelligence_System/sample_csv_invoice_v2.csv


In [ ]:
def extract_csv_invoice(file_path):
    df = pd.read_csv(file_path, header=None)

    invoice = {
        "invoice_number": None,
        "invoice_date": None,
        "supplier_name": None,
        "customer_name": None,
        "subtotal": None,
        "tax": None,
        "total": None,
        "currency": None,
        "line_items": []
    }

    field_mapping = {
        "Invoice Number": "invoice_number",
        "Date": "invoice_date",
        "Supplier": "supplier_name",
        "Customer": "customer_name",
        "Subtotal": "subtotal",
        "Tax": "tax",
        "Total": "total",
        "Currency": "currency"
    }

    # Extract invoice summary fields
    for _, row in df.iterrows():
        field = str(row[0]).strip()
        value = row[1]

        if field in field_mapping and pd.notna(value):
            invoice[field_mapping[field]] = str(value)

    # Extract line items
    current_item = {}

    for _, row in df.iterrows():
        field = str(row[0]).strip()
        value = row[1]

        if field == "Description":
            if current_item:
                invoice["line_items"].append(current_item)

            current_item = {
                "description": str(value),
                "quantity": None,
                "unit_price": None,
                "amount": None
            }

        elif field == "Quantity" and current_item:
            current_item["quantity"] = float(value)

        elif field == "Unit Price" and current_item:
            current_item["unit_price"] = float(value)

        elif field == "Amount" and current_item:
            current_item["amount"] = float(value)

    if current_item:
        invoice["line_items"].append(current_item)

    return invoice


print("CSV extraction function updated successfully.")

CSV extraction function updated successfully.


In [ ]:
csv_result = extract_csv_invoice(csv_test_path_v2)

print(json.dumps(csv_result, indent=4))

{
    "invoice_number": "CSV-TEST-002",
    "invoice_date": "05/11/2026",
    "supplier_name": "CSV Supplies",
    "customer_name": "Demo Buyer",
    "subtotal": "500",
    "tax": "75",
    "total": "575",
    "currency": "BDT",
    "line_items": [
        {
            "description": "Keyboard",
            "quantity": 2.0,
            "unit_price": 150.0,
            "amount": 300.0
        },
        {
            "description": "Mouse",
            "quantity": 4.0,
            "unit_price": 50.0,
            "amount": 200.0
        }
    ]
}


In [ ]:
csv_validation = validate_invoice(csv_result)

print(json.dumps(csv_validation, indent=4))

{
    "validation_status": "passed",
    "errors": []
}


In [ ]:
csv_pipeline_result = process_document(csv_test_path_v2)

print(json.dumps(csv_pipeline_result, indent=4))

{
    "status": "success",
    "document_type": "spreadsheet",
    "extracted_text": "         Field        Value\nInvoice Number CSV-TEST-002\n          Date   05/11/2026\n      Supplier CSV Supplies\n      Customer   Demo Buyer\n      Subtotal          500\n           Tax           75\n         Total          575\n      Currency          BDT\n   Description     Keyboard\n      Quantity            2\n    Unit Price          150\n        Amount          300\n   Description        Mouse\n      Quantity            4\n    Unit Price           50\n        Amount          200",
    "invoice_data": {
        "invoice_number": "CSV-TEST-002",
        "invoice_date": "05/11/2026",
        "supplier_name": "CSV Supplies",
        "customer_name": "Demo Buyer",
        "subtotal": "500",
        "tax": "75",
        "total": "575",
        "currency": "BDT",
        "line_items": [
            {
                "description": "Keyboard",
                "quantity": 2.0,
                "unit_pri

In [ ]:
from google.colab import files

files.download(csv_test_path_v2)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
cursor = conn.cursor()

cursor.execute("""
    SELECT line_items.description,
           line_items.quantity,
           line_items.unit_price,
           line_items.amount
    FROM line_items
    JOIN invoices ON line_items.invoice_id = invoices.id
    WHERE invoices.document_id = 10
""")

print(cursor.fetchall())

[('Keyboard', 2.0, 150.0, 300.0), ('Mouse', 4.0, 50.0, 200.0)]


In [ ]:
print([route.path for route in app.routes if "search" in route.path])

['/search']


In [ ]:
import inspect

print(inspect.getsource(search_documents))

@app.get("/search")
def search_documents(keyword: str):
    cursor = conn.cursor()

    cursor.execute("""
    SELECT * FROM invoices
    WHERE invoice_number LIKE ?
    OR supplier_name LIKE ?
    OR customer_name LIKE ?
    """, (f"%{keyword}%", f"%{keyword}%", f"%{keyword}%"))

    return cursor.fetchall()



In [ ]:
@app.get("/search")
def search_documents(keyword: str):
    cursor = conn.cursor()

    search_term = f"%{keyword}%"

    cursor.execute("""
        SELECT DISTINCT
            invoices.id,
            invoices.document_id,
            invoices.invoice_number,
            invoices.invoice_date,
            invoices.supplier_name,
            invoices.customer_name,
            invoices.subtotal,
            invoices.tax,
            invoices.total,
            invoices.currency
        FROM invoices
        LEFT JOIN extracted_text
            ON invoices.document_id = extracted_text.document_id
        WHERE invoices.invoice_number LIKE ?
           OR invoices.supplier_name LIKE ?
           OR invoices.customer_name LIKE ?
           OR extracted_text.extracted_content LIKE ?
    """, (search_term, search_term, search_term, search_term))

    return cursor.fetchall()

In [ ]:
server.should_exit = True
time.sleep(3)

config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=8000,
    log_level="info"
)

server = uvicorn.Server(config)

api_thread = threading.Thread(
    target=server.run,
    daemon=True
)

api_thread.start()
time.sleep(5)

print("Server running:", server.started)

INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [1511]
INFO:     Started server process [1511]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Server running: True


In [ ]:
cursor = conn.cursor()

cursor.execute("""
    SELECT document_id, extracted_content
    FROM extracted_text
    WHERE document_id = 10
""")

result = cursor.fetchall()

print(result)

[(10, '         Field        Value\nInvoice Number CSV-TEST-002\n          Date   05/11/2026\n      Supplier CSV Supplies\n      Customer   Demo Buyer\n      Subtotal          500\n           Tax           75\n         Total          575\n      Currency          BDT\n   Description     Keyboard\n      Quantity            2\n    Unit Price          150\n        Amount          300\n   Description        Mouse\n      Quantity            4\n    Unit Price           50\n        Amount          200')]


In [ ]:
cursor = conn.cursor()

cursor.execute("""
    SELECT document_id
    FROM extracted_text
    WHERE extracted_content LIKE '%Keyboard%'
""")

print(cursor.fetchall())

[(2,), (10,)]


In [ ]:
cursor = conn.cursor()

keyword = "Keyboard"
search_term = f"%{keyword}%"

cursor.execute("""
    SELECT DISTINCT
        invoices.id,
        invoices.document_id,
        invoices.invoice_number,
        invoices.invoice_date,
        invoices.supplier_name,
        invoices.customer_name,
        invoices.subtotal,
        invoices.tax,
        invoices.total,
        invoices.currency
    FROM invoices
    LEFT JOIN extracted_text
        ON invoices.document_id = extracted_text.document_id
    WHERE invoices.invoice_number LIKE ?
       OR invoices.supplier_name LIKE ?
       OR invoices.customer_name LIKE ?
       OR extracted_text.extracted_content LIKE ?
""", (search_term, search_term, search_term, search_term))

print(cursor.fetchall())

[(2, 2, 'UPLOAD-TEST-002', '05/11/2026', 'Demo Systems', 'Test Buyer', 200.0, 30.0, 200.0, 'BDT'), (10, 10, 'CSV-TEST-002', '05/11/2026', 'CSV Supplies', 'Demo Buyer', 500.0, 75.0, 575.0, 'BDT')]


In [ ]:
print(search_documents("Keyboard"))

[(2, 2, 'UPLOAD-TEST-002', '05/11/2026', 'Demo Systems', 'Test Buyer', 200.0, 30.0, 200.0, 'BDT'), (10, 10, 'CSV-TEST-002', '05/11/2026', 'CSV Supplies', 'Demo Buyer', 500.0, 75.0, 575.0, 'BDT')]


In [ ]:
print("Notebook database:", conn.execute("PRAGMA database_list").fetchall())
print("API function:", search_documents("Keyboard"))

Notebook database: [(0, 'main', '/content/drive/MyDrive/AI_Document_Intelligence_System/document_intelligence/documents.db')]
API function: [(2, 2, 'UPLOAD-TEST-002', '05/11/2026', 'Demo Systems', 'Test Buyer', 200.0, 30.0, 200.0, 'BDT'), (10, 10, 'CSV-TEST-002', '05/11/2026', 'CSV Supplies', 'Demo Buyer', 500.0, 75.0, 575.0, 'BDT')]


In [ ]:
print("Public URL:", public_url)
print("Search URL:", public_url + "/search")
print("Docs URL:", public_url + "/docs")

Public URL: https://8000-m-s-usc1c9-2xw777i57big4-c.us-central1.prod.r.colab.dev
Search URL: https://8000-m-s-usc1c9-2xw777i57big4-c.us-central1.prod.r.colab.dev/search
Docs URL: https://8000-m-s-usc1c9-2xw777i57big4-c.us-central1.prod.r.colab.dev/docs


In [ ]:
response = requests.get(
    public_url + "/search",
    params={"keyword": "Keyboard"}
)

print("Status:", response.status_code)
print("Final URL:", response.url)
print("Content type:", response.headers.get("content-type"))
print("Response text:", response.text[:1000])

Status: 404
Final URL: https://8000-m-s-usc1c9-2xw777i57big4-c.us-central1.prod.r.colab.dev/search?keyword=Keyboard
Content type: None
Response text: 


In [ ]:
response = requests.get(public_url + "/")

print("Status:", response.status_code)
print("Response:", response.text[:500])

Status: 404
Response: 


In [ ]:
response = requests.get(
    "http://127.0.0.1:8000/search",
    params={"keyword": "Keyboard"}
)

print("Status:", response.status_code)
print("Response:", response.text)

INFO:     127.0.0.1:50048 - "GET /search?keyword=Keyboard HTTP/1.1" 200 OK
Status: 200
Response: []


In [ ]:
for route in app.routes:
    if getattr(route, "path", None) == "/search":
        print("Registered function:", route.endpoint.__name__)
        print(inspect.getsource(route.endpoint))
        print("--------------------")

Registered function: search_documents
@app.get("/search")
def search_documents(keyword: str):
    cursor = conn.cursor()

    cursor.execute("""
    SELECT * FROM invoices
    WHERE invoice_number LIKE ?
    OR supplier_name LIKE ?
    OR customer_name LIKE ?
    """, (f"%{keyword}%", f"%{keyword}%", f"%{keyword}%"))

    return cursor.fetchall()

--------------------
Registered function: search_documents
@app.get("/search")
def search_documents(keyword: str):
    cursor = conn.cursor()

    search_term = f"%{keyword}%"

    cursor.execute("""
        SELECT DISTINCT
            invoices.id,
            invoices.document_id,
            invoices.invoice_number,
            invoices.invoice_date,
            invoices.supplier_name,
            invoices.customer_name,
            invoices.subtotal,
            invoices.tax,
            invoices.total,
            invoices.currency
        FROM invoices
        LEFT JOIN extracted_text
            ON invoices.document_id = extracted_text.

In [ ]:
search_routes = [
    route for route in app.routes
    if getattr(route, "path", None) == "/search"
]

print("Search routes before:", len(search_routes))

# Keep the latest search route and remove the old one
app.routes.remove(search_routes[0])

print(
    "Search routes after:",
    len([
        route for route in app.routes
        if getattr(route, "path", None) == "/search"
    ])
)

Search routes before: 2
Search routes after: 1


In [ ]:
response = requests.get(
    "http://127.0.0.1:8000/search",
    params={"keyword": "Keyboard"}
)

print("Status:", response.status_code)
print("Response:", response.json())

INFO:     127.0.0.1:43266 - "GET /search?keyword=Keyboard HTTP/1.1" 200 OK
Status: 200
Response: [[2, 2, 'UPLOAD-TEST-002', '05/11/2026', 'Demo Systems', 'Test Buyer', 200.0, 30.0, 200.0, 'BDT'], [10, 10, 'CSV-TEST-002', '05/11/2026', 'CSV Supplies', 'Demo Buyer', 500.0, 75.0, 575.0, 'BDT']]


In [ ]:
server.should_exit = True
time.sleep(3)

config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=8000,
    log_level="info"
)

server = uvicorn.Server(config)

api_thread = threading.Thread(
    target=server.run,
    daemon=True
)

api_thread.start()
time.sleep(5)

print("Server running:", server.started)

INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [1511]
INFO:     Started server process [1511]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Server running: True


In [ ]:
import inspect

print(inspect.getsource(is_duplicate_before_save))

def is_duplicate_before_save(invoice, exclude_document_id=None):
    invoice_number = invoice.get("invoice_number")
    supplier_name = invoice.get("supplier_name")

    if not invoice_number or not supplier_name:
        return False

    cursor = conn.cursor()

    query = """
        SELECT COUNT(*)
        FROM invoices
        WHERE LOWER(invoice_number) = LOWER(?)
        AND LOWER(supplier_name) = LOWER(?)
    """

    params = [invoice_number, supplier_name]

    if exclude_document_id is not None:
        query += " AND document_id != ?"
        params.append(exclude_document_id)

    cursor.execute(query, params)

    return cursor.fetchone()[0] > 0



In [ ]:
test_invoice = {
    "invoice_number": "CSV-TEST-002",
    "supplier_name": "CSV Supplies"
}

print("Duplicate detected:", is_duplicate_before_save(test_invoice))

Duplicate detected: True


In [ ]:
test_invoice = {
    "invoice_number": "CSV-TEST-002",
    "supplier_name": "csv supplies"
}

print("Duplicate detected:", is_duplicate_before_save(test_invoice))

Duplicate detected: True


In [ ]:
test_invoice = {
    "invoice_number": "CSV-TEST-002",
    "supplier_name": "CSV Supplies"
}

print(
    "Duplicate detected:",
    is_duplicate_before_save(
        test_invoice,
        exclude_document_id=10
    )
)

Duplicate detected: False


In [ ]:
import inspect

print(inspect.getsource(revalidate_invoice_record))

def revalidate_invoice_record(invoice_id):
    cursor = conn.cursor()

    cursor.execute("""
        SELECT document_id, invoice_number, invoice_date,
               supplier_name, customer_name, subtotal, tax,
               total, currency
        FROM invoices
        WHERE id = ?
    """, (invoice_id,))

    record = cursor.fetchone()

    if not record:
        return {"status": "failed", "error": "Invoice not found"}

    document_id = record[0]

    invoice = {
        "invoice_number": record[1],
        "invoice_date": record[2],
        "supplier_name": record[3],
        "customer_name": record[4],
        "subtotal": record[5],
        "tax": record[6],
        "total": record[7],
        "currency": record[8],
        "line_items": []
    }

    cursor.execute("""
        SELECT description, quantity, unit_price, amount
        FROM line_items
        WHERE invoice_id = ?
    """, (invoice_id,))

    for item in cursor.fetchall():
        invoice["line_items"].append({
  

In [ ]:
import inspect

print(inspect.getsource(correct_invoice))

def correct_invoice(invoice_id, field_name, new_value):
    cursor = conn.cursor()

    allowed_fields = [
        "invoice_number",
        "invoice_date",
        "supplier_name",
        "customer_name",
        "subtotal",
        "tax",
        "total",
        "currency"
    ]

    if field_name not in allowed_fields:
        raise HTTPException(status_code=400, detail="Invalid field name")

    cursor.execute(
        "SELECT document_id FROM invoices WHERE id = ?",
        (invoice_id,)
    )
    record = cursor.fetchone()

    if not record:
        raise HTTPException(status_code=404, detail="Invoice not found")

    document_id = record[0]

    cursor.execute(
        f"SELECT {field_name} FROM invoices WHERE id = ?",
        (invoice_id,)
    )
    old_value = cursor.fetchone()[0]

    cursor.execute(
        f"UPDATE invoices SET {field_name} = ? WHERE id = ?",
        (new_value, invoice_id)
    )

    cursor.execute("""
        INSERT INTO correction_history
        (doc

In [ ]:
def correct_invoice(invoice_id, field_name, new_value):
    cursor = conn.cursor()

    allowed_fields = [
        "invoice_number",
        "invoice_date",
        "supplier_name",
        "customer_name",
        "subtotal",
        "tax",
        "total",
        "currency"
    ]

    if field_name not in allowed_fields:
        raise HTTPException(
            status_code=400,
            detail="Invalid field name"
        )

    cursor.execute(
        "SELECT document_id FROM invoices WHERE id = ?",
        (invoice_id,)
    )

    record = cursor.fetchone()

    if not record:
        raise HTTPException(
            status_code=404,
            detail="Invoice not found"
        )

    document_id = record[0]

    cursor.execute(
        f"SELECT {field_name} FROM invoices WHERE id = ?",
        (invoice_id,)
    )

    old_value = cursor.fetchone()[0]

    cursor.execute(
        f"UPDATE invoices SET {field_name} = ? WHERE id = ?",
        (new_value, invoice_id)
    )

    cursor.execute("""
        INSERT INTO correction_history
        (document_id, field_name, old_value, new_value)
        VALUES (?, ?, ?, ?)
    """, (document_id, field_name, str(old_value), str(new_value)))

    conn.commit()

    # Automatically revalidate after correction
    validation_result = revalidate_invoice_record(invoice_id)

    return {
        "message": "Corrected successfully",
        "invoice_id": invoice_id,
        "field_name": field_name,
        "old_value": old_value,
        "new_value": new_value,
        "revalidation": validation_result
    }

In [ ]:

print(inspect.getsource(correct_invoice))

def correct_invoice(invoice_id, field_name, new_value):
    cursor = conn.cursor()

    allowed_fields = [
        "invoice_number",
        "invoice_date",
        "supplier_name",
        "customer_name",
        "subtotal",
        "tax",
        "total",
        "currency"
    ]

    if field_name not in allowed_fields:
        raise HTTPException(
            status_code=400,
            detail="Invalid field name"
        )

    cursor.execute(
        "SELECT document_id FROM invoices WHERE id = ?",
        (invoice_id,)
    )

    record = cursor.fetchone()

    if not record:
        raise HTTPException(
            status_code=404,
            detail="Invoice not found"
        )

    document_id = record[0]

    cursor.execute(
        f"SELECT {field_name} FROM invoices WHERE id = ?",
        (invoice_id,)
    )

    old_value = cursor.fetchone()[0]

    cursor.execute(
        f"UPDATE invoices SET {field_name} = ? WHERE id = ?",
        (new_value, invoice_id)
    )

    

In [ ]:
server.should_exit = True
time.sleep(3)

config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=8000,
    log_level="info"
)

server = uvicorn.Server(config)
api_thread = threading.Thread(target=server.run, daemon=True)
api_thread.start()

time.sleep(5)
print("Server running:", server.started)

INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [1511]
INFO:     Started server process [1511]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Server running: True


In [ ]:
cursor = conn.cursor()

cursor.execute("""
    SELECT document_id, field_name, old_value, new_value
    FROM correction_history
    WHERE document_id = 7
""")

print(cursor.fetchall())

[(7, 'total', '250.0', '230'), (7, 'total', '230.0', '230'), (7, 'total', '230.0', '230')]


In [ ]:
for route in app.routes:
    if "correct" in getattr(route, "path", ""):
        print("Path:", route.path)
        print("Function name:", route.endpoint.__name__)

Path: /invoices/{invoice_id}/correct
Function name: correct_invoice_api


In [ ]:
print(inspect.getsource(correct_invoice_api))

@app.put("/invoices/{invoice_id}/correct")
def correct_invoice_api(
    invoice_id: int,
    correction: CorrectionRequest
):
    correction_result = correct_invoice(
        invoice_id,
        correction.field_name,
        correction.new_value
    )

    validation_result = revalidate_invoice_record(invoice_id)

    return {
        "correction": correction_result,
        "revalidation": validation_result
    }



In [ ]:
@app.put("/invoices/{invoice_id}/correct")
def correct_invoice_api(
    invoice_id: int,
    correction: CorrectionRequest
):
    correction_result = correct_invoice(
        invoice_id,
        correction.field_name,
        correction.new_value
    )

    return correction_result

In [ ]:
server.should_exit = True
time.sleep(3)

config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=8000,
    log_level="info"
)

server = uvicorn.Server(config)
api_thread = threading.Thread(target=server.run, daemon=True)
api_thread.start()

time.sleep(5)

print("Server running:", server.started)

INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [1511]
INFO:     Started server process [1511]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Server running: True


In [ ]:
for route in app.routes:
    if getattr(route, "path", None) == "/invoices/{invoice_id}/correct":
        print(inspect.getsource(route.endpoint))

@app.put("/invoices/{invoice_id}/correct")
def correct_invoice_api(
    invoice_id: int,
    correction: CorrectionRequest
):
    correction_result = correct_invoice(
        invoice_id,
        correction.field_name,
        correction.new_value
    )

    validation_result = revalidate_invoice_record(invoice_id)

    return {
        "correction": correction_result,
        "revalidation": validation_result
    }

@app.put("/invoices/{invoice_id}/correct")
def correct_invoice_api(
    invoice_id: int,
    correction: CorrectionRequest
):
    correction_result = correct_invoice(
        invoice_id,
        correction.field_name,
        correction.new_value
    )

    return correction_result



In [ ]:
correction_routes = [
    route for route in app.routes
    if getattr(route, "path", None) == "/invoices/{invoice_id}/correct"
]

print("Routes before:", len(correction_routes))

app.routes.remove(correction_routes[0])

remaining_routes = [
    route for route in app.routes
    if getattr(route, "path", None) == "/invoices/{invoice_id}/correct"
]

print("Routes after:", len(remaining_routes))
print(inspect.getsource(remaining_routes[0].endpoint))

Routes before: 2
Routes after: 1
@app.put("/invoices/{invoice_id}/correct")
def correct_invoice_api(
    invoice_id: int,
    correction: CorrectionRequest
):
    correction_result = correct_invoice(
        invoice_id,
        correction.field_name,
        correction.new_value
    )

    return correction_result



In [ ]:
server.should_exit = True
time.sleep(3)

config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=8000,
    log_level="info"
)

server = uvicorn.Server(config)
api_thread = threading.Thread(target=server.run, daemon=True)
api_thread.start()

time.sleep(5)

print("Server running:", server.started)

INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [1511]
INFO:     Started server process [1511]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Server running: True


In [ ]:
print(inspect.getsource(retry_document))

@app.post("/retry/{document_id}")
def retry_document(document_id: int):

    cursor = conn.cursor()

    cursor.execute(
        "SELECT filename FROM documents WHERE id = ?",
        (document_id,)
    )

    record = cursor.fetchone()

    if not record:
        raise HTTPException(
            status_code=404,
            detail="Document not found"
        )

    file_path = os.path.join(
        "document_intelligence/uploads",
        record[0]
    )

    if not os.path.exists(file_path):
        raise HTTPException(
            status_code=404,
            detail="Uploaded file not found"
        )

    result = process_document(file_path)

    if result.get("status") == "failed":
        cursor.execute("""
            UPDATE documents
            SET processing_status = 'failed'
            WHERE id = ?
        """, (document_id,))
        conn.commit()

        return {
            "document_id": document_id,
            "status": "failed",
            "error": result.get("err

In [ ]:
print(inspect.getsource(update_existing_document))

def update_existing_document(document_id, file_path, result):
    cursor = conn.cursor()

    status = result.get("status", "failed")
    invoice = result.get("invoice_data")
    validation_errors = result.get("validation", {}).get("errors", [])

    duplicate = (
        is_duplicate_before_save(invoice, exclude_document_id=document_id)
        if invoice else False
    )

    if duplicate:
        validation_errors.append("Duplicate invoice detected")

    if validation_errors:
        status = "needs_review"

    cursor.execute("""
        UPDATE documents
        SET processing_status = ?
        WHERE id = ?
    """, (status, document_id))

    if result.get("extracted_text"):
        cursor.execute("""
            UPDATE extracted_text
            SET extracted_content = ?
            WHERE document_id = ?
        """, (result["extracted_text"], document_id))

    conn.commit()

    return {
        "document_id": document_id,
        "status": status,
        "duplicate_detected

In [ ]:
def update_existing_document(document_id, file_path, result):
    cursor = conn.cursor()

    invoice = result.get("invoice_data")

    if not invoice:
        raise ValueError("Invoice data missing from processing result")

    validation = validate_invoice(invoice)
    validation_errors = validation["errors"]

    duplicate = is_duplicate_before_save(
        invoice,
        exclude_document_id=document_id
    )

    if duplicate:
        validation_errors.append("Duplicate invoice detected")

    status = "needs_review" if validation_errors else "success"

    # Update document status
    cursor.execute("""
        UPDATE documents
        SET processing_status = ?
        WHERE id = ?
    """, (status, document_id))

    # Update existing invoice record
    cursor.execute("""
        UPDATE invoices
        SET invoice_number = ?,
            invoice_date = ?,
            supplier_name = ?,
            customer_name = ?,
            subtotal = ?,
            tax = ?,
            total = ?,
            currency = ?
        WHERE document_id = ?
    """, (
        invoice.get("invoice_number"),
        invoice.get("invoice_date"),
        invoice.get("supplier_name"),
        invoice.get("customer_name"),
        invoice.get("subtotal"),
        invoice.get("tax"),
        invoice.get("total"),
        invoice.get("currency"),
        document_id
    ))

    # Find the existing invoice ID
    cursor.execute(
        "SELECT id FROM invoices WHERE document_id = ?",
        (document_id,)
    )
    invoice_record = cursor.fetchone()

    if not invoice_record:
        raise ValueError("Existing invoice record not found")

    invoice_id = invoice_record[0]

    # Replace old line items
    cursor.execute(
        "DELETE FROM line_items WHERE invoice_id = ?",
        (invoice_id,)
    )

    for item in invoice.get("line_items", []):
        cursor.execute("""
            INSERT INTO line_items
            (invoice_id, description, quantity, unit_price, amount)
            VALUES (?, ?, ?, ?, ?)
        """, (
            invoice_id,
            item.get("description"),
            item.get("quantity"),
            item.get("unit_price"),
            item.get("amount")
        ))

    # Replace old validation errors
    cursor.execute(
        "DELETE FROM validation_errors WHERE document_id = ?",
        (document_id,)
    )

    for error in validation_errors:
        cursor.execute("""
            INSERT INTO validation_errors
            (document_id, error_message)
            VALUES (?, ?)
        """, (document_id, error))

    # Update extracted text
    if result.get("extracted_text"):
        cursor.execute("""
            UPDATE extracted_text
            SET extracted_content = ?
            WHERE document_id = ?
        """, (result["extracted_text"], document_id))

    conn.commit()

    return {
        "document_id": document_id,
        "status": status,
        "duplicate_detected": duplicate,
        "validation_errors": validation_errors
    }

In [ ]:
cursor = conn.cursor()

cursor.execute("""
    SELECT error_message
    FROM validation_errors
    WHERE document_id = 3
""")

print(cursor.fetchall())

[('Subtotal + tax does not equal total',), ('Duplicate invoice detected',)]


In [ ]:
cursor = conn.cursor()

cursor.execute("""
    SELECT invoices.id, invoices.invoice_number,
           line_items.description,
           line_items.quantity,
           line_items.unit_price,
           line_items.amount
    FROM invoices
    LEFT JOIN line_items
        ON invoices.id = line_items.invoice_id
    WHERE invoices.document_id = 3
""")

for row in cursor.fetchall():
    print(row)

(3, 'INVALID-TEST-001', None, None, None, None)


In [ ]:
cursor = conn.cursor()

cursor.execute("""
    SELECT invoices.id, invoices.invoice_number,
           line_items.description,
           line_items.quantity,
           line_items.unit_price,
           line_items.amount
    FROM invoices
    LEFT JOIN line_items
        ON invoices.id = line_items.invoice_id
    WHERE invoices.document_id = 10
""")

for row in cursor.fetchall():
    print(row)

(10, 'CSV-TEST-002', 'Keyboard', 2.0, 150.0, 300.0)
(10, 'CSV-TEST-002', 'Mouse', 4.0, 50.0, 200.0)


In [ ]:
print(inspect.getsource(process_document))

def process_document(file_path):
    extension = os.path.splitext(file_path)[1].lower()

    try:
        if extension == ".pdf":
            text = extract_pdf_text(file_path)

            if not text.strip():
                text = extract_scanned_pdf_text(file_path)

            invoice = extract_invoice_data(text)

        elif extension in [".jpg", ".jpeg", ".png", ".tiff", ".tif"]:
            text = extract_image_text(file_path)
            invoice = extract_invoice_data(text)

        elif extension == ".csv":
            text = extract_spreadsheet_text(file_path)
            invoice = extract_csv_invoice(file_path)

        elif extension in [".xlsx", ".xls"]:
            text = extract_spreadsheet_text(file_path)
            invoice = extract_spreadsheet_invoice(file_path)

        else:
            raise ValueError("Unsupported file format")

        validation = validate_invoice(invoice)

        return {
            "status": "success",
            "document_type": (
     

In [ ]:
def extract_pdf_text(file_path):
    pdf = fitz.open(file_path)
    extracted_pages = []

    try:
        for page in pdf:
            text = page.get_text().strip()

            if not text:
                pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
                image = Image.frombytes(
                    "RGB",
                    [pix.width, pix.height],
                    pix.samples
                )
                text = pytesseract.image_to_string(image)

            extracted_pages.append(text)

    finally:
        pdf.close()

    return "\n".join(extracted_pages)

In [ ]:
pdf_test = extract_pdf_text(
    "/content/drive/MyDrive/AI_Document_Intelligence_System/document_intelligence/uploads/22fb4b83-054c-40f0-bd80-13a4b16698a0.pdf"
)

print("Extracted characters:", len(pdf_test))
print("Preview:")
print(pdf_test[:500])

Extracted characters: 139
Preview:
INVOICE
Invoice Number: INVALID-TEST-001
Date: 05/11/2026
Supplier: Test Supplies
Customer: Demo Buyer
Subtotal: 200
Tax: 30
Total: 250 BDT


In [ ]:
corrupted_path = os.path.join(
    project_root,
    "document_intelligence",
    "uploads",
    "corrupted_test.pdf"
)

with open(corrupted_path, "w") as f:
    f.write("This is not a valid PDF file.")

corrupted_result = process_document(corrupted_path)

print(corrupted_result)

{'status': 'failed', 'error': "Failed to open file '/content/drive/MyDrive/AI_Document_Intelligence_System/document_intelligence/uploads/corrupted_test.pdf' as type pdf."}


In [ ]:
unsupported_path = os.path.join(
    project_root,
    "document_intelligence",
    "uploads",
    "test_file.txt"
)

with open(unsupported_path, "w") as f:
    f.write("Testing unsupported file format.")

unsupported_result = process_document(unsupported_path)

print(unsupported_result)

{'status': 'failed', 'error': 'Unsupported file format'}


In [ ]:
pdf = fitz.open(
    "/content/drive/MyDrive/AI_Document_Intelligence_System/document_intelligence/uploads/22fb4b83-054c-40f0-bd80-13a4b16698a0.pdf"
)

print("Total pages:", len(pdf))

pdf.close()

Total pages: 1


In [ ]:
from reportlab.pdfgen import canvas

test_pdf_path = os.path.join(
    project_root,
    "document_intelligence",
    "uploads",
    "multi_page_test.pdf"
)

pdf_canvas = canvas.Canvas(test_pdf_path)

for page_number in range(1, 6):
    pdf_canvas.drawString(
        100,
        750,
        f"Testing PDF page number {page_number}"
    )
    pdf_canvas.showPage()

pdf_canvas.save()

test_pdf = fitz.open(test_pdf_path)

print("Test PDF created.")
print("Total pages:", len(test_pdf))

test_pdf.close()

Test PDF created.
Total pages: 5


In [ ]:
multi_page_text = extract_pdf_text(test_pdf_path)

print("Extracted text:")
print(multi_page_text)

print("\nAll pages detected:",
      all(f"page number {i}" in multi_page_text
          for i in range(1, 6)))

Extracted text:
Testing PDF page number 1
Testing PDF page number 2
Testing PDF page number 3
Testing PDF page number 4
Testing PDF page number 5

All pages detected: True


In [ ]:
import time

start_time = time.time()

multi_page_text = extract_pdf_text(test_pdf_path)

end_time = time.time()

print("Pages processed:", 5)
print("Processing time:", round(end_time - start_time, 3), "seconds")
print("Characters extracted:", len(multi_page_text))

Pages processed: 5
Processing time: 0.017 seconds
Characters extracted: 129


In [ ]:
def get_pdf_page_count(file_path):
    pdf = fitz.open(file_path)

    try:
        return len(pdf)
    finally:
        pdf.close()


page_count = get_pdf_page_count(test_pdf_path)

print("PDF page count:", page_count)

PDF page count: 5


In [ ]:
def evaluate_pdf_extraction(file_path):
    start_time = time.time()

    page_count = get_pdf_page_count(file_path)
    extracted_text = extract_pdf_text(file_path)

    processing_time = time.time() - start_time

    return {
        "filename": os.path.basename(file_path),
        "total_pages": page_count,
        "processing_time_seconds": round(processing_time, 4),
        "extracted_characters": len(extracted_text),
        "extraction_successful": bool(extracted_text.strip())
    }


evaluation = evaluate_pdf_extraction(test_pdf_path)

print(json.dumps(evaluation, indent=4))

{
    "filename": "multi_page_test.pdf",
    "total_pages": 5,
    "processing_time_seconds": 0.0239,
    "extracted_characters": 129,
    "extraction_successful": true
}


In [ ]:
evaluation_path = os.path.join(
    project_root,
    "document_intelligence",
    "logs",
    "pdf_evaluation_results.json"
)

os.makedirs(os.path.dirname(evaluation_path), exist_ok=True)

with open(evaluation_path, "w") as f:
    json.dump(evaluation, f, indent=4)

print("Evaluation results saved successfully.")
print("Location:", evaluation_path)

Evaluation results saved successfully.
Location: /content/drive/MyDrive/AI_Document_Intelligence_System/document_intelligence/logs/pdf_evaluation_results.json


In [ ]:
from PIL import Image, ImageDraw

image_path = os.path.join(
    project_root,
    "document_intelligence",
    "uploads",
    "ocr_test.png"
)

img = Image.new("RGB", (1200, 700), "white")
draw = ImageDraw.Draw(img)

invoice_lines = [
    "INVOICE",
    "Invoice Number: OCR-TEST-001",
    "Date: 05/11/2026",
    "Supplier: OCR Supplies",
    "Customer: Demo Buyer",
    "Subtotal: 500",
    "Tax: 75",
    "Total: 575 BDT"
]

for i, line in enumerate(invoice_lines):
    draw.text((80, 60 + i * 70), line, fill="black")

img.save(image_path)

print("Test image created:", image_path)

Test image created: /content/drive/MyDrive/AI_Document_Intelligence_System/document_intelligence/uploads/ocr_test.png


In [ ]:
ocr_text = extract_image_text(image_path)

print("OCR extracted text:")
print(ocr_text)

OCR extracted text:
INVOICE

Invoice Number: OCR TEST-001

Date: 05/11/2028

Supplier. CCR Supplies

Customer: Demo Buyer

Subtotat 500

Tax 75

Total 675 BOT



In [ ]:
ocr_invoice = extract_invoice_data(ocr_text)

print(json.dumps(ocr_invoice, indent=4))

{
    "invoice_number": "OCR",
    "invoice_date": "05/11/2028",
    "supplier_name": ". CCR Supplies",
    "customer_name": "Demo Buyer",
    "subtotal": null,
    "tax": "75",
    "total": "675",
    "currency": null,
    "line_items": []
}


In [ ]:
def extract_invoice_data(text):
    invoice = {
        "invoice_number": None,
        "invoice_date": None,
        "supplier_name": None,
        "customer_name": None,
        "subtotal": None,
        "tax": None,
        "total": None,
        "currency": None,
        "line_items": []
    }

    patterns = {
        "invoice_number": r"Invoice\s*(?:No|Number|#)\s*[:\-]?\s*([^\r\n]+)",
        "invoice_date": r"Date\s*[:\-]?\s*(\d{2}[/-]\d{2}[/-]\d{4})",
        "subtotal": r"\bSubtota[lt]\b\s*[:\-]?\s*([\d,]+\.?\d*)",
        "tax": r"\b(?:Tax|VAT)\b\s*[:\-]?\s*([\d,]+\.?\d*)",
        "total": r"(?<!Sub)\bTotal\b\s*[:\-]?\s*([\d,]+\.?\d*)"
    }

    for field, pattern in patterns.items():
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            value = match.group(1).strip()

            if field == "invoice_number":
                invoice[field] = value.strip(" .:-")
            else:
                invoice[field] = value

    supplier = re.search(
        r"Supplier\s*[:.]?\s*([^\r\n]+)",
        text,
        re.IGNORECASE
    )

    customer = re.search(
        r"Customer\s*[:.]?\s*([^\r\n]+)",
        text,
        re.IGNORECASE
    )

    if supplier:
        invoice["supplier_name"] = supplier.group(1).strip(" .:-")

    if customer:
        invoice["customer_name"] = customer.group(1).strip(" .:-")

    currency = re.search(
        r"\b(BDT|USD|EUR|GBP)\b",
        text,
        re.IGNORECASE
    )

    if currency:
        invoice["currency"] = currency.group(1).upper()

    return invoice

In [ ]:
ocr_invoice = extract_invoice_data(ocr_text)

print(json.dumps(ocr_invoice, indent=4))

{
    "invoice_number": "OCR TEST-001",
    "invoice_date": "05/11/2028",
    "supplier_name": "CCR Supplies",
    "customer_name": "Demo Buyer",
    "subtotal": "500",
    "tax": "75",
    "total": "675",
    "currency": null,
    "line_items": []
}


In [ ]:
def check_ocr_quality(ocr_text, invoice):
    warnings = []

    if invoice.get("subtotal") and invoice.get("tax") and invoice.get("total"):
        subtotal = float(invoice["subtotal"].replace(",", ""))
        tax = float(invoice["tax"].replace(",", ""))
        total = float(invoice["total"].replace(",", ""))

        if abs(subtotal + tax - total) > 0.01:
            warnings.append("Invoice amount mismatch: manual review required")

    if not invoice.get("currency"):
        warnings.append("Currency not confidently detected")

    if not invoice.get("invoice_number"):
        warnings.append("Invoice number missing or uncertain")

    if not invoice.get("invoice_date"):
        warnings.append("Invoice date missing or uncertain")

    return {
        "manual_review_required": bool(warnings),
        "warnings": warnings
    }


ocr_quality = check_ocr_quality(ocr_text, ocr_invoice)

print(json.dumps(ocr_quality, indent=4))

{
    "manual_review_required": true,
    "warnings": [
        "Invoice amount mismatch: manual review required",
        "Currency not confidently detected"
    ]
}


In [ ]:
def process_document(file_path):
    extension = os.path.splitext(file_path)[1].lower()

    try:
        if extension == ".pdf":
            text = extract_pdf_text(file_path)
            invoice = extract_invoice_data(text)

        elif extension in [".jpg", ".jpeg", ".png", ".tiff", ".tif"]:
            text = extract_image_text(file_path)
            invoice = extract_invoice_data(text)

        elif extension == ".csv":
            text = extract_spreadsheet_text(file_path)
            invoice = extract_csv_invoice(file_path)

        elif extension in [".xlsx", ".xls"]:
            text = extract_spreadsheet_text(file_path)
            invoice = extract_spreadsheet_invoice(file_path)

        else:
            raise ValueError("Unsupported file format")

        validation = validate_invoice(invoice)

        ocr_quality = (
            check_ocr_quality(text, invoice)
            if extension in [".jpg", ".jpeg", ".png", ".tiff", ".tif"]
            else {
                "manual_review_required": False,
                "warnings": []
            }
        )

        if ocr_quality["manual_review_required"]:
            validation["errors"].extend(ocr_quality["warnings"])
            validation["validation_status"] = "failed"

        return {
            "status": "success",
            "document_type": (
                "spreadsheet"
                if extension in [".csv", ".xlsx", ".xls"]
                else "document"
            ),
            "extracted_text": text,
            "invoice_data": invoice,
            "validation": validation,
            "ocr_quality": ocr_quality
        }

    except Exception as e:
        return {
            "status": "failed",
            "error": str(e)
        }

In [ ]:
image_result = process_document(image_path)

print(json.dumps(image_result, indent=4))

{
    "status": "success",
    "document_type": "document",
    "extracted_text": "INVOICE\n\nInvoice Number: OCR TEST-001\n\nDate: 05/11/2028\n\nSupplier. CCR Supplies\n\nCustomer: Demo Buyer\n\nSubtotat 500\n\nTax 75\n\nTotal 675 BOT\n",
    "invoice_data": {
        "invoice_number": "OCR TEST-001",
        "invoice_date": "05/11/2028",
        "supplier_name": "CCR Supplies",
        "customer_name": "Demo Buyer",
        "subtotal": "500",
        "tax": "75",
        "total": "675",
        "currency": null,
        "line_items": []
    },
    "validation": {
        "validation_status": "failed",
        "errors": [
            "Subtotal + tax does not equal total",
            "Invoice amount mismatch: manual review required",
            "Currency not confidently detected"
        ]
    },
    "ocr_quality": {
        "manual_review_required": true,
        "warnings": [
            "Invoice amount mismatch: manual review required",
            "Currency not confidently detect

In [ ]:
from google.colab import files

files.download(image_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
cursor = conn.cursor()

cursor.execute("""
    SELECT error_message
    FROM validation_errors
    WHERE document_id = 12
""")

errors = cursor.fetchall()

print("Validation errors for document 12:")
for error in errors:
    print("-", error[0])

Validation errors for document 12:
- Subtotal + tax does not equal total
- Invoice amount mismatch: manual review required
- Currency not confidently detected
- Duplicate invoice detected


In [ ]:
def update_existing_document(document_id, file_path, result):
    cursor = conn.cursor()

    invoice = result.get("invoice_data")

    if not invoice:
        raise ValueError("Invoice data missing from processing result")

    # Preserve all validation errors from the original processing
    validation = result.get("validation", validate_invoice(invoice))
    validation_errors = list(validation.get("errors", []))

    # Check for duplicates
    duplicate = is_duplicate_before_save(
        invoice,
        exclude_document_id=document_id
    )

    if duplicate and "Duplicate invoice detected" not in validation_errors:
        validation_errors.append("Duplicate invoice detected")

    status = "needs_review" if validation_errors else "success"

    cursor.execute("""
        UPDATE documents
        SET processing_status = ?
        WHERE id = ?
    """, (status, document_id))

    cursor.execute("""
        UPDATE invoices
        SET invoice_number = ?, invoice_date = ?,
            supplier_name = ?, customer_name = ?,
            subtotal = ?, tax = ?, total = ?, currency = ?
        WHERE document_id = ?
    """, (
        invoice.get("invoice_number"),
        invoice.get("invoice_date"),
        invoice.get("supplier_name"),
        invoice.get("customer_name"),
        invoice.get("subtotal"),
        invoice.get("tax"),
        invoice.get("total"),
        invoice.get("currency"),
        document_id
    ))

    cursor.execute(
        "SELECT id FROM invoices WHERE document_id = ?",
        (document_id,)
    )
    invoice_record = cursor.fetchone()

    if not invoice_record:
        raise ValueError("Existing invoice record not found")

    invoice_id = invoice_record[0]

    cursor.execute(
        "DELETE FROM line_items WHERE invoice_id = ?",
        (invoice_id,)
    )

    for item in invoice.get("line_items", []):
        cursor.execute("""
            INSERT INTO line_items
            (invoice_id, description, quantity, unit_price, amount)
            VALUES (?, ?, ?, ?, ?)
        """, (
            item.get("description"),
            item.get("quantity"),
            item.get("unit_price"),
            item.get("amount")
        ))

    cursor.execute(
        "DELETE FROM validation_errors WHERE document_id = ?",
        (document_id,)
    )

    for error in validation_errors:
        cursor.execute("""
            INSERT INTO validation_errors (document_id, error_message)
            VALUES (?, ?)
        """, (document_id, error))

    if result.get("extracted_text"):
        cursor.execute("""
            UPDATE extracted_text
            SET extracted_content = ?
            WHERE document_id = ?
        """, (result["extracted_text"], document_id))

    conn.commit()

    return {
        "document_id": document_id,
        "status": status,
        "duplicate_detected": duplicate,
        "validation_errors": validation_errors
    }

In [ ]:
cursor = conn.cursor()

cursor.execute("""
    SELECT id, document_id, invoice_number, total
    FROM invoices
    WHERE document_id = 12
""")

print(cursor.fetchone())

(12, 12, 'OCR TEST-001', 675.0)


In [ ]:
cursor = conn.cursor()

cursor.execute("""
    SELECT invoice_number, subtotal, tax, total
    FROM invoices
    WHERE document_id = 12
""")

print("Corrected invoice:", cursor.fetchone())

cursor.execute("""
    SELECT field_name, old_value, new_value
    FROM correction_history
    WHERE document_id = 12
""")

print("Correction history:", cursor.fetchall())

Corrected invoice: ('OCR TEST-001', 500.0, 75.0, 575.0)
Correction history: [('total', '675.0', '575')]


In [ ]:
for route in app.routes:
    if getattr(route, "path", None) == "/upload":
        print(inspect.getsource(route.endpoint))

@app.post("/upload")
async def upload_document(file: UploadFile = File(...)):
    allowed = [".pdf", ".jpg", ".jpeg", ".png", ".tiff",
               ".tif", ".xlsx", ".xls", ".csv"]

    extension = os.path.splitext(file.filename)[1].lower()

    if extension not in allowed:
        raise HTTPException(status_code=400,
                            detail="Unsupported file format")

    unique_name = str(uuid.uuid4()) + extension
    file_path = os.path.join(
        "document_intelligence/uploads", unique_name
    )

    with open(file_path, "wb") as f:
        f.write(await file.read())

    result = process_document(file_path)

    if result.get("status") == "failed":
        return {"status": "failed", "error": result.get("error")}

    invoice = result.get("invoice_data")
    duplicate = is_duplicate_before_save(invoice) if invoice else False

    validation_errors = result.get("validation", {}).get("errors", [])

    needs_review = duplicate or bool(validation_errors)

    documen

In [ ]:
print(inspect.getsource(save_invoice_to_database))

def save_invoice_to_database(file_path, result):
    cursor = conn.cursor()

    cursor.execute("""
    INSERT INTO documents (filename, file_type, processing_status)
    VALUES (?, ?, ?)
    """, (
        os.path.basename(file_path),
        os.path.splitext(file_path)[1].lower(),
        result.get("status", "failed")
    ))

    document_id = cursor.lastrowid

    # Save extracted text
    extracted_text = result.get("extracted_text", "")

    if extracted_text:
        save_extracted_text(document_id, extracted_text)

    # Save invoice information
    invoice = result.get("invoice_data")

    if invoice:
        def safe_float(value):
            try:
                return float(str(value).replace(",", "")) if value is not None else None
            except (ValueError, TypeError):
                return None

        cursor.execute("""
        INSERT INTO invoices (
            document_id, invoice_number, invoice_date,
            supplier_name, customer_name, subtotal,
       

In [ ]:
print(inspect.getsource(save_invoice_to_database))

def save_invoice_to_database(file_path, result):
    cursor = conn.cursor()

    cursor.execute("""
    INSERT INTO documents (filename, file_type, processing_status)
    VALUES (?, ?, ?)
    """, (
        os.path.basename(file_path),
        os.path.splitext(file_path)[1].lower(),
        result.get("status", "failed")
    ))

    document_id = cursor.lastrowid

    # Save extracted text
    extracted_text = result.get("extracted_text", "")

    if extracted_text:
        save_extracted_text(document_id, extracted_text)

    # Save invoice information
    invoice = result.get("invoice_data")

    if invoice:
        def safe_float(value):
            try:
                return float(str(value).replace(",", "")) if value is not None else None
            except (ValueError, TypeError):
                return None

        cursor.execute("""
        INSERT INTO invoices (
            document_id, invoice_number, invoice_date,
            supplier_name, customer_name, subtotal,
       

In [ ]:
from fastapi import BackgroundTasks

# Background processing function
async def process_upload_background(document_id, file_path):
    cursor = conn.cursor()

    try:
        result = process_document(file_path)

        if result.get("status") == "failed":
            cursor.execute("""
                UPDATE documents
                SET processing_status = 'failed'
                WHERE id = ?
            """, (document_id,))
            conn.commit()
            return

        invoice = result.get("invoice_data")
        validation_errors = list(
            result.get("validation", {}).get("errors", [])
        )

        duplicate = False

        if invoice:
            duplicate = is_duplicate_before_save(invoice)

            if duplicate:
                validation_errors.append("Duplicate invoice detected")

            def safe_float(value):
                try:
                    return float(str(value).replace(",", "")) if value is not None else None
                except (ValueError, TypeError):
                    return None

            cursor.execute("""
                INSERT INTO invoices (
                    document_id, invoice_number, invoice_date,
                    supplier_name, customer_name, subtotal,
                    tax, total, currency
                )
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                document_id,
                invoice.get("invoice_number"),
                invoice.get("invoice_date"),
                invoice.get("supplier_name"),
                invoice.get("customer_name"),
                safe_float(invoice.get("subtotal")),
                safe_float(invoice.get("tax")),
                safe_float(invoice.get("total")),
                invoice.get("currency")
            ))

            invoice_id = cursor.lastrowid

            for item in invoice.get("line_items", []):
                cursor.execute("""
                    INSERT INTO line_items (
                        invoice_id, description, quantity,
                        unit_price, amount
                    )
                    VALUES (?, ?, ?, ?, ?)
                """, (
                    invoice_id,
                    item.get("description"),
                    safe_float(item.get("quantity")),
                    safe_float(item.get("unit_price")),
                    safe_float(item.get("amount"))
                ))

        extracted_text = result.get("extracted_text", "")

        if extracted_text:
            save_extracted_text(document_id, extracted_text)

        for error in validation_errors:
            cursor.execute("""
                INSERT INTO validation_errors
                (document_id, error_message)
                VALUES (?, ?)
            """, (document_id, error))

        status = "needs_review" if validation_errors else "success"

        cursor.execute("""
            UPDATE documents
            SET processing_status = ?
            WHERE id = ?
        """, (status, document_id))

        conn.commit()

    except Exception as e:
        conn.rollback()

        cursor.execute("""
            UPDATE documents
            SET processing_status = 'failed'
            WHERE id = ?
        """, (document_id,))

        conn.commit()
        print("Background processing error:", str(e))


# Remove the previous upload route
upload_routes = [
    route for route in app.routes
    if getattr(route, "path", None) == "/upload"
]

for route in upload_routes:
    app.routes.remove(route)


# New asynchronous upload endpoint
@app.post("/upload")
async def upload_document(
    background_tasks: BackgroundTasks,
    file: UploadFile = File(...)
):
    allowed = [
        ".pdf", ".jpg", ".jpeg", ".png",
        ".tiff", ".tif", ".xlsx", ".xls", ".csv"
    ]

    extension = os.path.splitext(file.filename)[1].lower()

    if extension not in allowed:
        raise HTTPException(
            status_code=400,
            detail="Unsupported file format"
        )

    unique_name = str(uuid.uuid4()) + extension

    file_path = os.path.join(
        "document_intelligence/uploads",
        unique_name
    )

    with open(file_path, "wb") as f:
        f.write(await file.read())

    cursor = conn.cursor()

    cursor.execute("""
        INSERT INTO documents (
            filename, file_type, processing_status
        )
        VALUES (?, ?, ?)
    """, (unique_name, extension, "pending"))

    document_id = cursor.lastrowid
    conn.commit()

    background_tasks.add_task(
        process_upload_background,
        document_id,
        file_path
    )

    return {
        "document_id": document_id,
        "filename": file.filename,
        "status": "pending",
        "message": "Document uploaded. Processing started."
    }


print("Background processing endpoint configured.")

Background processing endpoint configured.


In [ ]:
server.should_exit = True
time.sleep(3)

config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=8000,
    log_level="info"
)

server = uvicorn.Server(config)

api_thread = threading.Thread(
    target=server.run,
    daemon=True
)

api_thread.start()
time.sleep(5)

print("Server running:", server.started)

INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [1511]
INFO:     Started server process [1511]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Server running: True


In [ ]:
!pip install gradio requests

In [ ]:
import gradio as gr
import requests
import json
import os

API_BASE = "http://127.0.0.1:8000"

def upload_invoice(file):
    if file is None:
        return "Please select a document."

    try:
        with open(file, "rb") as f:
            response = requests.post(
                f"{API_BASE}/upload",
                files={"file": (os.path.basename(file), f)},
                timeout=120
            )

        try:
            return json.dumps(response.json(), indent=4)
        except ValueError:
            return f"HTTP {response.status_code}\n{response.text[:1000]}"

    except Exception as e:
        return f"Upload error: {str(e)}"

def check_status(document_id):
    if not str(document_id).strip():
        return "Enter a document ID."

    try:
        response = requests.get(
            f"{API_BASE}/documents/{int(document_id)}/status"
        )
        return json.dumps(response.json(), indent=4)
    except Exception as e:
        return f"Status error: {str(e)}"

def search_documents(keyword):
    if not keyword.strip():
        return "Enter a search keyword."

    try:
        response = requests.get(
            f"{API_BASE}/search",
            params={"keyword": keyword}
        )
        return json.dumps(response.json(), indent=4)
    except Exception as e:
        return f"Search error: {str(e)}"

with gr.Blocks(title="AI Document Intelligence System") as demo:
    gr.Markdown("# AI Document Intelligence System")
    gr.Markdown("Upload, process, validate and search invoices.")

    with gr.Tab("Upload Document"):
        file_input = gr.File(label="Select an invoice", type="filepath")
        upload_button = gr.Button("Upload")
        upload_output = gr.Textbox(label="Upload Result", lines=10)
        upload_button.click(
            upload_invoice,
            inputs=file_input,
            outputs=upload_output
        )

    with gr.Tab("Check Status"):
        document_id = gr.Textbox(label="Document ID")
        status_button = gr.Button("Check Status")
        status_output = gr.Textbox(label="Processing Status", lines=5)
        status_button.click(
            check_status,
            inputs=document_id,
            outputs=status_output
        )

    with gr.Tab("Search Documents"):
        keyword = gr.Textbox(label="Search Keyword")
        search_button = gr.Button("Search")
        search_output = gr.Textbox(label="Search Results", lines=10)
        search_button.click(
            search_documents,
            inputs=keyword,
            outputs=search_output
        )

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ec3c33f4f0e71a1b89.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://ec3c33f4f0e71a1b89.gradio.live


In [ ]:
print("API URL:", API_BASE)

try:
    response = requests.get(API_BASE + "/")
    print("HTTP Status:", response.status_code)
    print("Response:", response.text[:1000])
except Exception as e:
    print("Connection error:", str(e))

API URL: https://8000-m-s-usc1c9-2xw777i57big4-c.us-central1.prod.r.colab.dev
HTTP Status: 404
Response: 


In [ ]:
for endpoint in ["/docs", "/openapi.json", "/documents"]:
    try:
        r = requests.get(API_BASE + endpoint)
        print(endpoint, "->", r.status_code)
        print(r.text[:300])
        print("-" * 40)
    except Exception as e:
        print(endpoint, "->", str(e))

/docs -> 404

----------------------------------------
/openapi.json -> 404

----------------------------------------
/documents -> 404

----------------------------------------


In [ ]:
from google.colab.output import eval_js

print("Server running:", server.started)

print("\nRegistered API routes:")
for route in app.routes:
    if hasattr(route, "methods"):
        print(route.methods, route.path)

public_url = eval_js("google.colab.kernel.proxyPort(8000)")

print("\nFresh API URL:", public_url)

API_BASE = public_url.rstrip("/")

response = requests.get(API_BASE + "/docs")

print("Swagger status:", response.status_code)

Server running: True

Registered API routes:
{'GET', 'HEAD'} /openapi.json
{'GET', 'HEAD'} /docs
{'GET', 'HEAD'} /docs/oauth2-redirect
{'GET', 'HEAD'} /redoc
{'GET'} /
{'GET'} /documents
{'GET'} /invoices
{'GET'} /documents/{document_id}/status
{'POST'} /retry/{document_id}
{'GET'} /search
{'PUT'} /invoices/{invoice_id}/correct
{'POST'} /upload

Fresh API URL: https://8000-m-s-usc1c9-2xw777i57big4-c.us-central1.prod.r.colab.dev
Swagger status: 404


In [ ]:
API_BASE = "http://127.0.0.1:8000"

for endpoint in ["/", "/docs", "/openapi.json", "/documents"]:
    try:
        r = requests.get(API_BASE + endpoint)
        print(endpoint, "->", r.status_code)
        print(r.text[:200])
        print("-" * 40)
    except Exception as e:
        print(endpoint, "->", str(e))

/ -> 200
{"message":"Document Intelligence API is running"}
----------------------------------------
/docs -> 200

    <!DOCTYPE html>
    <html>
    <head>
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <link type="text/css" rel="stylesheet" href="https://cdn.jsdelivr.net/npm/swag
----------------------------------------
/openapi.json -> 200
{"openapi":"3.1.0","info":{"title":"AI Document Intelligence System","version":"0.1.0"},"paths":{"/":{"get":{"summary":"Home","operationId":"home__get","responses":{"200":{"description":"Successful Re
----------------------------------------
/documents -> 200
[[1,"test_invoice.pdf",".pdf","success","2026-11-05 06:55:12"],[2,"8263d77b-8b1d-4c32-a38a-4a4e35b3d7bd.pdf",".pdf","success","2026-11-05 06:59:11"],[3,"22fb4b83-054c-40f0-bd80-13a4b16698a0.pdf",".pdf
----------------------------------------


In [ ]:
cursor = conn.cursor()

cursor.execute("""
    SELECT error_message
    FROM validation_errors
    WHERE document_id = 14
""")

print("Validation errors for document 14:")
for row in cursor.fetchall():
    print("-", row[0])

Validation errors for document 14:
- Subtotal + tax does not equal total
- Invoice amount mismatch: manual review required
- Currency not confidently detected
- Duplicate invoice detected


In [ ]:
print("API running:", requests.get("http://127.0.0.1:8000/").status_code)
print("Correction endpoint available:",
      any(getattr(r, "path", "") == "/invoices/{invoice_id}/correct"
          for r in app.routes))

API running: 200
Correction endpoint available: True


In [ ]:
response = requests.get("http://127.0.0.1:8000/invoices")

print("Status:", response.status_code)
print(json.dumps(response.json()[:3], indent=4))

Status: 200
[
    [
        1,
        1,
        "TEST-001",
        "05/11/2026",
        "Demo Technologies",
        "Sample Company",
        500.0,
        75.0,
        575.0,
        "BDT"
    ],
    [
        2,
        2,
        "UPLOAD-TEST-002",
        "05/11/2026",
        "Demo Systems",
        "Test Buyer",
        200.0,
        30.0,
        200.0,
        "BDT"
    ],
    [
        3,
        3,
        "INVALID-TEST-001",
        "05/11/2026",
        "Test Supplies",
        "Demo Buyer",
        200.0,
        30.0,
        250.0,
        "BDT"
    ]
]


In [ ]:
import gradio as gr
import requests
import json

API_BASE = "http://127.0.0.1:8000"

def get_invoice_choices():
    response = requests.get(f"{API_BASE}/invoices")
    records = response.json()

    return [
        (f"ID {row[0]} | {row[2]} | {row[4]}", row[0])
        for row in records
    ]

def view_invoice(invoice_id):
    if invoice_id is None:
        return "Select an invoice."

    records = requests.get(f"{API_BASE}/invoices").json()

    for row in records:
        if row[0] == invoice_id:
            invoice = {
                "invoice_id": row[0],
                "document_id": row[1],
                "invoice_number": row[2],
                "invoice_date": row[3],
                "supplier_name": row[4],
                "customer_name": row[5],
                "subtotal": row[6],
                "tax": row[7],
                "total": row[8],
                "currency": row[9]
            }
            return json.dumps(invoice, indent=4)

    return "Invoice not found."

def correct_invoice(invoice_id, field_name, new_value):
    if invoice_id is None:
        return "Please select an invoice."

    if not field_name or not new_value.strip():
        return "Select a field and enter its corrected value."

    try:
        response = requests.put(
            f"{API_BASE}/invoices/{invoice_id}/correct",
            json={
                "field_name": field_name,
                "new_value": new_value
            }
        )
        return json.dumps(response.json(), indent=4)

    except Exception as e:
        return f"Correction error: {str(e)}"


with gr.Blocks(title="AI Document Intelligence System") as review_demo:

    gr.Markdown("# AI Document Intelligence System")
    gr.Markdown("Upload, process, validate, search and correct invoices.")

    with gr.Tab("Upload Document"):
        file_input = gr.File(label="Select an invoice", type="filepath")
        upload_button = gr.Button("Upload")
        upload_output = gr.Textbox(label="Upload Result", lines=8)

        def upload_invoice(file):
            if file is None:
                return "Please select a document."

            with open(file, "rb") as f:
                response = requests.post(
                    f"{API_BASE}/upload",
                    files={"file": (os.path.basename(file), f)}
                )

            return json.dumps(response.json(), indent=4)

        upload_button.click(
            upload_invoice, inputs=file_input, outputs=upload_output
        )

    with gr.Tab("Check Status"):
        document_id = gr.Textbox(label="Document ID")
        status_button = gr.Button("Check Status")
        status_output = gr.Textbox(label="Processing Status", lines=5)

        def check_status(document_id):
            if not str(document_id).strip():
                return "Enter a document ID."

            response = requests.get(
                f"{API_BASE}/documents/{int(document_id)}/status"
            )
            return json.dumps(response.json(), indent=4)

        status_button.click(
            check_status, inputs=document_id, outputs=status_output
        )

    with gr.Tab("Search Documents"):
        keyword = gr.Textbox(label="Search Keyword")
        search_button = gr.Button("Search")
        search_output = gr.Textbox(label="Search Results", lines=10)

        def search_documents(keyword):
            if not keyword.strip():
                return "Enter a search keyword."

            response = requests.get(
                f"{API_BASE}/search",
                params={"keyword": keyword}
            )
            return json.dumps(response.json(), indent=4)

        search_button.click(
            search_documents, inputs=keyword, outputs=search_output
        )

    with gr.Tab("Review & Correct"):
        invoice_choices = get_invoice_choices()

        invoice_selector = gr.Dropdown(
            choices=invoice_choices,
            label="Select Invoice"
        )

        view_button = gr.Button("View Invoice")
        invoice_display = gr.Textbox(
            label="Extracted Invoice Details",
            lines=12
        )

        view_button.click(
            view_invoice,
            inputs=invoice_selector,
            outputs=invoice_display
        )

        gr.Markdown("### Correct an Invoice Field")

        field_selector = gr.Dropdown(
            choices=[
                "invoice_number",
                "invoice_date",
                "supplier_name",
                "customer_name",
                "subtotal",
                "tax",
                "total",
                "currency"
            ],
            label="Field to Correct"
        )

        new_value = gr.Textbox(label="Corrected Value")

        correct_button = gr.Button("Submit Correction")
        correction_output = gr.Textbox(
            label="Correction Result",
            lines=8
        )

        correct_button.click(
            correct_invoice,
            inputs=[invoice_selector, field_selector, new_value],
            outputs=correction_output
        )

review_demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d97044a07a37cbb616.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://d97044a07a37cbb616.gradio.live


In [ ]:
try:
    response = requests.put(
        "http://127.0.0.1:8000/invoices/14/correct",
        json={
            "field_name": "total",
            "new_value": "575"
        },
        timeout=20
    )
    print("Status:", response.status_code)
    print(response.text)
except Exception as e:
    print("Error:", repr(e))

Error: ReadTimeout(ReadTimeoutError("HTTPConnectionPool(host='127.0.0.1', port=8000): Read timed out. (read timeout=20)"))


In [ ]:
print("Server running:", server.started)

try:
    r = requests.get(
        "http://127.0.0.1:8000/",
        timeout=5
    )
    print("API status:", r.status_code)
    print("API response:", r.text)
except Exception as e:
    print("API error:", repr(e))

cursor = conn.cursor()
cursor.execute("""
    SELECT invoice_number, subtotal, tax, total
    FROM invoices
    WHERE id = 14
""")
print("Current invoice record:", cursor.fetchone())

Server running: True
API error: ReadTimeout(ReadTimeoutError("HTTPConnectionPool(host='127.0.0.1', port=8000): Read timed out. (read timeout=5)"))
Current invoice record: ('OCR TEST-001', 500.0, 75.0, 675.0)


In [ ]:
server.should_exit = True
time.sleep(3)

print("Old server thread alive:", api_thread.is_alive())

Old server thread alive: True


In [ ]:
time.sleep(5)
print("Old server thread alive:", api_thread.is_alive())
print("Server running:", server.started)

Old server thread alive: True
Server running: True


In [ ]:
import sys
import traceback

print("API thread ID:", api_thread.ident)

frames = sys._current_frames()
frame = frames.get(api_thread.ident)

if frame:
    print("\nAPI thread activity:")
    traceback.print_stack(frame)
else:
    print("API thread stack not available.")

API thread ID: 140384059905728

API thread activity:


  File "/usr/lib/python3.13/threading.py", line 1015, in _bootstrap
    self._bootstrap_inner()
  File "/usr/lib/python3.13/threading.py", line 1044, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.13/threading.py", line 995, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/server.py", line 86, in run
    return asyncio_run(self.serve(sockets=sockets), loop_factory=self.config.get_loop_factory())
  File "/usr/lib/python3.13/asyncio/runners.py", line 196, in run
    return runner.run(main)
  File "/usr/lib/python3.13/asyncio/runners.py", line 119, in run
    return self._loop.run_until_complete(task)


In [ ]:
import threading
import sys
import traceback

frames = sys._current_frames()

for thread in threading.enumerate():
    print("\nTHREAD:", thread.name)
    print("Alive:", thread.is_alive())

    frame = frames.get(thread.ident)

    if frame:
        print("".join(traceback.format_stack(frame)[-12:]))


THREAD: MainThread
Alive: True
  File "/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py", line 3578, in run_code
    return outflag
  File "/tmp/ipykernel_1511/659395970.py", line 5, in <cell line: 0>
    frames = sys._current_frames()


THREAD: IOPub
Alive: True
  File "/usr/lib/python3.13/asyncio/base_events.py", line 684, in run_forever
    self._run_once()
  File "/usr/local/lib/python3.13/dist-packages/nest_asyncio.py", line 139, in _run_once
    handle = None
  File "/usr/lib/python3.13/asyncio/events.py", line 105, in _run
    self = None  # Needed to break cycles when an exception occurs.
  File "/usr/local/lib/python3.13/dist-packages/tornado/platform/asyncio.py", line 200, in _handle_events
    handler_func(fileobj, events)
  File "/usr/local/lib/python3.13/dist-packages/zmq/eventloop/zmqstream.py", line 609, in _handle_events
    self._rebuild_io_state()
  File "/usr/local/lib/python3.13/dist-packages/zmq/eventloop/zmqstream.py", line 629, in _handle

In [ ]:
import sys
import traceback

frames = sys._current_frames()
frame = frames.get(api_thread.ident)

if frame:
    print("API SERVER THREAD:")
    traceback.print_stack(frame)
else:
    print("API server thread not found.")

API SERVER THREAD:


  File "/usr/lib/python3.13/threading.py", line 1015, in _bootstrap
    self._bootstrap_inner()
  File "/usr/lib/python3.13/threading.py", line 1044, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.13/threading.py", line 995, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/server.py", line 86, in run
    return asyncio_run(self.serve(sockets=sockets), loop_factory=self.config.get_loop_factory())
  File "/usr/lib/python3.13/asyncio/runners.py", line 196, in run
    return runner.run(main)
  File "/usr/lib/python3.13/asyncio/runners.py", line 119, in run
    return self._loop.run_until_complete(task)


In [ ]:
server.force_exit = True
server.should_exit = True

time.sleep(5)

print("Old server thread alive:", api_thread.is_alive())
print("Server running:", server.started)

ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 422, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        self.scope, self.receive, self.send
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/middleware/proxy_headers.py", line 63, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/fastapi/applications.py", line 1163, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.13/dist-packages/starlette/applications.py", line 96, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.13/dist-packages/starlette/middleware/errors.py", line 164, i

Old server thread alive: False
Server running: True


In [ ]:
config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=8000,
    log_level="info"
)

server = uvicorn.Server(config)

api_thread = threading.Thread(
    target=server.run,
    daemon=True
)

api_thread.start()

time.sleep(5)

print("New server thread alive:", api_thread.is_alive())

try:
    r = requests.get(
        "http://127.0.0.1:8000/",
        timeout=10
    )
    print("API status:", r.status_code)
    print("Response:", r.text)
except Exception as e:
    print("API error:", repr(e))

INFO:     Started server process [1511]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


New server thread alive: True
INFO:     127.0.0.1:56544 - "GET / HTTP/1.1" 200 OK
API status: 200
Response: {"message":"Document Intelligence API is running"}


In [ ]:
import inspect

print("CORRECT INVOICE FUNCTION:")
print(inspect.getsource(correct_invoice))

print("\nREVALIDATION FUNCTION:")
print(inspect.getsource(revalidate_invoice_record))

CORRECT INVOICE FUNCTION:
def correct_invoice(invoice_id, field_name, new_value):
    if invoice_id is None:
        return "Please select an invoice."

    if not field_name or not new_value.strip():
        return "Select a field and enter its corrected value."

    try:
        response = requests.put(
            f"{API_BASE}/invoices/{invoice_id}/correct",
            json={
                "field_name": field_name,
                "new_value": new_value
            }
        )
        return json.dumps(response.json(), indent=4)

    except Exception as e:
        return f"Correction error: {str(e)}"


REVALIDATION FUNCTION:
def revalidate_invoice_record(invoice_id):
    cursor = conn.cursor()

    cursor.execute("""
        SELECT document_id, invoice_number, invoice_date,
               supplier_name, customer_name, subtotal, tax,
               total, currency
        FROM invoices
        WHERE id = ?
    """, (invoice_id,))

    record = cursor.fetchone()

    if not record:

In [ ]:
def correct_invoice(invoice_id, field_name, new_value):
    allowed_fields = [
        "invoice_number", "invoice_date", "supplier_name",
        "customer_name", "subtotal", "tax", "total", "currency"
    ]

    if field_name not in allowed_fields:
        return {"status": "failed", "error": "Invalid field name"}

    if new_value is None or not str(new_value).strip():
        return {"status": "failed", "error": "Corrected value cannot be empty"}

    cursor = conn.cursor()

    cursor.execute(
        "SELECT document_id, " + field_name +
        " FROM invoices WHERE id = ?",
        (invoice_id,)
    )

    record = cursor.fetchone()

    if not record:
        return {"status": "failed", "error": "Invoice not found"}

    document_id, old_value = record

    if field_name in ["subtotal", "tax", "total"]:
        try:
            new_value = float(str(new_value).replace(",", ""))
        except ValueError:
            return {"status": "failed", "error": "Enter a valid number"}

    try:
        cursor.execute(
            f"UPDATE invoices SET {field_name} = ? WHERE id = ?",
            (new_value, invoice_id)
        )

        cursor.execute("""
            INSERT INTO correction_history
            (document_id, field_name, old_value, new_value)
            VALUES (?, ?, ?, ?)
        """, (document_id, field_name, str(old_value), str(new_value)))

        conn.commit()

        revalidation_result = revalidate_invoice_record(invoice_id)

        return {
            "message": "Corrected successfully",
            "invoice_id": invoice_id,
            "field_name": field_name,
            "old_value": old_value,
            "new_value": new_value,
            "revalidation": revalidation_result
        }

    except Exception as e:
        conn.rollback()
        return {"status": "failed", "error": str(e)}


print("Database correction function restored.")
print("Current invoice 14:")

cursor = conn.cursor()
cursor.execute("""
    SELECT invoice_number, subtotal, tax, total
    FROM invoices WHERE id = 14
""")
print(cursor.fetchone())

Database correction function restored.
Current invoice 14:
('OCR TEST-001', 500.0, 75.0, 675.0)


In [ ]:
print(inspect.getsource(correct_invoice_api))

cursor = conn.cursor()
cursor.execute("PRAGMA table_info(correction_history)")
print("\nCorrection history table:")
print(cursor.fetchall())

@app.put("/invoices/{invoice_id}/correct")
def correct_invoice_api(
    invoice_id: int,
    correction: CorrectionRequest
):
    correction_result = correct_invoice(
        invoice_id,
        correction.field_name,
        correction.new_value
    )

    return correction_result


Correction history table:
[(0, 'id', 'INTEGER', 0, None, 1), (1, 'document_id', 'INTEGER', 0, None, 0), (2, 'field_name', 'TEXT', 0, None, 0), (3, 'old_value', 'TEXT', 0, None, 0), (4, 'new_value', 'TEXT', 0, None, 0), (5, 'corrected_at', 'TIMESTAMP', 0, 'CURRENT_TIMESTAMP', 0)]


In [ ]:
def correct_invoice_db(invoice_id, field_name, new_value):
    if invoice_id is None:
        return {"error": "Please select an invoice."}

    allowed_fields = [
        "invoice_number", "invoice_date", "supplier_name",
        "customer_name", "subtotal", "tax", "total", "currency"
    ]

    if field_name not in allowed_fields:
        return {"error": "Invalid field selected."}

    if new_value is None or not str(new_value).strip():
        return {"error": "Enter a corrected value."}

    cursor = conn.cursor()

    cursor.execute(
        f"SELECT document_id, {field_name} FROM invoices WHERE id = ?",
        (invoice_id,)
    )
    record = cursor.fetchone()

    if not record:
        return {"error": "Invoice not found."}

    document_id, old_value = record

    try:
        if field_name in ["subtotal", "tax", "total"]:
            new_value = float(str(new_value).replace(",", ""))

        cursor.execute(
            f"UPDATE invoices SET {field_name} = ? WHERE id = ?",
            (new_value, invoice_id)
        )

        cursor.execute("""
            INSERT INTO correction_history
            (document_id, field_name, old_value, new_value)
            VALUES (?, ?, ?, ?)
        """, (document_id, field_name, str(old_value), str(new_value)))

        conn.commit()

        validation_result = revalidate_invoice_record(invoice_id)

        return {
            "message": "Corrected successfully",
            "invoice_id": invoice_id,
            "field_name": field_name,
            "old_value": old_value,
            "new_value": new_value,
            "revalidation": validation_result
        }

    except Exception as e:
        conn.rollback()
        return {"error": str(e)}


print("Database correction function restored:", correct_invoice_db.__name__)

Database correction function restored: correct_invoice_db


In [ ]:
@app.put("/invoices/{invoice_id}/correct")
def correct_invoice_api(
    invoice_id: int,
    correction: CorrectionRequest
):
    return correct_invoice_db(
        invoice_id,
        correction.field_name,
        correction.new_value
    )

print("Correction API endpoint updated.")

Correction API endpoint updated.


In [ ]:
correction_routes = [
    route for route in app.routes
    if getattr(route, "path", None) == "/invoices/{invoice_id}/correct"
    and "PUT" in getattr(route, "methods", set())
]

print("Correction routes found:", len(correction_routes))

# Keep only the latest route, which calls correct_invoice_db
if correction_routes:
    latest_route = correction_routes[-1]

    app.router.routes = [
        route for route in app.router.routes
        if route not in correction_routes
    ]

    app.router.routes.append(latest_route)

print("Correction routes remaining:", sum(
    1 for route in app.routes
    if getattr(route, "path", None) == "/invoices/{invoice_id}/correct"
    and "PUT" in getattr(route, "methods", set())
))

Correction routes found: 2
Correction routes remaining: 1


In [ ]:
def correct_invoice_gradio(invoice_id, field_name, new_value):
    if invoice_id is None:
        return "Please select an invoice."

    if not field_name or not str(new_value).strip():
        return "Select a field and enter its corrected value."

    try:
        response = requests.put(
            f"{API_BASE}/invoices/{invoice_id}/correct",
            json={
                "field_name": field_name,
                "new_value": new_value
            },
            timeout=30
        )
        return json.dumps(response.json(), indent=4)

    except Exception as e:
        return f"Correction error: {str(e)}"


print("Separate Gradio correction callback created.")

Separate Gradio correction callback created.


In [ ]:
print("API_BASE:", API_BASE)
print("Correction function:", correct_invoice_gradio)
print("Requests imported:", requests is not None)
print("JSON imported:", json is not None)

API_BASE: http://127.0.0.1:8000
Correction function: <function correct_invoice_gradio at 0x7fad8d17f600>
Requests imported: True
JSON imported: True


In [ ]:
import requests

for endpoint in ["/", "/docs", "/openapi.json"]:
    try:
        response = requests.get(
            f"{API_BASE}{endpoint}",
            timeout=10
        )
        print(endpoint, "->", response.status_code)
        print(response.text[:300])
        print("-" * 50)

    except Exception as e:
        print(endpoint, "-> ERROR:", str(e))

INFO:     127.0.0.1:44854 - "GET / HTTP/1.1" 200 OK
/ -> 200
{"message":"Document Intelligence API is running"}
--------------------------------------------------
INFO:     127.0.0.1:44864 - "GET /docs HTTP/1.1" 200 OK
/docs -> 200

    <!DOCTYPE html>
    <html>
    <head>
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <link type="text/css" rel="stylesheet" href="https://cdn.jsdelivr.net/npm/swagger-ui-dist@5/swagger-ui.css">
    <link rel="shortcut icon" href="https://fastapi.tiangolo.com/img/
--------------------------------------------------
INFO:     127.0.0.1:44878 - "GET /openapi.json HTTP/1.1" 200 OK
/openapi.json -> 200
{"openapi":"3.1.0","info":{"title":"AI Document Intelligence System","version":"0.1.0"},"paths":{"/":{"get":{"summary":"Home","operationId":"home__get","responses":{"200":{"description":"Successful Response","content":{"application/json":{"schema":{}}}}}}},"/documents":{"get":{"summary":"Get Documen
------------------------------

In [ ]:
schema = requests.get(f"{API_BASE}/openapi.json").json()

print("REGISTERED INVOICE ENDPOINTS:")
for path, methods in schema["paths"].items():
    if "invoice" in path.lower():
        print(path, list(methods.keys()))

INFO:     127.0.0.1:43244 - "GET /openapi.json HTTP/1.1" 200 OK
REGISTERED INVOICE ENDPOINTS:
/invoices ['get']
/invoices/{invoice_id}/correct ['put']


In [ ]:
response = requests.get(f"{API_BASE}/invoices", timeout=30)

print("Status:", response.status_code)
print(json.dumps(response.json(), indent=4))

INFO:     127.0.0.1:59798 - "GET /invoices HTTP/1.1" 200 OK
Status: 200
[
    [
        1,
        1,
        "TEST-001",
        "05/11/2026",
        "Demo Technologies",
        "Sample Company",
        500.0,
        75.0,
        575.0,
        "BDT"
    ],
    [
        2,
        2,
        "UPLOAD-TEST-002",
        "05/11/2026",
        "Demo Systems",
        "Test Buyer",
        200.0,
        30.0,
        200.0,
        "BDT"
    ],
    [
        3,
        3,
        "INVALID-TEST-001",
        "05/11/2026",
        "Test Supplies",
        "Demo Buyer",
        200.0,
        30.0,
        250.0,
        "BDT"
    ],
    [
        4,
        4,
        "INVALID-TEST-001",
        "05/11/2026",
        "Test Supplies",
        "Demo Buyer",
        200.0,
        30.0,
        250.0,
        "BDT"
    ],
    [
        5,
        5,
        "XLS-TEST-001",
        "05/11/2026",
        "Spreadsheet Supplies",
        "Demo Buyer",
        350.0,
        52.5,
        402

In [ ]:
result = correct_invoice_gradio(
    invoice_id=10,
    field_name="supplier_name",
    new_value="CSV Supplies Updated"
)

print(result)

INFO:     127.0.0.1:35956 - "PUT /invoices/10/correct HTTP/1.1" 200 OK
{
    "message": "Corrected successfully",
    "invoice_id": 10,
    "field_name": "supplier_name",
    "old_value": "CSV Supplies",
    "new_value": "CSV Supplies Updated",
    "revalidation": {
        "document_id": 10,
        "status": "success",
        "validation_errors": []
    }
}


In [ ]:
response = requests.get(f"{API_BASE}/invoices", timeout=30)

for invoice in response.json():
    if invoice[0] == 10:
        print("Updated invoice:", invoice)

INFO:     127.0.0.1:41836 - "GET /invoices HTTP/1.1" 200 OK
Updated invoice: [10, 10, 'CSV-TEST-002', '05/11/2026', 'CSV Supplies Updated', 'Demo Buyer', 500.0, 75.0, 575.0, 'BDT']


In [ ]:
print(review_demo)

Gradio Blocks instance: 5 backend functions
-------------------------------------------
fn_index=0
 inputs:
 |-<gradio.components.file.File object at 0x7fad8d444690>
 outputs:
 |-<gradio.components.textbox.Textbox object at 0x7fad8d444910>
fn_index=1
 inputs:
 |-<gradio.components.textbox.Textbox object at 0x7fad8d444cd0>
 outputs:
 |-<gradio.components.textbox.Textbox object at 0x7fad8d444f50>
fn_index=2
 inputs:
 |-<gradio.components.textbox.Textbox object at 0x7fad8d445310>
 outputs:
 |-<gradio.components.textbox.Textbox object at 0x7fad8d445590>
fn_index=3
 inputs:
 |-<gradio.components.dropdown.Dropdown object at 0x7fad8d954980>
 outputs:
 |-<gradio.components.textbox.Textbox object at 0x7fad8d445a90>
fn_index=4
 inputs:
 |-<gradio.components.dropdown.Dropdown object at 0x7fad8d954980>
 |-<gradio.components.dropdown.Dropdown object at 0x7fad8d445e50>
 |-<gradio.components.textbox.Textbox object at 0x7fad8d445f90>
 outputs:
 |-<gradio.components.textbox.Textbox object at 0x7fad8d44

In [ ]:
print("GRADIO DEPENDENCIES:")
for i, dep in enumerate(review_demo.config["dependencies"]):
    print(
        f"\nEvent {i}:",
        "inputs =", dep.get("inputs"),
        "| outputs =", dep.get("outputs"),
        "| api_name =", dep.get("api_name")
    )

GRADIO DEPENDENCIES:

Event 0: inputs = [46] | outputs = [48] | api_name = upload_invoice

Event 1: inputs = [51] | outputs = [53] | api_name = check_status

Event 2: inputs = [57] | outputs = [59] | api_name = search_documents

Event 3: inputs = [63] | outputs = [65] | api_name = view_invoice

Event 4: inputs = [63, 67, 68] | outputs = [70] | api_name = correct_invoice


In [ ]:
print("Local URL:", getattr(review_demo, "local_url", None))
print("Public URL:", getattr(review_demo, "share_url", None))
print("Server:", getattr(review_demo, "server", None))

Local URL: http://127.0.0.1:7860/
Public URL: https://d97044a07a37cbb616.gradio.live
Server: <gradio.http_server.Server object at 0x7fad8d39bd90>


In [ ]:
print("Running:", review_demo.is_running)
print("Public URL:", review_demo.share_url)

Running: True
Public URL: https://d97044a07a37cbb616.gradio.live


In [ ]:
for url in [
    "http://127.0.0.1:7860/",
    review_demo.share_url
]:
    try:
        r = requests.get(url, timeout=20)
        print(url)
        print("Status:", r.status_code)
        print("Response:", r.text[:200])
        print("-" * 40)
    except Exception as e:
        print("Error:", str(e))

Error: HTTPConnectionPool(host='127.0.0.1', port=7860): Max retries exceeded with url: / (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7fad8d64f570>: Failed to establish a new connection: [Errno 111] Connection refused'))
https://d97044a07a37cbb616.gradio.live
Status: 404
Response: 
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>No interface is running</title>
</head>
<style>
    body{
    font-family: 'Lato', sans-serif;
    color: #888;
    text-
----------------------------------------


In [ ]:
import requests
import gradio as gr

print("Gradio version:", gr.__version__)
print("App running:", review_demo.is_running)
print("Local URL:", review_demo.local_url)
print("Public URL:", review_demo.share_url)

try:
    r = requests.get("http://127.0.0.1:7860/", timeout=10)
    print("Local server status:", r.status_code)
except Exception as e:
    print("Local server error:", repr(e))

try:
    r = requests.get(review_demo.share_url, timeout=20)
    print("Public server status:", r.status_code)
    print("Public response:", r.text[:150])
except Exception as e:
    print("Public server error:", repr(e))

Gradio version: 6.31.0
App running: True
Local URL: http://localhost:7860/
Public URL: https://d97044a07a37cbb616.gradio.live
Local server status: 200
Public server status: 404
Public response: 
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>No interface is running</title>
</head>
<style>
    body{
    font-fami


In [ ]:
from google.colab.output import eval_js

proxy_url = eval_js(
    "google.colab.kernel.proxyPort(7860)"
)

print("YOUR APP LINK:", proxy_url)

YOUR APP LINK: https://7860-m-s-usc1c9-2xw777i57big4-c.us-central1.prod.r.colab.dev


In [ ]:
for component in review_demo.config["components"]:
    if component["id"] in [48, 63]:
        print("\nComponent ID:", component["id"])
        print("Type:", component["type"])
        print("Properties:", component.get("props", {}))


Component ID: 48
Type: textbox
Properties: {'type': 'text', 'lines': 8, 'label': 'Upload Result', 'show_label': True, 'container': True, 'min_width': 160, 'visible': True, 'autofocus': False, 'autoscroll': True, 'elem_classes': [], 'preserved_by_key': ['value'], 'rtl': False, 'buttons': [], 'submit_btn': False, 'stop_btn': False, 'name': 'textbox', '_selectable': False}

Component ID: 63
Type: dropdown
Properties: {'choices': [('ID 1 | TEST-001 | Demo Technologies', 1), ('ID 2 | UPLOAD-TEST-002 | Demo Systems', 2), ('ID 3 | INVALID-TEST-001 | Test Supplies', 3), ('ID 4 | INVALID-TEST-001 | Test Supplies', 4), ('ID 5 | XLS-TEST-001 | Spreadsheet Supplies', 5), ('ID 6 | CSV-TEST-001 | CSV Supplies', 6), ('ID 7 | INVALID-TEST-001 | Test Supplies', 7), ('ID 8 | XLS-TEST-001 | Legacy Supplies', 8), ('ID 9 | XLS-TEST-001 | Legacy Supplies', 9), ('ID 10 | CSV-TEST-002 | CSV Supplies', 10), ('ID 11 | OCR TEST-001 | CCR Supplies', 11), ('ID 12 | OCR TEST-001 | CCR Supplies', 12), ('ID 13 | OCR

In [ ]:
print("Registered Gradio functions:")

for index, block_fn in review_demo.fns.items():
    print(
        "\nFunction index:", index,
        "\nFunction name:", getattr(block_fn.fn, "__name__", "unknown")
    )

Registered Gradio functions:

Function index: 0 
Function name: upload_invoice

Function index: 1 
Function name: check_status

Function index: 2 
Function name: search_documents

Function index: 3 
Function name: view_invoice

Function index: 4 
Function name: correct_invoice


In [ ]:
import inspect

print("VIEW INVOICE FUNCTION:")
print(inspect.getsource(view_invoice))

print("\nUPLOAD FUNCTION:")
print(inspect.getsource(upload_invoice))

VIEW INVOICE FUNCTION:
def view_invoice(invoice_id):
    if invoice_id is None:
        return "Select an invoice."

    records = requests.get(f"{API_BASE}/invoices").json()

    for row in records:
        if row[0] == invoice_id:
            invoice = {
                "invoice_id": row[0],
                "document_id": row[1],
                "invoice_number": row[2],
                "invoice_date": row[3],
                "supplier_name": row[4],
                "customer_name": row[5],
                "subtotal": row[6],
                "tax": row[7],
                "total": row[8],
                "currency": row[9]
            }
            return json.dumps(invoice, indent=4)

    return "Invoice not found."


UPLOAD FUNCTION:
        def upload_invoice(file):
            if file is None:
                return "Please select a document."

            with open(file, "rb") as f:
                response = requests.post(
                    f"{API_BASE}/upload",
             

In [ ]:
invoice_dropdown = review_demo.blocks[63]

print("Dropdown component:", invoice_dropdown)
print("Current choices:", invoice_dropdown.choices)
print("Current value:", invoice_dropdown.value)

Dropdown component: <gradio.components.dropdown.Dropdown object at 0x7fad8d954980>
Current choices: [('ID 1 | TEST-001 | Demo Technologies', 1), ('ID 2 | UPLOAD-TEST-002 | Demo Systems', 2), ('ID 3 | INVALID-TEST-001 | Test Supplies', 3), ('ID 4 | INVALID-TEST-001 | Test Supplies', 4), ('ID 5 | XLS-TEST-001 | Spreadsheet Supplies', 5), ('ID 6 | CSV-TEST-001 | CSV Supplies', 6), ('ID 7 | INVALID-TEST-001 | Test Supplies', 7), ('ID 8 | XLS-TEST-001 | Legacy Supplies', 8), ('ID 9 | XLS-TEST-001 | Legacy Supplies', 9), ('ID 10 | CSV-TEST-002 | CSV Supplies', 10), ('ID 11 | OCR TEST-001 | CCR Supplies', 11), ('ID 12 | OCR TEST-001 | CCR Supplies', 12), ('ID 13 | OCR TEST-001 | CCR Supplies', 13), ('ID 14 | OCR TEST-001 | CCR Supplies', 14)]
Current value: 1


In [ ]:
# Fetch the latest invoices from FastAPI
response = requests.get(f"{API_BASE}/invoices", timeout=30)
records = response.json()

# Generate updated dropdown choices
new_choices = [
    (f"ID {row[0]} | {row[2]} | {row[4]}", row[0])
    for row in records
]

# Update the existing dropdown
invoice_dropdown.choices = new_choices
invoice_dropdown.value = 15

print("Updated invoice choices:")
for choice in new_choices:
    print(choice)

# Restart Gradio with updated dropdown configuration
review_demo.close()

launch_result = review_demo.launch(
    server_name="0.0.0.0",
    server_port=7860,
    share=False,
    prevent_thread_lock=True
)

print("\nGradio relaunched successfully.")

Updated invoice choices:
('ID 1 | TEST-001 | Demo Technologies', 1)
('ID 2 | UPLOAD-TEST-002 | Demo Systems', 2)
('ID 3 | INVALID-TEST-001 | Test Supplies', 3)
('ID 4 | INVALID-TEST-001 | Test Supplies', 4)
('ID 5 | XLS-TEST-001 | Spreadsheet Supplies', 5)
('ID 6 | CSV-TEST-001 | CSV Supplies', 6)
('ID 7 | INVALID-TEST-001 | Test Supplies', 7)
('ID 8 | XLS-TEST-001 | Legacy Supplies', 8)
('ID 9 | XLS-TEST-001 | Legacy Supplies', 9)
('ID 10 | CSV-TEST-002 | CSV Supplies Updated', 10)
('ID 11 | OCR TEST-001 | CCR Supplies', 11)
('ID 12 | OCR TEST-001 | CCR Supplies', 12)
('ID 13 | OCR TEST-001 | CCR Supplies', 13)
('ID 14 | OCR TEST-001 | CCR Supplies Verified', 14)
('ID 15 | OCR TEST-001 | CCR Supplies', 15)
Closing server running on port: 7860
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>


Gradio relaunched successfully.


In [ ]:
import inspect

functions_to_check = [
    "extract_pdf_text",
    "extract_pdf_with_ocr",
    "extract_image_text",
    "process_spreadsheet",
    "process_document"
]

for name in functions_to_check:
    func = globals().get(name)

    print(f"\n--- {name} ---")

    if callable(func):
        print(inspect.signature(func))
    else:
        print("Not found")


--- extract_pdf_text ---
(file_path)

--- extract_pdf_with_ocr ---
Not found

--- extract_image_text ---
(file_path)

--- process_spreadsheet ---
Not found

--- process_document ---
(file_path)


In [ ]:
#inpect_process_document
import inspect

print(inspect.getsource(process_document))

def process_document(file_path):
    extension = os.path.splitext(file_path)[1].lower()

    try:
        if extension == ".pdf":
            text = extract_pdf_text(file_path)
            invoice = extract_invoice_data(text)

        elif extension in [".jpg", ".jpeg", ".png", ".tiff", ".tif"]:
            text = extract_image_text(file_path)
            invoice = extract_invoice_data(text)

        elif extension == ".csv":
            text = extract_spreadsheet_text(file_path)
            invoice = extract_csv_invoice(file_path)

        elif extension in [".xlsx", ".xls"]:
            text = extract_spreadsheet_text(file_path)
            invoice = extract_spreadsheet_invoice(file_path)

        else:
            raise ValueError("Unsupported file format")

        validation = validate_invoice(invoice)

        ocr_quality = (
            check_ocr_quality(text, invoice)
            if extension in [".jpg", ".jpeg", ".png", ".tiff", ".tif"]
            else {
                "manu

In [ ]:
import inspect

print(inspect.getsource(extract_pdf_text))

def extract_pdf_text(file_path):
    pdf = fitz.open(file_path)
    extracted_pages = []

    try:
        for page in pdf:
            text = page.get_text().strip()

            if not text:
                pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
                image = Image.frombytes(
                    "RGB",
                    [pix.width, pix.height],
                    pix.samples
                )
                text = pytesseract.image_to_string(image)

            extracted_pages.append(text)

    finally:
        pdf.close()

    return "\n".join(extracted_pages)



In [ ]:
import inspect

print(inspect.getsource(extract_image_text))

def extract_image_text(file_path):
    image = Image.open(file_path)
    text = pytesseract.image_to_string(image)
    return text



In [ ]:
import inspect

print(inspect.getsource(extract_spreadsheet_text))

def extract_spreadsheet_text(file_path):
    extension = os.path.splitext(file_path)[1].lower()

    if extension == ".csv":
        df = pd.read_csv(file_path)
        return df.to_string(index=False)

    elif extension in [".xlsx", ".xls"]:
        sheets = pd.read_excel(file_path, sheet_name=None)

        extracted_text = ""

        for sheet_name, df in sheets.items():
            extracted_text += f"\nSheet: {sheet_name}\n"
            extracted_text += df.to_string(index=False)
            extracted_text += "\n"

        return extracted_text

    else:
        raise ValueError("Unsupported spreadsheet format")



In [ ]:
import inspect

print(inspect.getsource(extract_invoice_data))

def extract_invoice_data(text):
    invoice = {
        "invoice_number": None,
        "invoice_date": None,
        "supplier_name": None,
        "customer_name": None,
        "subtotal": None,
        "tax": None,
        "total": None,
        "currency": None,
        "line_items": []
    }

    patterns = {
        "invoice_number": r"Invoice\s*(?:No|Number|#)\s*[:\-]?\s*([^\r\n]+)",
        "invoice_date": r"Date\s*[:\-]?\s*(\d{2}[/-]\d{2}[/-]\d{4})",
        "subtotal": r"\bSubtota[lt]\b\s*[:\-]?\s*([\d,]+\.?\d*)",
        "tax": r"\b(?:Tax|VAT)\b\s*[:\-]?\s*([\d,]+\.?\d*)",
        "total": r"(?<!Sub)\bTotal\b\s*[:\-]?\s*([\d,]+\.?\d*)"
    }

    for field, pattern in patterns.items():
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            value = match.group(1).strip()

            if field == "invoice_number":
                invoice[field] = value.strip(" .:-")
            else:
                invoice[field] = value

    supplier = re.sear

In [ ]:
import inspect

print(inspect.getsource(extract_csv_invoice))
print("\n--- EXCEL INVOICE EXTRACTION ---\n")
print(inspect.getsource(extract_spreadsheet_invoice))

def extract_csv_invoice(file_path):
    df = pd.read_csv(file_path, header=None)

    invoice = {
        "invoice_number": None,
        "invoice_date": None,
        "supplier_name": None,
        "customer_name": None,
        "subtotal": None,
        "tax": None,
        "total": None,
        "currency": None,
        "line_items": []
    }

    field_mapping = {
        "Invoice Number": "invoice_number",
        "Date": "invoice_date",
        "Supplier": "supplier_name",
        "Customer": "customer_name",
        "Subtotal": "subtotal",
        "Tax": "tax",
        "Total": "total",
        "Currency": "currency"
    }

    # Extract invoice summary fields
    for _, row in df.iterrows():
        field = str(row[0]).strip()
        value = row[1]

        if field in field_mapping and pd.notna(value):
            invoice[field_mapping[field]] = str(value)

    # Extract line items
    current_item = {}

    for _, row in df.iterrows():
        field = str(row[0]).strip()
  

In [ ]:
import inspect

print(inspect.getsource(extract_csv_invoice))

def extract_csv_invoice(file_path):
    df = pd.read_csv(file_path, header=None)

    invoice = {
        "invoice_number": None,
        "invoice_date": None,
        "supplier_name": None,
        "customer_name": None,
        "subtotal": None,
        "tax": None,
        "total": None,
        "currency": None,
        "line_items": []
    }

    field_mapping = {
        "Invoice Number": "invoice_number",
        "Date": "invoice_date",
        "Supplier": "supplier_name",
        "Customer": "customer_name",
        "Subtotal": "subtotal",
        "Tax": "tax",
        "Total": "total",
        "Currency": "currency"
    }

    # Extract invoice summary fields
    for _, row in df.iterrows():
        field = str(row[0]).strip()
        value = row[1]

        if field in field_mapping and pd.notna(value):
            invoice[field_mapping[field]] = str(value)

    # Extract line items
    current_item = {}

    for _, row in df.iterrows():
        field = str(row[0]).strip()
  

In [ ]:
import inspect

print(inspect.getsource(validate_invoice))

def validate_invoice(invoice):
    errors = []

    required_fields = [
        "invoice_number",
        "invoice_date",
        "supplier_name",
        "customer_name",
        "subtotal",
        "tax",
        "total"
    ]

    for field in required_fields:
        if not invoice.get(field):
            errors.append(f"Missing required field: {field}")

    try:
        subtotal = float(str(invoice["subtotal"]).replace(",", ""))
        tax = float(str(invoice["tax"]).replace(",", ""))
        total = float(str(invoice["total"]).replace(",", ""))

        if abs((subtotal + tax) - total) > 0.01:
            errors.append("Subtotal + tax does not equal total")

        line_items = invoice.get("line_items", [])

        if line_items:
            line_item_total = sum(
                float(item["amount"]) for item in line_items
            )

            if abs(line_item_total - subtotal) > 0.01:
                errors.append(
                    "Sum of line item amounts does no

In [ ]:
import inspect

for name in [
    "check_duplicate_invoice",
    "is_duplicate_invoice",
    "revalidate_invoice_record"
]:
    func = globals().get(name)

    print(f"\n--- {name} ---")

    if callable(func):
        print(inspect.getsource(func))
    else:
        print("Not found")


--- check_duplicate_invoice ---
def check_duplicate_invoice(invoice_number, supplier_name):
    cursor = conn.cursor()

    cursor.execute("""
    SELECT id, document_id, invoice_number, supplier_name
    FROM invoices
    WHERE LOWER(invoice_number) = LOWER(?)
    AND LOWER(supplier_name) = LOWER(?)
    """, (invoice_number, supplier_name))

    matches = cursor.fetchall()

    return {
        "is_duplicate": len(matches) > 0,
        "matching_records": matches
    }


--- is_duplicate_invoice ---
Not found

--- revalidate_invoice_record ---
def revalidate_invoice_record(invoice_id):
    cursor = conn.cursor()

    cursor.execute("""
        SELECT document_id, invoice_number, invoice_date,
               supplier_name, customer_name, subtotal, tax,
               total, currency
        FROM invoices
        WHERE id = ?
    """, (invoice_id,))

    record = cursor.fetchone()

    if not record:
        return {"status": "failed", "error": "Invoice not found"}

    document_id = r

In [ ]:
import inspect

print(inspect.getsource(is_duplicate_before_save))

def is_duplicate_before_save(invoice, exclude_document_id=None):
    invoice_number = invoice.get("invoice_number")
    supplier_name = invoice.get("supplier_name")

    if not invoice_number or not supplier_name:
        return False

    cursor = conn.cursor()

    query = """
        SELECT COUNT(*)
        FROM invoices
        WHERE LOWER(invoice_number) = LOWER(?)
        AND LOWER(supplier_name) = LOWER(?)
    """

    params = [invoice_number, supplier_name]

    if exclude_document_id is not None:
        query += " AND document_id != ?"
        params.append(exclude_document_id)

    cursor.execute(query, params)

    return cursor.fetchone()[0] > 0



In [ ]:
print("EXISTING API ENDPOINTS\n")

for route in app.routes:
    if hasattr(route, "methods"):
        print(f"{', '.join(route.methods)}  {route.path}")

EXISTING API ENDPOINTS

GET, HEAD  /openapi.json
GET, HEAD  /docs
GET, HEAD  /docs/oauth2-redirect
GET, HEAD  /redoc
GET  /
GET  /documents
GET  /invoices
GET  /documents/{document_id}/status
POST  /retry/{document_id}
GET  /search
POST  /upload
PUT  /invoices/{invoice_id}/correct


In [ ]:
import inspect

print(inspect.getsource(retry_document))

@app.post("/retry/{document_id}")
def retry_document(document_id: int):

    cursor = conn.cursor()

    cursor.execute(
        "SELECT filename FROM documents WHERE id = ?",
        (document_id,)
    )

    record = cursor.fetchone()

    if not record:
        raise HTTPException(
            status_code=404,
            detail="Document not found"
        )

    file_path = os.path.join(
        "document_intelligence/uploads",
        record[0]
    )

    if not os.path.exists(file_path):
        raise HTTPException(
            status_code=404,
            detail="Uploaded file not found"
        )

    result = process_document(file_path)

    if result.get("status") == "failed":
        cursor.execute("""
            UPDATE documents
            SET processing_status = 'failed'
            WHERE id = ?
        """, (document_id,))
        conn.commit()

        return {
            "document_id": document_id,
            "status": "failed",
            "error": result.get("err

In [ ]:
import inspect

print(inspect.getsource(update_existing_document))

def update_existing_document(document_id, file_path, result):
    cursor = conn.cursor()

    invoice = result.get("invoice_data")

    if not invoice:
        raise ValueError("Invoice data missing from processing result")

    # Preserve all validation errors from the original processing
    validation = result.get("validation", validate_invoice(invoice))
    validation_errors = list(validation.get("errors", []))

    # Check for duplicates
    duplicate = is_duplicate_before_save(
        invoice,
        exclude_document_id=document_id
    )

    if duplicate and "Duplicate invoice detected" not in validation_errors:
        validation_errors.append("Duplicate invoice detected")

    status = "needs_review" if validation_errors else "success"

    cursor.execute("""
        UPDATE documents
        SET processing_status = ?
        WHERE id = ?
    """, (status, document_id))

    cursor.execute("""
        UPDATE invoices
        SET invoice_number = ?, invoice_date = ?,
         

In [ ]:
def update_existing_document(document_id, file_path, result):
    cursor = conn.cursor()

    invoice = result.get("invoice_data")

    if not invoice:
        raise ValueError("Invoice data missing from processing result")

    validation = result.get("validation", validate_invoice(invoice))
    validation_errors = list(validation.get("errors", []))

    duplicate = is_duplicate_before_save(
        invoice,
        exclude_document_id=document_id
    )

    if duplicate and "Duplicate invoice detected" not in validation_errors:
        validation_errors.append("Duplicate invoice detected")

    status = "needs_review" if validation_errors else "success"

    cursor.execute("""
        UPDATE documents
        SET processing_status = ?
        WHERE id = ?
    """, (status, document_id))

    cursor.execute("""
        UPDATE invoices
        SET invoice_number = ?, invoice_date = ?,
            supplier_name = ?, customer_name = ?,
            subtotal = ?, tax = ?, total = ?, currency = ?
        WHERE document_id = ?
    """, (
        invoice.get("invoice_number"),
        invoice.get("invoice_date"),
        invoice.get("supplier_name"),
        invoice.get("customer_name"),
        invoice.get("subtotal"),
        invoice.get("tax"),
        invoice.get("total"),
        invoice.get("currency"),
        document_id
    ))

    cursor.execute(
        "SELECT id FROM invoices WHERE document_id = ?",
        (document_id,)
    )
    invoice_record = cursor.fetchone()

    if not invoice_record:
        raise ValueError("Existing invoice record not found")

    invoice_id = invoice_record[0]

    cursor.execute(
        "DELETE FROM line_items WHERE invoice_id = ?",
        (invoice_id,)
    )

    for item in invoice.get("line_items", []):
        cursor.execute("""
            INSERT INTO line_items
            (invoice_id, description, quantity, unit_price, amount)
            VALUES (?, ?, ?, ?, ?)
        """, (
            invoice_id,
            item.get("description"),
            item.get("quantity"),
            item.get("unit_price"),
            item.get("amount")
        ))

    cursor.execute(
        "DELETE FROM validation_errors WHERE document_id = ?",
        (document_id,)
    )

    for error in validation_errors:
        cursor.execute("""
            INSERT INTO validation_errors (document_id, error_message)
            VALUES (?, ?)
        """, (document_id, error))

    if result.get("extracted_text"):
        cursor.execute("""
            UPDATE extracted_text
            SET extracted_content = ?
            WHERE document_id = ?
        """, (result["extracted_text"], document_id))

    conn.commit()

    return {
        "document_id": document_id,
        "status": status,
        "duplicate_detected": duplicate,
        "validation_errors": validation_errors
    }

In [ ]:
cursor = conn.cursor()

cursor.execute("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name
""")

print("DATABASE TABLES:")
for row in cursor.fetchall():
    print("-", row[0])

DATABASE TABLES:
- correction_history
- documents
- extracted_text
- invoices
- line_items
- sqlite_sequence
- validation_errors


In [ ]:
import inspect

print(inspect.getsource(search_documents))

        def search_documents(keyword):
            if not keyword.strip():
                return "Enter a search keyword."

            response = requests.get(
                f"{API_BASE}/search",
                params={"keyword": keyword}
            )
            return json.dumps(response.json(), indent=4)



In [ ]:
import inspect

for route in app.routes:
    if getattr(route, "path", "") == "/search":
        print(inspect.getsource(route.endpoint))

@app.get("/search")
def search_documents(keyword: str):
    cursor = conn.cursor()

    search_term = f"%{keyword}%"

    cursor.execute("""
        SELECT DISTINCT
            invoices.id,
            invoices.document_id,
            invoices.invoice_number,
            invoices.invoice_date,
            invoices.supplier_name,
            invoices.customer_name,
            invoices.subtotal,
            invoices.tax,
            invoices.total,
            invoices.currency
        FROM invoices
        LEFT JOIN extracted_text
            ON invoices.document_id = extracted_text.document_id
        WHERE invoices.invoice_number LIKE ?
           OR invoices.supplier_name LIKE ?
           OR invoices.customer_name LIKE ?
           OR extracted_text.extracted_content LIKE ?
    """, (search_term, search_term, search_term, search_term))

    return cursor.fetchall()



In [ ]:
cursor = conn.cursor()

cursor.execute("""
PRAGMA table_info(correction_history)
""")

print("CORRECTION HISTORY STRUCTURE:")
for row in cursor.fetchall():
    print(row)

CORRECTION HISTORY STRUCTURE:
(0, 'id', 'INTEGER', 0, None, 1)
(1, 'document_id', 'INTEGER', 0, None, 0)
(2, 'field_name', 'TEXT', 0, None, 0)
(3, 'old_value', 'TEXT', 0, None, 0)
(4, 'new_value', 'TEXT', 0, None, 0)
(5, 'corrected_at', 'TIMESTAMP', 0, 'CURRENT_TIMESTAMP', 0)


In [ ]:
import inspect

print(inspect.getsource(correct_invoice))

def correct_invoice(invoice_id, field_name, new_value):
    allowed_fields = [
        "invoice_number", "invoice_date", "supplier_name",
        "customer_name", "subtotal", "tax", "total", "currency"
    ]

    if field_name not in allowed_fields:
        return {"status": "failed", "error": "Invalid field name"}

    if new_value is None or not str(new_value).strip():
        return {"status": "failed", "error": "Corrected value cannot be empty"}

    cursor = conn.cursor()

    cursor.execute(
        "SELECT document_id, " + field_name +
        " FROM invoices WHERE id = ?",
        (invoice_id,)
    )

    record = cursor.fetchone()

    if not record:
        return {"status": "failed", "error": "Invoice not found"}

    document_id, old_value = record

    if field_name in ["subtotal", "tax", "total"]:
        try:
            new_value = float(str(new_value).replace(",", ""))
        except ValueError:
            return {"status": "failed", "error": "Enter a valid number"}



In [ ]:
#existing_automated_test

import os

print("TEST FILES:")

for root, dirs, files in os.walk("."):
    dirs[:] = [
        d for d in dirs
        if d not in ["content", ".git", "__pycache__", "site-packages"]
    ]

    for file in files:
        if file.startswith("test_") or file.endswith("_test.py"):
            print(os.path.join(root, file))

TEST FILES:
./document_intelligence/uploads/test_file.txt


In [ ]:
import pytest

print("Pytest version:", pytest.__version__)

Pytest version: 8.4.2


In [ ]:
def test_extract_invoice_data():
    sample = """
    Invoice Number: TEST-001
    Date: 05/11/2026
    Supplier: ABC Supplies
    Customer: Demo Buyer
    Subtotal: 500
    Tax: 75
    Total: 575
    Currency: BDT
    """

    result = extract_invoice_data(sample)

    assert result["invoice_number"] == "TEST-001"
    assert result["supplier_name"] == "ABC Supplies"
    assert result["customer_name"] == "Demo Buyer"
    assert result["subtotal"] == "500"
    assert result["tax"] == "75"
    assert result["total"] == "575"
    assert result["currency"] == "BDT"


def test_validate_invoice_success():
    invoice = {
        "invoice_number": "TEST-001",
        "invoice_date": "05/11/2026",
        "supplier_name": "ABC Supplies",
        "customer_name": "Demo Buyer",
        "subtotal": "500",
        "tax": "75",
        "total": "575",
        "currency": "BDT",
        "line_items": []
    }

    result = validate_invoice(invoice)

    assert result["validation_status"] == "passed"
    assert result["errors"] == []


def test_validate_invoice_total_mismatch():
    invoice = {
        "invoice_number": "TEST-002",
        "invoice_date": "05/11/2026",
        "supplier_name": "ABC Supplies",
        "customer_name": "Demo Buyer",
        "subtotal": "500",
        "tax": "75",
        "total": "600",
        "currency": "BDT",
        "line_items": []
    }

    result = validate_invoice(invoice)

    assert result["validation_status"] == "failed"
    assert "Subtotal + tax does not equal total" in result["errors"]


results = pytest.main(["-v", "--tb=short", "-q", "--disable-warnings"])

print("\nAutomated test execution finished.")

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0
rootdir: /content/drive/MyDrive/AI_Document_Intelligence_System
plugins: langsmith-0.14.15, platformdirs-4.14.0, anyio-4.15.1, typeguard-4.6.0
collected 0 items

============================== 1 warning in 0.14s ==============================

Automated test execution finished.


In [ ]:
test_functions = [
    test_extract_invoice_data,
    test_validate_invoice_success,
    test_validate_invoice_total_mismatch
]

passed = 0
failed = 0

for test in test_functions:
    try:
        test()
        print(f"PASS: {test.__name__}")
        passed += 1
    except Exception as e:
        print(f"FAIL: {test.__name__} - {e}")
        failed += 1

print("\nTEST SUMMARY")
print("Passed:", passed)
print("Failed:", failed)
print("Total:", passed + failed)

PASS: test_extract_invoice_data
PASS: test_validate_invoice_success
PASS: test_validate_invoice_total_mismatch

TEST SUMMARY
Passed: 3
Failed: 0
Total: 3


In [ ]:
def test_missing_required_fields():
    invoice = {
        "invoice_number": None,
        "invoice_date": None,
        "supplier_name": None,
        "customer_name": None,
        "subtotal": "500",
        "tax": "75",
        "total": "575",
        "currency": "BDT",
        "line_items": []
    }

    result = validate_invoice(invoice)

    assert result["validation_status"] == "failed"
    assert any("Missing required field" in e for e in result["errors"])


def test_line_item_total_mismatch():
    invoice = {
        "invoice_number": "TEST-003",
        "invoice_date": "05/11/2026",
        "supplier_name": "ABC Supplies",
        "customer_name": "Demo Buyer",
        "subtotal": "500",
        "tax": "75",
        "total": "575",
        "currency": "BDT",
        "line_items": [
            {"description": "Item A", "quantity": 2,
             "unit_price": 100, "amount": 200}
        ]
    }

    result = validate_invoice(invoice)

    assert result["validation_status"] == "failed"
    assert any("line item amounts" in e for e in result["errors"])


for test in [test_missing_required_fields, test_line_item_total_mismatch]:
    try:
        test()
        print("PASS:", test.__name__)
    except Exception as e:
        print("FAIL:", test.__name__, "-", e)

PASS: test_missing_required_fields
PASS: test_line_item_total_mismatch


In [ ]:
def test_duplicate_invoice_detection():
    invoice = {
        "invoice_number": "OCR TEST-001",
        "supplier_name": "OCR Supplies"
    }

    result = is_duplicate_before_save(invoice)

    assert result is True


def test_non_duplicate_invoice():
    invoice = {
        "invoice_number": "UNIQUE-TEST-999",
        "supplier_name": "Unique Test Supplier"
    }

    result = is_duplicate_before_save(invoice)

    assert result is False


for test in [
    test_duplicate_invoice_detection,
    test_non_duplicate_invoice
]:
    try:
        test()
        print("PASS:", test.__name__)
    except Exception as e:
        print("FAIL:", test.__name__, "-", e)

FAIL: test_duplicate_invoice_detection - 
PASS: test_non_duplicate_invoice


In [ ]:
cursor = conn.cursor()

cursor.execute("""
SELECT invoice_number, supplier_name, COUNT(*) AS occurrences
FROM invoices
WHERE invoice_number IS NOT NULL
AND supplier_name IS NOT NULL
GROUP BY LOWER(invoice_number), LOWER(supplier_name)
HAVING COUNT(*) > 1
""")

duplicates = cursor.fetchall()

print("Existing duplicate invoices:")
for row in duplicates:
    print(row)

Existing duplicate invoices:
('INVALID-TEST-001', 'Test Supplies', 3)
('OCR TEST-001', 'CCR Supplies', 4)
('XLS-TEST-001', 'Legacy Supplies', 2)


In [ ]:
def test_duplicate_invoice_detection():
    invoice = {
        "invoice_number": "OCR TEST-001",
        "supplier_name": "CCR Supplies"
    }

    result = is_duplicate_before_save(invoice)

    assert result is True


def test_non_duplicate_invoice():
    invoice = {
        "invoice_number": "UNIQUE-TEST-999",
        "supplier_name": "Unique Test Supplier"
    }

    result = is_duplicate_before_save(invoice)

    assert result is False


for test in [
    test_duplicate_invoice_detection,
    test_non_duplicate_invoice
]:
    try:
        test()
        print("PASS:", test.__name__)
    except Exception as e:
        print("FAIL:", test.__name__, "-", repr(e))

PASS: test_duplicate_invoice_detection
PASS: test_non_duplicate_invoice


In [ ]:
import inspect

print(inspect.signature(retry_document))

(document_id: int)


In [ ]:
cursor = conn.cursor()

cursor.execute("""
SELECT id, filename, processing_status
FROM documents
ORDER BY id DESC
LIMIT 5
""")

for row in cursor.fetchall():
    print(row)

(15, '5937e267-0de1-4451-ac13-a712ec67a490.png', 'needs_review')
(14, 'f7c21676-cc30-4d83-b4a1-a355b46e44a3.png', 'success')
(13, 'd02c4e38-24d2-4cb0-82be-7fccea922d5c.png', 'needs_review')
(12, 'ec118d9c-eb50-4051-95a3-daf64e1d4d63.png', 'needs_review')
(11, 'e6170b30-c7dd-432b-9a03-eb0c2566e992.png', 'needs_review')


In [ ]:
result = retry_document(14)

print("Retry result:")
print(result)

cursor = conn.cursor()
cursor.execute("""
SELECT id, processing_status
FROM documents
WHERE id = 14
""")
print("Updated status:", cursor.fetchone())

Retry result:
{'document_id': 14, 'status': 'needs_review', 'duplicate_detected': True, 'validation_errors': ['Subtotal + tax does not equal total', 'Invoice amount mismatch: manual review required', 'Currency not confidently detected', 'Duplicate invoice detected']}
Updated status: (14, 'needs_review')


In [ ]:
cursor = conn.cursor()

cursor.execute("PRAGMA table_info(validation_errors)")
print(cursor.fetchall())

[(0, 'id', 'INTEGER', 0, None, 1), (1, 'document_id', 'INTEGER', 0, None, 0), (2, 'error_message', 'TEXT', 0, None, 0)]


In [ ]:
cursor = conn.cursor()

cursor.execute("""
SELECT document_id, error_message
FROM validation_errors
WHERE document_id = 14
""")

for row in cursor.fetchall():
    print(row)

(14, 'Subtotal + tax does not equal total')
(14, 'Invoice amount mismatch: manual review required')
(14, 'Currency not confidently detected')
(14, 'Duplicate invoice detected')


In [ ]:
cursor = conn.cursor()

cursor.execute("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name
""")

print("Database tables:")
for row in cursor.fetchall():
    print(row[0])

Database tables:
correction_history
documents
extracted_text
invoices
line_items
sqlite_sequence
validation_errors


In [ ]:
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS processing_history (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    document_id INTEGER NOT NULL,
    processing_status TEXT,
    details TEXT,
    processed_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

conn.commit()

print("Processing history table created successfully.")

cursor.execute("PRAGMA table_info(processing_history)")
print(cursor.fetchall())

Processing history table created successfully.
[(0, 'id', 'INTEGER', 0, None, 1), (1, 'document_id', 'INTEGER', 1, None, 0), (2, 'processing_status', 'TEXT', 0, None, 0), (3, 'details', 'TEXT', 0, None, 0), (4, 'processed_at', 'TIMESTAMP', 0, 'CURRENT_TIMESTAMP', 0)]


In [ ]:
import inspect

print(inspect.getsource(retry_document))

@app.post("/retry/{document_id}")
def retry_document(document_id: int):

    cursor = conn.cursor()

    cursor.execute(
        "SELECT filename FROM documents WHERE id = ?",
        (document_id,)
    )

    record = cursor.fetchone()

    if not record:
        raise HTTPException(
            status_code=404,
            detail="Document not found"
        )

    file_path = os.path.join(
        "document_intelligence/uploads",
        record[0]
    )

    if not os.path.exists(file_path):
        raise HTTPException(
            status_code=404,
            detail="Uploaded file not found"
        )

    result = process_document(file_path)

    if result.get("status") == "failed":
        cursor.execute("""
            UPDATE documents
            SET processing_status = 'failed'
            WHERE id = ?
        """, (document_id,))
        conn.commit()

        return {
            "document_id": document_id,
            "status": "failed",
            "error": result.get("err

In [ ]:
@app.post("/retry/{document_id}")
def retry_document(document_id: int):

    cursor = conn.cursor()

    cursor.execute(
        "SELECT filename FROM documents WHERE id = ?",
        (document_id,)
    )

    record = cursor.fetchone()

    if not record:
        raise HTTPException(
            status_code=404,
            detail="Document not found"
        )

    file_path = os.path.join(
        "document_intelligence/uploads",
        record[0]
    )

    if not os.path.exists(file_path):
        raise HTTPException(
            status_code=404,
            detail="Uploaded file not found"
        )

    result = process_document(file_path)

    if result.get("status") == "failed":

        cursor.execute("""
            UPDATE documents
            SET processing_status = 'failed'
            WHERE id = ?
        """, (document_id,))

        cursor.execute("""
            INSERT INTO processing_history
            (document_id, processing_status, details)
            VALUES (?, ?, ?)
        """, (
            document_id,
            "failed",
            str(result.get("error"))
        ))

        conn.commit()

        return {
            "document_id": document_id,
            "status": "failed",
            "error": result.get("error")
        }

    update_result = update_existing_document(
        document_id,
        file_path,
        result
    )

    cursor.execute("""
        INSERT INTO processing_history
        (document_id, processing_status, details)
        VALUES (?, ?, ?)
    """, (
        document_id,
        update_result.get("status", "processed"),
        str(update_result.get("validation_errors", []))
    ))

    conn.commit()

    return update_result

In [ ]:
result = retry_document(14)

print("Retry result:", result)

cursor = conn.cursor()
cursor.execute("""
SELECT document_id, processing_status, details, processed_at
FROM processing_history
WHERE document_id = 14
ORDER BY id DESC
LIMIT 3
""")

print("\nProcessing history:")
for row in cursor.fetchall():
    print(row)

Retry result: {'document_id': 14, 'status': 'needs_review', 'duplicate_detected': True, 'validation_errors': ['Subtotal + tax does not equal total', 'Invoice amount mismatch: manual review required', 'Currency not confidently detected', 'Duplicate invoice detected']}

Processing history:
(14, 'needs_review', "['Subtotal + tax does not equal total', 'Invoice amount mismatch: manual review required', 'Currency not confidently detected', 'Duplicate invoice detected']", '2026-11-05 15:06:50')


In [ ]:
cursor = conn.cursor()

cursor.execute("""
SELECT document_id, field_name, old_value, new_value, corrected_at
FROM correction_history
ORDER BY id DESC
LIMIT 5
""")

for row in cursor.fetchall():
    print(row)

(15, 'total', '675.0', '575.0', '2026-11-05 13:05:45')
(14, 'total', '675.0', '575.0', '2026-11-05 13:01:28')
(14, 'supplier_name', 'CCR Supplies', 'CCR Supplies Verified', '2026-11-05 13:00:38')
(10, 'supplier_name', 'CSV Supplies', 'CSV Supplies Updated', '2026-11-05 12:45:50')
(12, 'total', '675.0', '575', '2026-11-05 09:58:33')


In [ ]:
import inspect

print(inspect.getsource(upload_document))

@app.post("/upload")
async def upload_document(
    background_tasks: BackgroundTasks,
    file: UploadFile = File(...)
):
    allowed = [
        ".pdf", ".jpg", ".jpeg", ".png",
        ".tiff", ".tif", ".xlsx", ".xls", ".csv"
    ]

    extension = os.path.splitext(file.filename)[1].lower()

    if extension not in allowed:
        raise HTTPException(
            status_code=400,
            detail="Unsupported file format"
        )

    unique_name = str(uuid.uuid4()) + extension

    file_path = os.path.join(
        "document_intelligence/uploads",
        unique_name
    )

    with open(file_path, "wb") as f:
        f.write(await file.read())

    cursor = conn.cursor()

    cursor.execute("""
        INSERT INTO documents (
            filename, file_type, processing_status
        )
        VALUES (?, ?, ?)
    """, (unique_name, extension, "pending"))

    document_id = cursor.lastrowid
    conn.commit()

    background_tasks.add_task(
        process_upload_background,
 

In [ ]:
import inspect

print(inspect.getsource(process_upload_background))

async def process_upload_background(document_id, file_path):
    cursor = conn.cursor()

    try:
        result = process_document(file_path)

        if result.get("status") == "failed":
            cursor.execute("""
                UPDATE documents
                SET processing_status = 'failed'
                WHERE id = ?
            """, (document_id,))
            conn.commit()
            return

        invoice = result.get("invoice_data")
        validation_errors = list(
            result.get("validation", {}).get("errors", [])
        )

        duplicate = False

        if invoice:
            duplicate = is_duplicate_before_save(invoice)

            if duplicate:
                validation_errors.append("Duplicate invoice detected")

            def safe_float(value):
                try:
                    return float(str(value).replace(",", "")) if value is not None else None
                except (ValueError, TypeError):
                    return None

       

In [ ]:

async def process_upload_background(document_id, file_path):
    cursor = conn.cursor()

    try:
        result = process_document(file_path)

        if result.get("status") == "failed":
            error = str(result.get("error", "Processing failed"))

            cursor.execute("""
                UPDATE documents
                SET processing_status = 'failed'
                WHERE id = ?
            """, (document_id,))

            cursor.execute("""
                INSERT INTO processing_history
                (document_id, processing_status, details)
                VALUES (?, ?, ?)
            """, (document_id, "failed", error))

            conn.commit()
            return

        invoice = result.get("invoice_data")
        validation_errors = list(
            result.get("validation", {}).get("errors", [])
        )

        duplicate = False

        if invoice:
            duplicate = is_duplicate_before_save(invoice)

            if duplicate:
                validation_errors.append(
                    "Duplicate invoice detected"
                )

            def safe_float(value):
                try:
                    return (
                        float(str(value).replace(",", ""))
                        if value is not None else None
                    )
                except (ValueError, TypeError):
                    return None

            cursor.execute("""
                INSERT INTO invoices (
                    document_id, invoice_number, invoice_date,
                    supplier_name, customer_name, subtotal,
                    tax, total, currency
                )
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                document_id,
                invoice.get("invoice_number"),
                invoice.get("invoice_date"),
                invoice.get("supplier_name"),
                invoice.get("customer_name"),
                safe_float(invoice.get("subtotal")),
                safe_float(invoice.get("tax")),
                safe_float(invoice.get("total")),
                invoice.get("currency")
            ))

            invoice_id = cursor.lastrowid

            for item in invoice.get("line_items", []):
                cursor.execute("""
                    INSERT INTO line_items (
                        invoice_id, description, quantity,
                        unit_price, amount
                    )
                    VALUES (?, ?, ?, ?, ?)
                """, (
                    invoice_id,
                    item.get("description"),
                    safe_float(item.get("quantity")),
                    safe_float(item.get("unit_price")),
                    safe_float(item.get("amount"))
                ))

        extracted_text = result.get("extracted_text", "")

        if extracted_text:
            save_extracted_text(document_id, extracted_text)

        for error in validation_errors:
            cursor.execute("""
                INSERT INTO validation_errors
                (document_id, error_message)
                VALUES (?, ?)
            """, (document_id, error))

        status = "needs_review" if validation_errors else "success"

        details = (
            str(validation_errors)
            if validation_errors
            else "Document processed successfully"
        )

        cursor.execute("""
            UPDATE documents
            SET processing_status = ?
            WHERE id = ?
        """, (status, document_id))

        cursor.execute("""
            INSERT INTO processing_history
            (document_id, processing_status, details)
            VALUES (?, ?, ?)
        """, (document_id, status, details))

        conn.commit()

    except Exception as e:
        conn.rollback()

        error = str(e)

        cursor.execute("""
            UPDATE documents
            SET processing_status = 'failed'
            WHERE id = ?
        """, (document_id,))

        cursor.execute("""
            INSERT INTO processing_history
            (document_id, processing_status, details)
            VALUES (?, ?, ?)
        """, (document_id, "failed", error))

        conn.commit()

        print("Background processing error:", error)


In [ ]:

@app.post("/upload")
async def upload_document(
    background_tasks: BackgroundTasks,
    file: UploadFile = File(...)
):
    allowed = [
        ".pdf", ".jpg", ".jpeg", ".png",
        ".tiff", ".tif", ".xlsx", ".xls", ".csv"
    ]

    extension = os.path.splitext(file.filename)[1].lower()

    if extension not in allowed:
        raise HTTPException(
            status_code=400,
            detail="Unsupported file format"
        )

    unique_name = str(uuid.uuid4()) + extension

    file_path = os.path.join(
        "document_intelligence/uploads",
        unique_name
    )

    with open(file_path, "wb") as f:
        f.write(await file.read())

    cursor = conn.cursor()

    cursor.execute("""
        INSERT INTO documents (
            filename, file_type, processing_status
        )
        VALUES (?, ?, ?)
    """, (unique_name, extension, "pending"))

    document_id = cursor.lastrowid

    cursor.execute("""
        INSERT INTO processing_history (
            document_id, processing_status, details
        )
        VALUES (?, ?, ?)
    """, (
        document_id,
        "pending",
        "Document uploaded; background processing started"
    ))

    conn.commit()

    background_tasks.add_task(
        process_upload_background,
        document_id,
        file_path
    )

    return {
        "document_id": document_id,
        "filename": file.filename,
        "status": "pending",
        "message": "Document uploaded. Processing started."
    }


In [ ]:
from PIL import Image, ImageDraw

img = Image.new("RGB", (1000, 700), "white")
draw = ImageDraw.Draw(img)

invoice_text = [
    "INVOICE",
    "",
    "Invoice Number: HISTORY-TEST-001",
    "Date: 05/11/2026",
    "Supplier: History Test Supplies",
    "Customer: Demo Buyer",
    "",
    "Subtotal: 500",
    "Tax: 75",
    "Total: 575",
    "Currency: BDT"
]

y = 50
for line in invoice_text:
    draw.text((60, y), line, fill="black", font_size=30)
    y += 55

img.save("history_test_invoice.png")

print("Test invoice image created successfully.")

Test invoice image created successfully.


In [ ]:
from google.colab import files

files.download("history_test_invoice.png")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import gradio as gr

print("demo exists:", "demo" in globals())
print("iface exists:", "iface" in globals())

demo exists: True
iface exists: False


In [ ]:
print("Demo type:", type(demo))
print("Launch method:", demo.launch)

Demo type: <class 'gradio.blocks.Blocks'>
Launch method: <bound method Blocks.launch of Gradio Blocks instance: 3 backend functions
-------------------------------------------
fn_index=0
 inputs:
 |-<gradio.components.file.File object at 0x7fad8d6dd590>
 outputs:
 |-<gradio.components.textbox.Textbox object at 0x7fad8d6dd810>
fn_index=1
 inputs:
 |-<gradio.components.textbox.Textbox object at 0x7fad8d6ddbd0>
 outputs:
 |-<gradio.components.textbox.Textbox object at 0x7fad8d6dde50>
fn_index=2
 inputs:
 |-<gradio.components.textbox.Textbox object at 0x7fad8d6de210>
 outputs:
 |-<gradio.components.textbox.Textbox object at 0x7fad8d6de490>>


In [ ]:
import gradio as gr

print("Gradio version:", gr.__version__)
print("Demo created:", demo is not None)
print("Components:", len(demo.blocks))
print("Current server:", demo.server_port)

Gradio version: 6.31.0
Demo created: True
Components: 20
Current server: 7860


In [ ]:
import socket

for port in [7860, 7861, 7862]:
    sock = socket.socket()
    result = sock.connect_ex(("127.0.0.1", port))
    print(f"Port {port}:", "In use" if result == 0 else "Available")
    sock.close()

Port 7860: In use
Port 7861: Available
Port 7862: Available


In [ ]:
demo.launch(
    share=True,
    debug=True,
    prevent_thread_lock=True,
    server_name="0.0.0.0",
    server_port=7862,
    show_error=True
)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ec3c33f4f0e71a1b89.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


KeyboardInterrupt: 

In [ ]:
print("Local URL:", demo.local_url)
print("Share URL:", demo.share_url)
print("Server running:", demo.is_running)

Local URL: http://127.0.0.1:7860/
Share URL: https://ec3c33f4f0e71a1b89.gradio.live
Server running: True


In [ ]:
import requests

for url in [demo.local_url, demo.share_url]:
    try:
        response = requests.get(url, timeout=15)
        print(url)
        print("HTTP Status:", response.status_code)
        print("Response:", response.text[:200])
    except Exception as e:
        print(url)
        print("Error:", str(e))

http://127.0.0.1:7860/
HTTP Status: 200
Response: <!doctype html>
<html
	lang="en"
	style="
		margin: 0;
		padding: 0;
		min-height: 100%;
		display: flex;
		flex-direction: column;
	"
>
	<head>
		<meta charset="utf-8" />
		<base href="./" />
		<meta
https://ec3c33f4f0e71a1b89.gradio.live
HTTP Status: 404
Response: 
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>No interface is running</title>
</head>
<style>
    body{
    font-family: 'Lato', sans-serif;
    color: #888;
    text-


In [ ]:
demo.close()

print("Gradio server closed.")

Closing server running on port: 7860
Gradio server closed.


In [ ]:
import requests

response = requests.get("http://127.0.0.1:8000/docs")

print("Backend status:", response.status_code)
print("Backend accessible:", response.status_code == 200)

Backend status: 200
Backend accessible: True


In [1]:
import requests
import os

print("App exists:", "app" in globals())
print("Demo exists:", "demo" in globals())
print("Database connection exists:", "conn" in globals())
print("Processing function exists:", "process_document" in globals())

try:
    response = requests.get(
        "http://127.0.0.1:8000/docs",
        timeout=5
    )
    print("Backend status:", response.status_code)
except Exception as e:
    print("Backend:", str(e))

print("Project folder exists:",
      os.path.exists("document_intelligence"))

App exists: False
Demo exists: False
Database connection exists: False
Processing function exists: False
Backend: HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: /docs (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7dabd9c86900>: Failed to establish a new connection: [Errno 111] Connection refused'))
Project folder exists: False


In [2]:
import os

print("Content folder:")
print(os.listdir("/content"))

print("\nGoogle Drive mounted:",
      os.path.exists("/content/drive/MyDrive"))

Content folder:
['.config', 'sample_data']

Google Drive mounted: False


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os

print("Drive mounted:",
      os.path.exists("/content/drive/MyDrive"))

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    dirs[:] = [d for d in dirs if d != ".ipynb_checkpoints"]

    for file in files:
        if file.endswith((".db", ".sqlite", ".sqlite3")):
            print("Database found:", os.path.join(root, file))

Drive mounted: True
Database found: /content/drive/MyDrive/AI_Document_Intelligence_System/document_intelligence/documents.db


In [5]:
import os

project_dir = "/content/drive/MyDrive/AI_Document_Intelligence_System"

for root, dirs, files in os.walk(project_dir):
    level = root.replace(project_dir, "").count(os.sep)
    indent = "  " * level

    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        print(f"{indent}  {file}")

AI_Document_Intelligence_System/
  sample_invoice_v2.pdf
  sample_invalid_invoice.pdf
  sample_spreadsheet_invoice.xlsx
  sample_csv_invoice.csv
  sample_xls_invoice.xls
  sample_csv_invoice_v2.csv
  history_test_invoice.png
  document_intelligence/
    documents.db
    uploads/
      8263d77b-8b1d-4c32-a38a-4a4e35b3d7bd.pdf
      22fb4b83-054c-40f0-bd80-13a4b16698a0.pdf
      45117bca-f5b5-4ded-a951-4ab2e99b9861.pdf
      faabfae7-dc3e-40af-aece-497efabee163.pdf
      82798143-06d9-4080-b862-80571eddad58.xls
      1b506a3e-53e7-49c3-a770-ce43a86fa2e0.xls
      a62b6354-1277-4919-aea7-6b37356e07a3.csv
      corrupted_test.pdf
      test_file.txt
      multi_page_test.pdf
      ocr_test.png
      e6170b30-c7dd-432b-9a03-eb0c2566e992.png
      ec118d9c-eb50-4051-95a3-daf64e1d4d63.png
      d02c4e38-24d2-4cb0-82be-7fccea922d5c.png
      f7c21676-cc30-4d83-b4a1-a355b46e44a3.png
      5937e267-0de1-4451-ac13-a712ec67a490.png
    extracted_text/
    results/
    logs/
      pdf_evaluation_re

In [6]:
import os
import sqlite3

PROJECT_DIR = "/content/drive/MyDrive/AI_Document_Intelligence_System"

os.chdir(PROJECT_DIR)

DB_PATH = os.path.join(
    PROJECT_DIR,
    "document_intelligence",
    "documents.db"
)

UPLOAD_DIR = os.path.join(
    PROJECT_DIR,
    "document_intelligence",
    "uploads"
)

conn = sqlite3.connect(DB_PATH, check_same_thread=False)

print("Working directory:", os.getcwd())
print("Database connected:", conn is not None)
print("Database exists:", os.path.exists(DB_PATH))
print("Upload folder exists:", os.path.exists(UPLOAD_DIR))

Working directory: /content/drive/MyDrive/AI_Document_Intelligence_System
Database connected: True
Database exists: True
Upload folder exists: True


In [7]:
cursor = conn.cursor()

cursor.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    ORDER BY name
""")

print("DATABASE TABLES:")
for row in cursor.fetchall():
    print(row[0])

print("\nDOCUMENT RECORDS:")
cursor.execute("SELECT COUNT(*) FROM documents")
print("Total documents:", cursor.fetchone()[0])

print("\nINVOICE RECORDS:")
cursor.execute("SELECT COUNT(*) FROM invoices")
print("Total invoices:", cursor.fetchone()[0])

print("\nPROCESSING HISTORY:")
cursor.execute("SELECT COUNT(*) FROM processing_history")
print("Total history entries:", cursor.fetchone()[0])

DATABASE TABLES:
correction_history
documents
extracted_text
invoices
line_items
processing_history
sqlite_sequence
validation_errors

DOCUMENT RECORDS:
Total documents: 15

INVOICE RECORDS:
Total invoices: 15

PROCESSING HISTORY:
Total history entries: 1


In [8]:
!pip install -q fastapi uvicorn gradio pandas numpy openpyxl xlrd pymupdf pytesseract pillow python-multipart requests

In [9]:
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [10]:
import os
import re
import uuid
import json
import sqlite3
import shutil
import traceback
import inspect

import pandas as pd
import numpy as np
import fitz
import pytesseract
import gradio as gr
import requests

from PIL import Image, ImageDraw
from fastapi import FastAPI, UploadFile, File, HTTPException, BackgroundTasks
from pydantic import BaseModel
from typing import Optional

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


In [11]:
import pymupdf

print("PyMuPDF version:", pymupdf.__version__)
print("PDF library imported successfully.")

PyMuPDF version: 2.1
PDF library imported successfully.


In [13]:
import pymupdf as fitz

print("PDF extraction library ready:", fitz.__name__)

PDF extraction library ready: pymupdf


In [14]:
print("Project directory:", PROJECT_DIR)
print("Database path:", DB_PATH)
print("Upload directory:", UPLOAD_DIR)

print("Database connected:", conn.execute("SELECT 1").fetchone())
print("Uploaded files:", len(os.listdir(UPLOAD_DIR)))

Project directory: /content/drive/MyDrive/AI_Document_Intelligence_System
Database path: /content/drive/MyDrive/AI_Document_Intelligence_System/document_intelligence/documents.db
Upload directory: /content/drive/MyDrive/AI_Document_Intelligence_System/document_intelligence/uploads
Database connected: (1,)
Uploaded files: 16


In [15]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for file in files:
        if file.endswith(".ipynb") and "AI_Document" in file:
            print(os.path.join(root, file))

In [16]:
print("Database:", DB_PATH)
print("Connection:", conn.execute("SELECT COUNT(*) FROM documents").fetchone())
print("Project directory:", os.getcwd())

Database: /content/drive/MyDrive/AI_Document_Intelligence_System/document_intelligence/documents.db
Connection: (15,)
Project directory: /content/drive/MyDrive/AI_Document_Intelligence_System


In [17]:
# ==========================================
# DOCUMENT EXTRACTION FUNCTIONS
# ==========================================

def extract_pdf_text(file_path):
    text = ""

    try:
        pdf = fitz.open(file_path)

        for page in pdf:
            page_text = page.get_text()

            if not page_text.strip():
                pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
                img = Image.frombytes(
                    "RGB",
                    [pix.width, pix.height],
                    pix.samples
                )
                page_text = pytesseract.image_to_string(img)

            text += page_text + "\n"

        pdf.close()
        return text

    except Exception as e:
        raise Exception(f"PDF extraction failed: {str(e)}")


def extract_image_text(file_path):
    try:
        img = Image.open(file_path)
        return pytesseract.image_to_string(img)

    except Exception as e:
        raise Exception(f"Image OCR failed: {str(e)}")


def extract_spreadsheet_text(file_path):
    try:
        extension = os.path.splitext(file_path)[1].lower()

        if extension == ".csv":
            df = pd.read_csv(file_path)
            return df.to_string(index=False)

        sheets = pd.read_excel(file_path, sheet_name=None)

        text = ""

        for sheet_name, df in sheets.items():
            text += f"\nSheet: {sheet_name}\n"
            text += df.to_string(index=False)

        return text

    except Exception as e:
        raise Exception(f"Spreadsheet extraction failed: {str(e)}")


def extract_invoice_data(text):

    invoice = {
        "invoice_number": None,
        "invoice_date": None,
        "supplier_name": None,
        "customer_name": None,
        "subtotal": None,
        "tax": None,
        "total": None,
        "currency": None,
        "line_items": []
    }

    patterns = {
        "invoice_number": r"Invoice\s*(?:Number|No\.?|#)\s*[:\-]?\s*([A-Za-z0-9\-/]+)",
        "invoice_date": r"(?:Invoice\s*)?Date\s*[:\-]?\s*(\d{1,2}[/-]\d{1,2}[/-]\d{2,4})",
        "supplier_name": r"Supplier\s*[:\-]?\s*(.+)",
        "customer_name": r"Customer\s*[:\-]?\s*(.+)",
        "subtotal": r"Subtotal\s*[:\-]?\s*([\d,]+\.?\d*)",
        "tax": r"Tax\s*[:\-]?\s*([\d,]+\.?\d*)",
        "total": r"Total\s*[:\-]?\s*([\d,]+\.?\d*)"
    }

    for field, pattern in patterns.items():
        match = re.search(pattern, text, re.IGNORECASE)

        if match:
            value = match.group(1).strip()

            if field in ["subtotal", "tax", "total"]:
                try:
                    value = float(value.replace(",", ""))
                except ValueError:
                    value = None

            invoice[field] = value

    currency_match = re.search(
        r"\b(BDT|USD|EUR|GBP)\b",
        text,
        re.IGNORECASE
    )

    if currency_match:
        invoice["currency"] = currency_match.group(1).upper()

    return invoice


def check_ocr_quality(text):
    if not text or len(text.strip()) < 20:
        return ["Low OCR confidence: insufficient extracted text"]

    return []

In [18]:
# ==========================================
# INVOICE VALIDATION AND SPREADSHEET PARSING
# ==========================================

def safe_float(value):
    try:
        if value is None or str(value).strip() == "":
            return None
        return float(str(value).replace(",", "").strip())
    except (ValueError, TypeError):
        return None


def validate_invoice(invoice):

    errors = []

    required_fields = [
        "invoice_number",
        "invoice_date",
        "supplier_name",
        "customer_name",
        "subtotal",
        "tax",
        "total"
    ]

    for field in required_fields:
        if invoice.get(field) is None or str(invoice.get(field)).strip() == "":
            errors.append(f"Missing required field: {field}")

    subtotal = safe_float(invoice.get("subtotal"))
    tax = safe_float(invoice.get("tax"))
    total = safe_float(invoice.get("total"))

    if subtotal is not None and tax is not None and total is not None:
        if abs((subtotal + tax) - total) > 0.01:
            errors.append("Subtotal + tax does not equal total")

    line_items = invoice.get("line_items", [])

    if line_items:
        item_total = sum(
            safe_float(item.get("amount")) or 0
            for item in line_items
        )

        if subtotal is not None and abs(item_total - subtotal) > 0.01:
            errors.append("Invoice amount mismatch: manual review required")

    return {"is_valid": len(errors) == 0, "errors": errors}


def extract_spreadsheet_invoice(file_path):

    sheets = pd.read_excel(file_path, sheet_name=None)

    invoice = {
        "invoice_number": None,
        "invoice_date": None,
        "supplier_name": None,
        "customer_name": None,
        "subtotal": None,
        "tax": None,
        "total": None,
        "currency": None,
        "line_items": []
    }

    for sheet_name, df in sheets.items():

        if {"Field", "Value"}.issubset(df.columns):

            for _, row in df.iterrows():
                field = str(row["Field"]).strip().lower()
                value = row["Value"]

                mapping = {
                    "invoice number": "invoice_number",
                    "invoice date": "invoice_date",
                    "supplier": "supplier_name",
                    "supplier name": "supplier_name",
                    "customer": "customer_name",
                    "customer name": "customer_name",
                    "subtotal": "subtotal",
                    "tax": "tax",
                    "total": "total",
                    "currency": "currency"
                }

                if field in mapping and pd.notna(value):
                    target = mapping[field]

                    if target in ["subtotal", "tax", "total"]:
                        value = safe_float(value)

                    invoice[target] = value

        if {"Description", "Quantity", "Unit Price"}.issubset(df.columns):

            for _, row in df.iterrows():
                if pd.isna(row["Description"]):
                    continue

                quantity = safe_float(row["Quantity"])
                price = safe_float(row["Unit Price"])

                if quantity is not None and price is not None:
                    invoice["line_items"].append({
                        "description": str(row["Description"]),
                        "quantity": quantity,
                        "unit_price": price,
                        "amount": quantity * price
                    })

    return invoice


def extract_csv_invoice(file_path):

    df = pd.read_csv(file_path, header=None)

    invoice = {
        "invoice_number": None,
        "invoice_date": None,
        "supplier_name": None,
        "customer_name": None,
        "subtotal": None,
        "tax": None,
        "total": None,
        "currency": None,
        "line_items": []
    }

    mapping = {
        "invoice number": "invoice_number",
        "invoice date": "invoice_date",
        "supplier": "supplier_name",
        "supplier name": "supplier_name",
        "customer": "customer_name",
        "customer name": "customer_name",
        "subtotal": "subtotal",
        "tax": "tax",
        "total": "total",
        "currency": "currency"
    }

    for _, row in df.iterrows():

        if len(row) < 2:
            continue

        key = str(row.iloc[0]).strip().lower()
        value = row.iloc[1]

        if key in mapping and pd.notna(value):
            target = mapping[key]

            if target in ["subtotal", "tax", "total"]:
                value = safe_float(value)

            invoice[target] = value

    return invoice


print("Validation and spreadsheet functions restored.")

Validation and spreadsheet functions restored.


In [19]:
# ==========================================
# MAIN DOCUMENT PROCESSING PIPELINE
# ==========================================

def process_document(file_path):

    try:
        extension = os.path.splitext(file_path)[1].lower()

        if extension == ".pdf":
            extracted_text = extract_pdf_text(file_path)
            invoice_data = extract_invoice_data(extracted_text)
            document_type = "pdf"

        elif extension in [".jpg", ".jpeg", ".png", ".tiff", ".tif"]:
            extracted_text = extract_image_text(file_path)
            invoice_data = extract_invoice_data(extracted_text)
            document_type = "image"

        elif extension == ".csv":
            extracted_text = extract_spreadsheet_text(file_path)
            invoice_data = extract_csv_invoice(file_path)
            document_type = "spreadsheet"

        elif extension in [".xlsx", ".xls"]:
            extracted_text = extract_spreadsheet_text(file_path)
            invoice_data = extract_spreadsheet_invoice(file_path)
            document_type = "spreadsheet"

        else:
            return {
                "status": "failed",
                "error": f"Unsupported file type: {extension}"
            }

        validation = validate_invoice(invoice_data)

        if document_type == "image":
            quality_errors = check_ocr_quality(extracted_text)
            validation["errors"].extend(quality_errors)
            validation["is_valid"] = len(validation["errors"]) == 0

        return {
            "status": "success",
            "document_type": document_type,
            "extracted_text": extracted_text,
            "invoice_data": invoice_data,
            "validation": validation
        }

    except Exception as e:
        return {
            "status": "failed",
            "error": str(e)
        }


print("Document processing pipeline restored.")

Document processing pipeline restored.


In [20]:
test_file = os.path.join(PROJECT_DIR, "history_test_invoice.png")

result = process_document(test_file)

print("Processing status:", result.get("status"))
print("Document type:", result.get("document_type"))
print("Invoice data:", result.get("invoice_data"))
print("Validation:", result.get("validation"))

Processing status: success
Document type: image
Invoice data: {'invoice_number': 'HISTORY-TEST-001', 'invoice_date': '05/11/2026', 'supplier_name': 'History Test Supplies', 'customer_name': 'Demo Buyer', 'subtotal': 500.0, 'tax': 75.0, 'total': 500.0, 'currency': 'BDT', 'line_items': []}
Validation: {'is_valid': False, 'errors': ['Subtotal + tax does not equal total']}


In [21]:
cursor = conn.cursor()

for table in ["documents", "invoices", "line_items",
              "validation_errors", "correction_history",
              "processing_history"]:

    cursor.execute(f"PRAGMA table_info({table})")

    print(f"\n{table.upper()}")

    for column in cursor.fetchall():
        print(column[1], "-", column[2])


DOCUMENTS
id - INTEGER
filename - TEXT
file_type - TEXT
processing_status - TEXT
upload_date - TIMESTAMP

INVOICES
id - INTEGER
document_id - INTEGER
invoice_number - TEXT
invoice_date - TEXT
supplier_name - TEXT
customer_name - TEXT
subtotal - REAL
tax - REAL
total - REAL
currency - TEXT

LINE_ITEMS
id - INTEGER
invoice_id - INTEGER
description - TEXT
quantity - REAL
unit_price - REAL
amount - REAL

VALIDATION_ERRORS
id - INTEGER
document_id - INTEGER
error_message - TEXT

CORRECTION_HISTORY
id - INTEGER
document_id - INTEGER
field_name - TEXT
old_value - TEXT
new_value - TEXT
corrected_at - TIMESTAMP

PROCESSING_HISTORY
id - INTEGER
document_id - INTEGER
processing_status - TEXT
details - TEXT
processed_at - TIMESTAMP


In [23]:

# DUPLICATE INVOICE DETECTION


def check_duplicate_invoice(invoice_number, supplier_name):
    cursor = conn.cursor()

    cursor.execute("""
        SELECT COUNT(*)
        FROM invoices
        WHERE LOWER(TRIM(invoice_number)) = LOWER(TRIM(?))
        AND LOWER(TRIM(supplier_name)) = LOWER(TRIM(?))
    """, (invoice_number, supplier_name))

    return cursor.fetchone()[0] > 0


def is_duplicate_before_save(invoice, exclude_document_id=None):

    cursor = conn.cursor()

    query = """
        SELECT COUNT(*)
        FROM invoices
        WHERE LOWER(TRIM(invoice_number)) = LOWER(TRIM(?))
        AND LOWER(TRIM(supplier_name)) = LOWER(TRIM(?))
    """

    params = [
        invoice.get("invoice_number"),
        invoice.get("supplier_name")
    ]

    if exclude_document_id is not None:
        query += " AND document_id != ?"
        params.append(exclude_document_id)

    cursor.execute(query, params)

    return cursor.fetchone()[0] > 0


print("Duplicate detection restored.")

Duplicate detection restored.


In [24]:

# SAVE EXTRACTED INVOICE TO DATABASE


def save_extracted_text(document_id, extracted_text):
    cursor = conn.cursor()

    cursor.execute("""
        INSERT INTO extracted_text (document_id, content)
        VALUES (?, ?)
    """, (document_id, extracted_text))


def save_invoice_record(document_id, invoice):

    cursor = conn.cursor()

    cursor.execute("""
        INSERT INTO invoices (
            document_id, invoice_number, invoice_date,
            supplier_name, customer_name,
            subtotal, tax, total, currency
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        document_id,
        invoice.get("invoice_number"),
        invoice.get("invoice_date"),
        invoice.get("supplier_name"),
        invoice.get("customer_name"),
        safe_float(invoice.get("subtotal")),
        safe_float(invoice.get("tax")),
        safe_float(invoice.get("total")),
        invoice.get("currency")
    ))

    invoice_id = cursor.lastrowid

    for item in invoice.get("line_items", []):

        cursor.execute("""
            INSERT INTO line_items (
                invoice_id, description, quantity,
                unit_price, amount
            )
            VALUES (?, ?, ?, ?, ?)
        """, (
            invoice_id,
            item.get("description"),
            safe_float(item.get("quantity")),
            safe_float(item.get("unit_price")),
            safe_float(item.get("amount"))
        ))

    return invoice_id


print("Database saving functions restored.")

Database saving functions restored.


In [25]:
cursor = conn.cursor()

cursor.execute("PRAGMA table_info(extracted_text)")
print("EXTRACTED TEXT TABLE:")
print(cursor.fetchall())

cursor.execute("PRAGMA table_info(invoices)")
print("\nINVOICES TABLE:")
print(cursor.fetchall())

EXTRACTED TEXT TABLE:
[(0, 'id', 'INTEGER', 0, None, 1), (1, 'document_id', 'INTEGER', 0, None, 0), (2, 'extracted_content', 'TEXT', 0, None, 0), (3, 'created_at', 'TIMESTAMP', 0, 'CURRENT_TIMESTAMP', 0)]

INVOICES TABLE:
[(0, 'id', 'INTEGER', 0, None, 1), (1, 'document_id', 'INTEGER', 0, None, 0), (2, 'invoice_number', 'TEXT', 0, None, 0), (3, 'invoice_date', 'TEXT', 0, None, 0), (4, 'supplier_name', 'TEXT', 0, None, 0), (5, 'customer_name', 'TEXT', 0, None, 0), (6, 'subtotal', 'REAL', 0, None, 0), (7, 'tax', 'REAL', 0, None, 0), (8, 'total', 'REAL', 0, None, 0), (9, 'currency', 'TEXT', 0, None, 0)]


In [26]:
def save_extracted_text(document_id, extracted_text):
    cursor = conn.cursor()
    cursor.execute("""
        INSERT INTO extracted_text (document_id, extracted_content)
        VALUES (?, ?)
    """, (document_id, extracted_text))


def save_invoice_record(document_id, invoice):
    cursor = conn.cursor()

    cursor.execute("""
        INSERT INTO invoices (
            document_id, invoice_number, invoice_date,
            supplier_name, customer_name,
            subtotal, tax, total, currency
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        document_id,
        invoice.get("invoice_number"),
        invoice.get("invoice_date"),
        invoice.get("supplier_name"),
        invoice.get("customer_name"),
        safe_float(invoice.get("subtotal")),
        safe_float(invoice.get("tax")),
        safe_float(invoice.get("total")),
        invoice.get("currency")
    ))

    invoice_id = cursor.lastrowid

    for item in invoice.get("line_items", []):
        cursor.execute("""
            INSERT INTO line_items (
                invoice_id, description, quantity,
                unit_price, amount
            )
            VALUES (?, ?, ?, ?, ?)
        """, (
            invoice_id,
            item.get("description"),
            safe_float(item.get("quantity")),
            safe_float(item.get("unit_price")),
            safe_float(item.get("amount"))
        ))

    return invoice_id

In [27]:
def update_existing_document(document_id, result):
    cursor = conn.cursor()

    invoice = result.get("invoice_data") or {}
    extracted_text = result.get("extracted_text", "")
    errors = list(result.get("validation", {}).get("errors", []))

    if invoice and is_duplicate_before_save(
        invoice, exclude_document_id=document_id
    ):
        errors.append("Duplicate invoice detected")

    cursor.execute(
        "SELECT id FROM invoices WHERE document_id = ?",
        (document_id,)
    )
    existing = cursor.fetchone()

    if existing:
        invoice_id = existing[0]

        cursor.execute("""
            UPDATE invoices SET
                invoice_number=?, invoice_date=?,
                supplier_name=?, customer_name=?,
                subtotal=?, tax=?, total=?, currency=?
            WHERE id=?
        """, (
            invoice.get("invoice_number"),
            invoice.get("invoice_date"),
            invoice.get("supplier_name"),
            invoice.get("customer_name"),
            safe_float(invoice.get("subtotal")),
            safe_float(invoice.get("tax")),
            safe_float(invoice.get("total")),
            invoice.get("currency"),
            invoice_id
        ))

        cursor.execute(
            "DELETE FROM line_items WHERE invoice_id=?",
            (invoice_id,)
        )

        for item in invoice.get("line_items", []):
            cursor.execute("""
                INSERT INTO line_items (
                    invoice_id, description, quantity, unit_price, amount
                ) VALUES (?, ?, ?, ?, ?)
            """, (
                invoice_id,
                item.get("description"),
                safe_float(item.get("quantity")),
                safe_float(item.get("unit_price")),
                safe_float(item.get("amount"))
            ))
    elif invoice:
        invoice_id = save_invoice_record(document_id, invoice)

    cursor.execute(
        "DELETE FROM extracted_text WHERE document_id=?",
        (document_id,)
    )
    if extracted_text:
        save_extracted_text(document_id, extracted_text)

    cursor.execute(
        "DELETE FROM validation_errors WHERE document_id=?",
        (document_id,)
    )

    for error in errors:
        cursor.execute("""
            INSERT INTO validation_errors (document_id, error_message)
            VALUES (?, ?)
        """, (document_id, error))

    status = "needs_review" if errors else "success"

    cursor.execute("""
        UPDATE documents
        SET processing_status=?
        WHERE id=?
    """, (status, document_id))

    conn.commit()
    return {"document_id": document_id, "status": status, "errors": errors}

In [28]:
async def process_upload_background(document_id, file_path):
    cursor = conn.cursor()

    try:
        cursor.execute("""
            INSERT INTO processing_history
            (document_id, processing_status, details)
            VALUES (?, ?, ?)
        """, (document_id, "processing", "Document processing started"))
        conn.commit()

        result = process_document(file_path)

        if result.get("status") == "failed":
            cursor.execute("""
                UPDATE documents
                SET processing_status='failed'
                WHERE id=?
            """, (document_id,))

            cursor.execute("""
                INSERT INTO processing_history
                (document_id, processing_status, details)
                VALUES (?, ?, ?)
            """, (document_id, "failed", str(result.get("error", "Processing failed"))))

            conn.commit()
            return

        saved = update_existing_document(document_id, result)
        status = saved["status"]

        cursor.execute("""
            INSERT INTO processing_history
            (document_id, processing_status, details)
            VALUES (?, ?, ?)
        """, (
            document_id,
            status,
            json.dumps(saved["errors"])
        ))

        conn.commit()

    except Exception as e:
        conn.rollback()

        cursor.execute("""
            UPDATE documents
            SET processing_status='failed'
            WHERE id=?
        """, (document_id,))

        cursor.execute("""
            INSERT INTO processing_history
            (document_id, processing_status, details)
            VALUES (?, ?, ?)
        """, (document_id, "failed", str(e)))

        conn.commit()
        print("Processing error:", str(e))

In [29]:
app = FastAPI(title="AI Document Intelligence System")

@app.get("/")
def home():
    return {"message": "AI Document Intelligence System is running"}

@app.post("/upload")
async def upload_document(
    background_tasks: BackgroundTasks,
    file: UploadFile = File(...)
):
    allowed_extensions = {
        ".pdf", ".jpg", ".jpeg", ".png",
        ".tiff", ".tif", ".xlsx", ".xls", ".csv"
    }

    extension = os.path.splitext(file.filename)[1].lower()

    if extension not in allowed_extensions:
        raise HTTPException(
            status_code=400,
            detail="Unsupported file format"
        )

    os.makedirs(UPLOAD_DIR, exist_ok=True)

    stored_filename = f"{uuid.uuid4()}{extension}"
    file_path = os.path.join(UPLOAD_DIR, stored_filename)

    try:
        with open(file_path, "wb") as buffer:
            shutil.copyfileobj(file.file, buffer)

        cursor = conn.cursor()

        cursor.execute("""
            INSERT INTO documents (
                filename, file_type, processing_status
            )
            VALUES (?, ?, ?)
        """, (file.filename, extension.lstrip("."), "pending"))

        document_id = cursor.lastrowid

        cursor.execute("""
            INSERT INTO processing_history (
                document_id, processing_status, details
            )
            VALUES (?, ?, ?)
        """, (document_id, "pending", "File uploaded"))

        conn.commit()

        background_tasks.add_task(
            process_upload_background,
            document_id,
            file_path
        )

        return {
            "message": "File uploaded successfully",
            "document_id": document_id,
            "filename": file.filename,
            "status": "pending"
        }

    except Exception as e:
        conn.rollback()
        raise HTTPException(status_code=500, detail=str(e))

In [30]:
@app.get("/documents")
def get_documents():
    cursor = conn.cursor()
    cursor.execute("""
        SELECT id, filename, file_type, processing_status, upload_date
        FROM documents
        ORDER BY id DESC
    """)
    rows = cursor.fetchall()

    return [
        {
            "document_id": r[0],
            "filename": r[1],
            "file_type": r[2],
            "status": r[3],
            "upload_date": r[4]
        }
        for r in rows
    ]


@app.get("/invoices")
def get_invoices():
    cursor = conn.cursor()
    cursor.execute("""
        SELECT id, document_id, invoice_number, invoice_date,
               supplier_name, customer_name, subtotal, tax, total, currency
        FROM invoices
        ORDER BY id DESC
    """)
    rows = cursor.fetchall()

    return [
        {
            "invoice_id": r[0],
            "document_id": r[1],
            "invoice_number": r[2],
            "invoice_date": r[3],
            "supplier_name": r[4],
            "customer_name": r[5],
            "subtotal": r[6],
            "tax": r[7],
            "total": r[8],
            "currency": r[9]
        }
        for r in rows
    ]


@app.get("/documents/{document_id}/status")
def get_document_status(document_id: int):
    cursor = conn.cursor()
    cursor.execute("""
        SELECT id, filename, processing_status
        FROM documents
        WHERE id=?
    """, (document_id,))

    row = cursor.fetchone()

    if not row:
        raise HTTPException(status_code=404, detail="Document not found")

    cursor.execute("""
        SELECT processing_status, details, processed_at
        FROM processing_history
        WHERE document_id=?
        ORDER BY id DESC
    """, (document_id,))

    history = cursor.fetchall()

    return {
        "document_id": row[0],
        "filename": row[1],
        "status": row[2],
        "history": [
            {
                "status": h[0],
                "details": h[1],
                "processed_at": h[2]
            }
            for h in history
        ]
    }

In [31]:
def revalidate_invoice_record(invoice_id):
    cursor = conn.cursor()

    cursor.execute("""
        SELECT document_id, invoice_number, invoice_date,
               supplier_name, customer_name, subtotal, tax, total, currency
        FROM invoices WHERE id=?
    """, (invoice_id,))

    row = cursor.fetchone()

    if not row:
        return {"error": "Invoice not found"}

    document_id = row[0]

    invoice = {
        "invoice_number": row[1],
        "invoice_date": row[2],
        "supplier_name": row[3],
        "customer_name": row[4],
        "subtotal": row[5],
        "tax": row[6],
        "total": row[7],
        "currency": row[8],
        "line_items": []
    }

    cursor.execute("""
        SELECT description, quantity, unit_price, amount
        FROM line_items WHERE invoice_id=?
    """, (invoice_id,))

    invoice["line_items"] = [
        {
            "description": r[0],
            "quantity": r[1],
            "unit_price": r[2],
            "amount": r[3]
        }
        for r in cursor.fetchall()
    ]

    errors = validate_invoice(invoice)

    if is_duplicate_before_save(invoice, exclude_document_id=document_id):
        errors.append("Duplicate invoice detected")

    cursor.execute(
        "DELETE FROM validation_errors WHERE document_id=?",
        (document_id,)
    )

    for error in errors:
        cursor.execute("""
            INSERT INTO validation_errors (document_id, error_message)
            VALUES (?, ?)
        """, (document_id, error))

    status = "needs_review" if errors else "success"

    cursor.execute("""
        UPDATE documents SET processing_status=?
        WHERE id=?
    """, (status, document_id))

    conn.commit()

    return {
        "invoice_id": invoice_id,
        "document_id": document_id,
        "status": status,
        "validation_errors": errors
    }


class CorrectionRequest(BaseModel):
    field_name: str
    new_value: str


@app.put("/invoices/{invoice_id}/correct")
def correct_invoice_api(invoice_id: int, correction: CorrectionRequest):
    allowed_fields = {
        "invoice_number", "invoice_date", "supplier_name",
        "customer_name", "subtotal", "tax", "total", "currency"
    }

    field = correction.field_name

    if field not in allowed_fields:
        raise HTTPException(status_code=400, detail="Invalid field name")

    cursor = conn.cursor()
    cursor.execute(
        f"SELECT document_id, {field} FROM invoices WHERE id=?",
        (invoice_id,)
    )
    row = cursor.fetchone()

    if not row:
        raise HTTPException(status_code=404, detail="Invoice not found")

    document_id, old_value = row

    new_value = correction.new_value.strip()

    if field in {"subtotal", "tax", "total"}:
        try:
            new_value = float(new_value)
        except ValueError:
            raise HTTPException(status_code=400, detail="Enter a valid number")

    cursor.execute(
        f"UPDATE invoices SET {field}=? WHERE id=?",
        (new_value, invoice_id)
    )

    cursor.execute("""
        INSERT INTO correction_history
        (document_id, field_name, old_value, new_value)
        VALUES (?, ?, ?, ?)
    """, (document_id, field, str(old_value), str(new_value)))

    conn.commit()

    return {
        "message": "Corrected successfully",
        "invoice_id": invoice_id,
        "field_name": field,
        "old_value": old_value,
        "new_value": new_value,
        "revalidation": revalidate_invoice_record(invoice_id)
    }

In [32]:
@app.get("/search")
def search_documents(q: str):
    cursor = conn.cursor()
    keyword = f"%{q}%"

    cursor.execute("""
        SELECT DISTINCT d.id, d.filename, d.processing_status,
               i.invoice_number, i.supplier_name, i.total
        FROM documents d
        LEFT JOIN extracted_text e ON d.id = e.document_id
        LEFT JOIN invoices i ON d.id = i.document_id
        WHERE d.filename LIKE ?
           OR e.extracted_content LIKE ?
           OR i.invoice_number LIKE ?
           OR i.supplier_name LIKE ?
           OR i.customer_name LIKE ?
        ORDER BY d.id DESC
    """, (keyword, keyword, keyword, keyword, keyword))

    return [
        {
            "document_id": r[0],
            "filename": r[1],
            "status": r[2],
            "invoice_number": r[3],
            "supplier_name": r[4],
            "total": r[5]
        }
        for r in cursor.fetchall()
    ]


@app.post("/retry/{document_id}")
def retry_document_api(
    document_id: int,
    background_tasks: BackgroundTasks
):
    cursor = conn.cursor()

    cursor.execute("""
        SELECT id FROM documents WHERE id=?
    """, (document_id,))

    if not cursor.fetchone():
        raise HTTPException(status_code=404, detail="Document not found")

    cursor.execute("""
        SELECT filename FROM documents WHERE id=?
    """, (document_id,))
    original_filename = cursor.fetchone()[0]

    matches = [
        f for f in os.listdir(UPLOAD_DIR)
        if f.startswith(original_filename)
    ]

    if not matches:
        raise HTTPException(
            status_code=404,
            detail="Original uploaded file not found"
        )

    file_path = os.path.join(UPLOAD_DIR, matches[0])

    cursor.execute("""
        UPDATE documents SET processing_status='pending'
        WHERE id=?
    """, (document_id,))

    conn.commit()

    background_tasks.add_task(
        process_upload_background,
        document_id,
        file_path
    )

    return {
        "message": "Retry started",
        "document_id": document_id,
        "status": "pending"
    }

In [33]:
import threading
import uvicorn
import time

def start_api():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="info"
    )

api_thread = threading.Thread(target=start_api, daemon=True)
api_thread.start()

time.sleep(3)

response = requests.get("http://127.0.0.1:8000/")

print("HTTP Status:", response.status_code)
print("Response:", response.json())

INFO:     Started server process [2958]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:60570 - "GET / HTTP/1.1" 200 OK
HTTP Status: 200
Response: {'message': 'AI Document Intelligence System is running'}


In [34]:
test_file = os.path.join(PROJECT_DIR, "history_test_invoice.png")

print("File exists:", os.path.exists(test_file))
print("File path:", test_file)

File exists: True
File path: /content/drive/MyDrive/AI_Document_Intelligence_System/history_test_invoice.png


In [35]:
with open(test_file, "rb") as f:
    response = requests.post(
        "http://127.0.0.1:8000/upload",
        files={"file": ("history_test_invoice.png", f, "image/png")}
    )

print("HTTP Status:", response.status_code)
print("Response:", response.json())

INFO:     127.0.0.1:52078 - "POST /upload HTTP/1.1" 200 OK
HTTP Status: 200
Response: {'message': 'File uploaded successfully', 'document_id': 16, 'filename': 'history_test_invoice.png', 'status': 'pending'}


In [36]:
import time

time.sleep(5)

response = requests.get(
    "http://127.0.0.1:8000/documents/16/status"
)

print(response.json())

INFO:     127.0.0.1:57784 - "GET /documents/16/status HTTP/1.1" 200 OK
{'document_id': 16, 'filename': 'history_test_invoice.png', 'status': 'needs_review', 'history': [{'status': 'needs_review', 'details': '["Subtotal + tax does not equal total"]', 'processed_at': '2026-11-06 06:18:16'}, {'status': 'processing', 'details': 'Document processing started', 'processed_at': '2026-11-06 06:18:15'}, {'status': 'pending', 'details': 'File uploaded', 'processed_at': '2026-11-06 06:18:15'}]}


In [37]:
response = requests.get("http://127.0.0.1:8000/invoices")

invoices = response.json()

for invoice in invoices:
    if invoice["document_id"] == 16:
        print(json.dumps(invoice, indent=4))

INFO:     127.0.0.1:55406 - "GET /invoices HTTP/1.1" 200 OK
{
    "invoice_id": 16,
    "document_id": 16,
    "invoice_number": "HISTORY-TEST-001",
    "invoice_date": "05/11/2026",
    "supplier_name": "History Test Supplies",
    "customer_name": "Demo Buyer",
    "subtotal": 500.0,
    "tax": 75.0,
    "total": 500.0,
    "currency": "BDT"
}


In [38]:
response = requests.put(
    "http://127.0.0.1:8000/invoices/16/correct",
    json={
        "field_name": "total",
        "new_value": "575"
    }
)

print("HTTP Status:", response.status_code)
print(json.dumps(response.json(), indent=4))

INFO:     127.0.0.1:47816 - "PUT /invoices/16/correct HTTP/1.1" 200 OK
HTTP Status: 200
{
    "message": "Corrected successfully",
    "invoice_id": 16,
    "field_name": "total",
    "old_value": 500.0,
    "new_value": 575.0,
    "revalidation": {
        "invoice_id": 16,
        "document_id": 16,
        "status": "needs_review",
        "validation_errors": {
            "is_valid": true,
            "errors": []
        }
    }
}


In [39]:
def revalidate_invoice_record(invoice_id):
    cursor = conn.cursor()

    cursor.execute("""
        SELECT document_id, invoice_number, invoice_date,
               supplier_name, customer_name, subtotal, tax, total, currency
        FROM invoices WHERE id=?
    """, (invoice_id,))

    row = cursor.fetchone()

    if not row:
        return {"error": "Invoice not found"}

    document_id = row[0]

    invoice = {
        "invoice_number": row[1],
        "invoice_date": row[2],
        "supplier_name": row[3],
        "customer_name": row[4],
        "subtotal": row[5],
        "tax": row[6],
        "total": row[7],
        "currency": row[8],
        "line_items": []
    }

    cursor.execute("""
        SELECT description, quantity, unit_price, amount
        FROM line_items WHERE invoice_id=?
    """, (invoice_id,))

    invoice["line_items"] = [
        {
            "description": r[0],
            "quantity": r[1],
            "unit_price": r[2],
            "amount": r[3]
        }
        for r in cursor.fetchall()
    ]

    validation = validate_invoice(invoice)
    errors = validation.get("errors", [])

    if is_duplicate_before_save(invoice, exclude_document_id=document_id):
        errors.append("Duplicate invoice detected")

    cursor.execute(
        "DELETE FROM validation_errors WHERE document_id=?",
        (document_id,)
    )

    for error in errors:
        cursor.execute("""
            INSERT INTO validation_errors (document_id, error_message)
            VALUES (?, ?)
        """, (document_id, error))

    status = "needs_review" if errors else "success"

    cursor.execute("""
        UPDATE documents SET processing_status=?
        WHERE id=?
    """, (status, document_id))

    conn.commit()

    return {
        "invoice_id": invoice_id,
        "document_id": document_id,
        "status": status,
        "validation_errors": errors
    }

In [40]:
response = requests.put(
    "http://127.0.0.1:8000/invoices/16/correct",
    json={
        "field_name": "total",
        "new_value": "575"
    }
)

print("HTTP Status:", response.status_code)
print(json.dumps(response.json(), indent=4))

INFO:     127.0.0.1:38396 - "PUT /invoices/16/correct HTTP/1.1" 200 OK
HTTP Status: 200
{
    "message": "Corrected successfully",
    "invoice_id": 16,
    "field_name": "total",
    "old_value": 575.0,
    "new_value": 575.0,
    "revalidation": {
        "invoice_id": 16,
        "document_id": 16,
        "status": "success",
        "validation_errors": []
    }
}


In [41]:
response = requests.get(
    "http://127.0.0.1:8000/search",
    params={"q": "HISTORY-TEST-001"}
)

print("HTTP Status:", response.status_code)
print(json.dumps(response.json(), indent=4))

INFO:     127.0.0.1:59556 - "GET /search?q=HISTORY-TEST-001 HTTP/1.1" 200 OK
HTTP Status: 200
[
    {
        "document_id": 16,
        "filename": "history_test_invoice.png",
        "status": "success",
        "invoice_number": "HISTORY-TEST-001",
        "supplier_name": "History Test Supplies",
        "total": 575.0
    }
]


In [42]:
print("Gradio version:", gr.__version__)
print("Interface exists:", "demo" in globals())

if "demo" in globals():
    print("Interface type:", type(demo))

Gradio version: 6.31.0
Interface exists: False


In [43]:
API_BASE = "http://127.0.0.1:8000"

def upload_files(files):
    if not files:
        return []

    results = []

    for file_path in files:
        try:
            with open(file_path, "rb") as f:
                response = requests.post(
                    f"{API_BASE}/upload",
                    files={
                        "file": (
                            os.path.basename(file_path),
                            f
                        )
                    }
                )

            results.append(response.json())

        except Exception as e:
            results.append({"error": str(e)})

    return results


def load_documents():
    response = requests.get(f"{API_BASE}/documents")
    return response.json()


def load_invoices():
    response = requests.get(f"{API_BASE}/invoices")
    return response.json()


def search_invoices(query):
    if not query:
        return []

    response = requests.get(
        f"{API_BASE}/search",
        params={"q": query}
    )
    return response.json()


def correct_invoice_ui(invoice_id, field, value):
    response = requests.put(
        f"{API_BASE}/invoices/{int(invoice_id)}/correct",
        json={
            "field_name": field,
            "new_value": str(value)
        }
    )
    return response.json()


with gr.Blocks(title="AI Document Intelligence System") as demo:

    gr.Markdown("# AI Document Intelligence System")
    gr.Markdown("Upload, process, search and review invoice documents.")

    with gr.Tab("Upload Documents"):
        file_input = gr.File(
            label="Select documents",
            file_count="multiple",
            type="filepath"
        )
        upload_button = gr.Button("Upload and Process")
        upload_output = gr.JSON(label="Upload Results")

        upload_button.click(
            upload_files,
            inputs=file_input,
            outputs=upload_output
        )

    with gr.Tab("Documents"):
        document_button = gr.Button("Refresh Documents")
        document_output = gr.JSON(label="Document Records")

        document_button.click(
            load_documents,
            outputs=document_output
        )

    with gr.Tab("Invoices"):
        invoice_button = gr.Button("Refresh Invoices")
        invoice_output = gr.JSON(label="Invoice Records")

        invoice_button.click(
            load_invoices,
            outputs=invoice_output
        )

    with gr.Tab("Search"):
        search_input = gr.Textbox(label="Search documents or invoices")
        search_button = gr.Button("Search")
        search_output = gr.JSON(label="Search Results")

        search_button.click(
            search_invoices,
            inputs=search_input,
            outputs=search_output
        )

    with gr.Tab("Correct Invoice"):
        invoice_id_input = gr.Number(
            label="Invoice ID",
            precision=0
        )

        field_input = gr.Dropdown(
            choices=[
                "invoice_number",
                "invoice_date",
                "supplier_name",
                "customer_name",
                "subtotal",
                "tax",
                "total",
                "currency"
            ],
            label="Field to correct"
        )

        value_input = gr.Textbox(label="Corrected value")
        correction_button = gr.Button("Submit Correction")
        correction_output = gr.JSON(label="Correction and Validation")

        correction_button.click(
            correct_invoice_ui,
            inputs=[
                invoice_id_input,
                field_input,
                value_input
            ],
            outputs=correction_output
        )

print("Gradio interface created successfully.")
print("Components:", len(demo.blocks))

Gradio interface created successfully.
Components: 25


In [44]:
demo.launch(
    share=True,
    server_name="0.0.0.0",
    server_port=7860
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://59f5bc44e9671fdd81.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [45]:
for folder, subfolders, files in os.walk(PROJECT_DIR):
    level = folder.replace(PROJECT_DIR, "").count(os.sep)

    if level > 2:
        subfolders[:] = []
        continue

    print("\n" + folder.replace(PROJECT_DIR, ""))

    for filename in files:
        print("  -", filename)



  - sample_invoice_v2.pdf
  - sample_invalid_invoice.pdf
  - sample_spreadsheet_invoice.xlsx
  - sample_csv_invoice.csv
  - sample_xls_invoice.xls
  - sample_csv_invoice_v2.csv
  - history_test_invoice.png

/document_intelligence
  - documents.db

/document_intelligence/uploads
  - 8263d77b-8b1d-4c32-a38a-4a4e35b3d7bd.pdf
  - 22fb4b83-054c-40f0-bd80-13a4b16698a0.pdf
  - 45117bca-f5b5-4ded-a951-4ab2e99b9861.pdf
  - faabfae7-dc3e-40af-aece-497efabee163.pdf
  - 82798143-06d9-4080-b862-80571eddad58.xls
  - 1b506a3e-53e7-49c3-a770-ce43a86fa2e0.xls
  - a62b6354-1277-4919-aea7-6b37356e07a3.csv
  - corrupted_test.pdf
  - test_file.txt
  - multi_page_test.pdf
  - ocr_test.png
  - e6170b30-c7dd-432b-9a03-eb0c2566e992.png
  - ec118d9c-eb50-4051-95a3-daf64e1d4d63.png
  - d02c4e38-24d2-4cb0-82be-7fccea922d5c.png
  - f7c21676-cc30-4d83-b4a1-a355b46e44a3.png
  - 5937e267-0de1-4451-ac13-a712ec67a490.png
  - 425b5e23-8d4f-49f0-89b5-113f78b107c4.png

/document_intelligence/extracted_text

/document_in

In [46]:
evaluation_file = os.path.join(
    PROJECT_DIR,
    "document_intelligence",
    "logs",
    "pdf_evaluation_results.json"
)

print("Evaluation file exists:", os.path.exists(evaluation_file))

if os.path.exists(evaluation_file):
    with open(evaluation_file, "r") as f:
        evaluation_data = json.load(f)

    print(json.dumps(evaluation_data, indent=4))

Evaluation file exists: True
{
    "filename": "multi_page_test.pdf",
    "total_pages": 5,
    "processing_time_seconds": 0.0239,
    "extracted_characters": 129,
    "extraction_successful": true
}


In [47]:
import time
import json
import os

test_files = [
    "sample_invoice_v2.pdf",
    "sample_invalid_invoice.pdf",
    "sample_spreadsheet_invoice.xlsx",
    "sample_csv_invoice.csv",
    "sample_xls_invoice.xls",
    "sample_csv_invoice_v2.csv",
    "history_test_invoice.png",
    "multi_page_test.pdf",
    "ocr_test.png",
    "corrupted_test.pdf",
    "test_file.txt"
]

evaluation_results = []

for filename in test_files:
    file_path = os.path.join(PROJECT_DIR, filename)

    if not os.path.exists(file_path):
        file_path = os.path.join(UPLOAD_DIR, filename)

    if not os.path.exists(file_path):
        evaluation_results.append({
            "filename": filename,
            "status": "file_not_found"
        })
        continue

    start = time.time()

    try:
        result = process_document(file_path)
        elapsed = round(time.time() - start, 4)

        invoice = result.get("invoice_data") or {}

        evaluation_results.append({
            "filename": filename,
            "status": result.get("status"),
            "processing_time_seconds": elapsed,
            "extracted_characters": len(
                result.get("extracted_text", "")
            ),
            "invoice_fields_extracted": sum(
                1 for key in [
                    "invoice_number", "invoice_date",
                    "supplier_name", "customer_name",
                    "subtotal", "tax", "total", "currency"
                ]
                if invoice.get(key) not in (None, "")
            ),
            "validation_errors": (
                result.get("validation", {}).get("errors", [])
            ),
            "error": result.get("error")
        })

    except Exception as e:
        evaluation_results.append({
            "filename": filename,
            "status": "failed",
            "error": str(e)
        })

print(json.dumps(evaluation_results, indent=4))

logs_dir = os.path.join(PROJECT_DIR, "document_intelligence", "logs")
os.makedirs(logs_dir, exist_ok=True)

evaluation_path = os.path.join(
    logs_dir,
    "multi_format_evaluation_results.json"
)

with open(evaluation_path, "w") as f:
    json.dump(evaluation_results, f, indent=4)

print("\nEvaluation saved to:", evaluation_path)

[
    {
        "filename": "sample_invoice_v2.pdf",
        "status": "success",
        "processing_time_seconds": 0.4218,
        "extracted_characters": 208,
        "invoice_fields_extracted": 8,
        "validation_errors": [
            "Subtotal + tax does not equal total"
        ],
        "error": null
    },
    {
        "filename": "sample_invalid_invoice.pdf",
        "status": "success",
        "processing_time_seconds": 0.247,
        "extracted_characters": 141,
        "invoice_fields_extracted": 8,
        "validation_errors": [
            "Subtotal + tax does not equal total"
        ],
        "error": null
    },
    {
        "filename": "sample_spreadsheet_invoice.xlsx",
        "status": "success",
        "processing_time_seconds": 3.7214,
        "extracted_characters": 494,
        "invoice_fields_extracted": 7,
        "validation_errors": [
            "Missing required field: invoice_date"
        ],
        "error": null
    },
    {
        "filename

In [48]:
import os

TEST_DIR = os.path.join(PROJECT_DIR, "tests")
os.makedirs(TEST_DIR, exist_ok=True)

print("Test directory created:", TEST_DIR)

Test directory created: /content/drive/MyDrive/AI_Document_Intelligence_System/tests


In [49]:
%%writefile /content/drive/MyDrive/AI_Document_Intelligence_System/tests/test_document_processing.py

import os
import sys
import pytest

PROJECT_DIR = "/content/drive/MyDrive/AI_Document_Intelligence_System"
sys.path.insert(0, PROJECT_DIR)

# These tests use the existing project sample files.
def get_file(name):
    path = os.path.join(PROJECT_DIR, name)
    if not os.path.exists(path):
        path = os.path.join(
            PROJECT_DIR, "document_intelligence", "uploads", name
        )
    return path


def test_pdf_processing():
    result = process_document(get_file("sample_invoice_v2.pdf"))
    assert result["status"] == "success"
    assert result["extracted_text"]


def test_excel_processing():
    result = process_document(get_file("sample_spreadsheet_invoice.xlsx"))
    assert result["status"] == "success"
    assert result["invoice_data"] is not None


def test_csv_processing():
    result = process_document(get_file("sample_csv_invoice.csv"))
    assert result["status"] == "success"


def test_image_ocr():
    result = process_document(get_file("ocr_test.png"))
    assert result["status"] == "success"
    assert len(result["extracted_text"]) > 0


def test_corrupted_pdf():
    result = process_document(get_file("corrupted_test.pdf"))
    assert result["status"] == "failed"


def test_unsupported_file():
    result = process_document(get_file("test_file.txt"))
    assert result["status"] == "failed"

Writing /content/drive/MyDrive/AI_Document_Intelligence_System/tests/test_document_processing.py


In [50]:
import os
import requests

def find_test_file(filename):
    possible_paths = [
        os.path.join(PROJECT_DIR, filename),
        os.path.join(UPLOAD_DIR, filename)
    ]

    for path in possible_paths:
        if os.path.exists(path):
            return path

    return None


tests = {
    "PDF processing": lambda: process_document(
        find_test_file("sample_invoice_v2.pdf")
    )["status"] == "success",

    "Excel processing": lambda: process_document(
        find_test_file("sample_spreadsheet_invoice.xlsx")
    )["status"] == "success",

    "CSV processing": lambda: process_document(
        find_test_file("sample_csv_invoice.csv")
    )["status"] == "success",

    "Image OCR": lambda: process_document(
        find_test_file("ocr_test.png")
    )["status"] == "success",

    "Corrupted PDF handling": lambda: process_document(
        find_test_file("corrupted_test.pdf")
    )["status"] == "failed",

    "Unsupported file handling": lambda: process_document(
        find_test_file("test_file.txt")
    )["status"] == "failed",

    "API health": lambda: requests.get(
        API_BASE + "/", timeout=10
    ).status_code == 200,

    "Keyword search": lambda: requests.get(
        API_BASE + "/search",
        params={"q": "HISTORY-TEST-001"},
        timeout=10
    ).status_code == 200
}

results = {}

for name, test in tests.items():
    try:
        results[name] = "PASS" if test() else "FAIL"
    except Exception as e:
        results[name] = f"ERROR: {e}"

print("AUTOMATED TEST RESULTS")
print("-" * 40)

for name, result in results.items():
    print(f"{name}: {result}")

passed = sum(value == "PASS" for value in results.values())
print("-" * 40)
print(f"Passed: {passed}/{len(results)}")

AUTOMATED TEST RESULTS
----------------------------------------
PDF processing: PASS
Excel processing: PASS
CSV processing: PASS
Image OCR: PASS
Corrupted PDF handling: PASS
Unsupported file handling: PASS
API health: PASS
Keyword search: PASS
----------------------------------------
Passed: 8/8


In [51]:
import json
from datetime import datetime

test_report = {
    "tested_at": datetime.now().isoformat(),
    "total_tests": len(results),
    "passed": passed,
    "failed": len(results) - passed,
    "results": results
}

report_path = os.path.join(
    PROJECT_DIR, "document_intelligence", "logs",
    "automated_test_results.json"
)

with open(report_path, "w") as f:
    json.dump(test_report, f, indent=4)

print("Test report saved:", report_path)

Test report saved: /content/drive/MyDrive/AI_Document_Intelligence_System/document_intelligence/logs/automated_test_results.json


In [52]:
readme_content = """
# AI Document Intelligence System

## Overview
An AI-assisted document processing system designed to upload, extract,
validate, search, and manage invoice information from multiple file formats.

## Supported Formats
- PDF
- JPG / JPEG
- PNG
- TIFF
- XLSX
- XLS
- CSV

## Main Features
- Text extraction from digital PDFs
- OCR for scanned PDFs and images
- Spreadsheet and CSV processing
- Structured invoice data extraction
- Invoice validation and arithmetic checks
- Duplicate invoice detection
- Manual invoice correction and revalidation
- Document processing history
- Keyword-based invoice search
- REST API using FastAPI
- Web interface using Gradio
- SQLite database storage

## Technologies
- Python
- FastAPI
- Gradio
- SQLite
- Pandas
- PyMuPDF
- Tesseract OCR
- Pillow
- Requests

## Project Structure
AI_Document_Intelligence_System/
- document_intelligence/
  - uploads/
  - logs/
  - documents.db
- tests/
  - test_document_processing.py
- README.md

## Running the System
Run the project notebook in Google Colab.
Mount Google Drive, install the dependencies, connect the database,
define the processing functions, and start the FastAPI and Gradio services.

## API Endpoints
- GET / : API health check
- POST /upload : Upload a document
- GET /documents : List documents
- GET /invoices : List extracted invoices
- GET /documents/{document_id}/status : Processing status
- GET /search?q= : Search invoices
- PUT /invoices/{invoice_id}/correct : Correct invoice fields
- POST /retry/{document_id} : Retry document processing

## Testing
Eight automated smoke checks passed:
- PDF processing
- Excel processing
- CSV processing
- Image OCR
- Corrupted PDF handling
- Unsupported file handling
- API health
- Keyword search

Results are stored in:
document_intelligence/logs/automated_test_results.json

## Current Limitations
- Invoice extraction is rule-based and may require manual review.
- Line-item extraction is limited for some document formats.
- Search currently uses keyword matching rather than embedding-based semantic search.
- Extraction success does not guarantee that every invoice field is accurate.
"""

readme_path = os.path.join(PROJECT_DIR, "README.md")

with open(readme_path, "w", encoding="utf-8") as f:
    f.write(readme_content)

print("README created:", readme_path)

README created: /content/drive/MyDrive/AI_Document_Intelligence_System/README.md


In [53]:
architecture = """flowchart TD
    A[User] --> B[Gradio Web Interface]
    B --> C[FastAPI Backend]
    C --> D[Document Upload]
    D --> E[File Type Detection]
    E --> F[Document Processing]
    F --> G[PDF Text Extraction / OCR / Spreadsheet Extraction]
    G --> H[Invoice Data Extraction]
    H --> I[Validation and Duplicate Detection]
    I --> J[SQLite Database]
    J --> K[Search and Correction APIs]
    K --> B
"""

architecture_path = os.path.join(PROJECT_DIR, "architecture.mmd")

with open(architecture_path, "w", encoding="utf-8") as f:
    f.write(architecture)

print("Architecture diagram source saved:", architecture_path)

Architecture diagram source saved: /content/drive/MyDrive/AI_Document_Intelligence_System/architecture.mmd


In [54]:
cursor = conn.cursor()

schema = {}

cursor.execute("""
    SELECT name FROM sqlite_master
    WHERE type='table'
    ORDER BY name
""")

for (table_name,) in cursor.fetchall():
    cursor.execute(f"PRAGMA table_info({table_name})")
    schema[table_name] = [
        {
            "column": row[1],
            "type": row[2],
            "primary_key": bool(row[5])
        }
        for row in cursor.fetchall()
    ]

schema_path = os.path.join(PROJECT_DIR, "database_schema.json")

with open(schema_path, "w", encoding="utf-8") as f:
    json.dump(schema, f, indent=4)

print("Database schema saved:", schema_path)

for table, columns in schema.items():
    print(f"\n{table}:")
    for column in columns:
        print(f"  {column['column']} ({column['type']})")

Database schema saved: /content/drive/MyDrive/AI_Document_Intelligence_System/database_schema.json

correction_history:
  id (INTEGER)
  document_id (INTEGER)
  field_name (TEXT)
  old_value (TEXT)
  new_value (TEXT)
  corrected_at (TIMESTAMP)

documents:
  id (INTEGER)
  filename (TEXT)
  file_type (TEXT)
  processing_status (TEXT)
  upload_date (TIMESTAMP)

extracted_text:
  id (INTEGER)
  document_id (INTEGER)
  extracted_content (TEXT)
  created_at (TIMESTAMP)

invoices:
  id (INTEGER)
  document_id (INTEGER)
  invoice_number (TEXT)
  invoice_date (TEXT)
  supplier_name (TEXT)
  customer_name (TEXT)
  subtotal (REAL)
  tax (REAL)
  total (REAL)
  currency (TEXT)

line_items:
  id (INTEGER)
  invoice_id (INTEGER)
  description (TEXT)
  quantity (REAL)
  unit_price (REAL)
  amount (REAL)

processing_history:
  id (INTEGER)
  document_id (INTEGER)
  processing_status (TEXT)
  details (TEXT)
  processed_at (TIMESTAMP)

sqlite_sequence:
  name ()
  seq ()

validation_errors:
  id (INTEG

In [55]:
schema_doc = """# Database Schema

The system uses SQLite for persistent storage.

## Application Tables

| Table | Purpose |
|---|---|
| documents | Uploaded document metadata and processing status |
| extracted_text | Text extracted from uploaded documents |
| invoices | Structured invoice information |
| line_items | Individual invoice items |
| processing_history | Document processing events |
| validation_errors | Invoice validation issues |
| correction_history | Manual correction records |

## Relationships

- documents.id → invoices.document_id
- documents.id → extracted_text.document_id
- documents.id → processing_history.document_id
- documents.id → validation_errors.document_id
- documents.id → correction_history.document_id
- invoices.id → line_items.invoice_id

## Database File

document_intelligence/documents.db

The database also contains sqlite_sequence, an internal SQLite table.
"""

schema_doc_path = os.path.join(PROJECT_DIR, "DATABASE_SCHEMA.md")

with open(schema_doc_path, "w", encoding="utf-8") as f:
    f.write(schema_doc)

print("Database documentation saved:", schema_doc_path)

Database documentation saved: /content/drive/MyDrive/AI_Document_Intelligence_System/DATABASE_SCHEMA.md


In [56]:
for root, dirs, files in os.walk(PROJECT_DIR):
    dirs[:] = [d for d in dirs if d not in [".gradio", ".pytest_cache", "__pycache__"]]

    level = root.replace(PROJECT_DIR, "").count(os.sep)
    indent = "    " * level

    print(f"{indent}{os.path.basename(root)}/")

    for filename in files:
        print(f"{indent}    {filename}")

AI_Document_Intelligence_System/
    sample_invoice_v2.pdf
    sample_invalid_invoice.pdf
    sample_spreadsheet_invoice.xlsx
    sample_csv_invoice.csv
    sample_xls_invoice.xls
    sample_csv_invoice_v2.csv
    history_test_invoice.png
    README.md
    architecture.mmd
    database_schema.json
    DATABASE_SCHEMA.md
    document_intelligence/
        documents.db
        uploads/
            8263d77b-8b1d-4c32-a38a-4a4e35b3d7bd.pdf
            22fb4b83-054c-40f0-bd80-13a4b16698a0.pdf
            45117bca-f5b5-4ded-a951-4ab2e99b9861.pdf
            faabfae7-dc3e-40af-aece-497efabee163.pdf
            82798143-06d9-4080-b862-80571eddad58.xls
            1b506a3e-53e7-49c3-a770-ce43a86fa2e0.xls
            a62b6354-1277-4919-aea7-6b37356e07a3.csv
            corrupted_test.pdf
            test_file.txt
            multi_page_test.pdf
            ocr_test.png
            e6170b30-c7dd-432b-9a03-eb0c2566e992.png
            ec118d9c-eb50-4051-95a3-daf64e1d4d63.png
            d02c4e38-2